## Model Development — Environment and Dataset Setup

This notebook begins the predictive modelling stage of the credit-risk
early-warning system.

The final modelling dataset was produced and independently validated during
the feature-engineering stage.

This notebook will:

1. Load the validated final modelling dataset.
2. Configure DuckDB for efficient large-scale inspection.
3. Confirm the dataset path and schema.
4. Verify the target definition and observation period.
5. Prepare the environment for temporal train/validation/test analysis.

The modelling workflow will use chronological rather than random splitting
to preserve the out-of-time nature of credit-risk prediction.

No model is trained in this cell.

In [1]:
# ============================================================
# CELL 1 — MODELLING ENVIRONMENT & DATASET SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import gc

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURE_OUTPUT_DIR = PROCESSED_DIR / "feature_engineering"

FINAL_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_final_modelling_features.parquet"
)

# ------------------------------------------------------------
# Check dataset
# ------------------------------------------------------------

print("=" * 70)
print("MODEL DEVELOPMENT — ENVIRONMENT SETUP")
print("=" * 70)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nFinal modelling dataset:")
print(FINAL_FEATURE_FILE)

print(
    "\nDataset exists:",
    FINAL_FEATURE_FILE.exists()
)

if not FINAL_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Final modelling dataset not found:\n"
        f"{FINAL_FEATURE_FILE}"
    )

# ------------------------------------------------------------
# DuckDB configuration
# ------------------------------------------------------------

MODEL_TEMP_DIR = (
    FEATURE_OUTPUT_DIR /
    "modeling_duckdb_temp"
)

MODEL_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_con = duckdb.connect()

model_con.execute(
    "SET memory_limit = '4GB'"
)

model_con.execute(
    "SET threads = 2"
)

model_con.execute(
    "SET preserve_insertion_order = false"
)

model_con.execute(
    f"SET temp_directory = '{MODEL_TEMP_DIR}'"
)

print("\nDuckDB version:")
print(
    model_con.execute(
        "SELECT version()"
    ).fetchone()[0]
)

print("\nDuckDB configuration:")
print("Memory limit : 4GB")
print("Threads      : 2")
print("Temp dir     :", MODEL_TEMP_DIR)

# ------------------------------------------------------------
# Dataset schema
# ------------------------------------------------------------

schema_df = model_con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    """
).fetchdf()

print("\nDataset schema:")
print(
    f"Total columns: {len(schema_df):,}"
)

display(schema_df)

# ------------------------------------------------------------
# Basic modelling dataset check
# ------------------------------------------------------------

basic_df = model_con.execute(
    f"""
    SELECT

        COUNT(*) AS observations,

        COUNT(DISTINCT loan_id)
            AS unique_loans,

        COUNT(DISTINCT reporting_date)
            AS reporting_months,

        MIN(reporting_date)
            AS min_reporting_date,

        MAX(reporting_date)
            AS max_reporting_date,

        SUM(
            CASE
                WHEN target_6m = 1
                THEN 1
                ELSE 0
            END
        ) AS events,

        SUM(
            CASE
                WHEN target_6m = 0
                THEN 1
                ELSE 0
            END
        ) AS non_events

    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    """
).fetchdf()

print("\nBasic dataset summary:")
display(basic_df)

# ------------------------------------------------------------
# Target validation
# ------------------------------------------------------------

target_values = model_con.execute(
    f"""
    SELECT DISTINCT target_6m
    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    ORDER BY target_6m
    """
).fetchdf()

print("\nTarget values:")
display(target_values)

# ------------------------------------------------------------
# Final setup checks
# ------------------------------------------------------------

assert FINAL_FEATURE_FILE.exists()

assert int(
    basic_df.loc[0, "observations"]
) == 20_155_269

assert int(
    basic_df.loc[0, "unique_loans"]
) == 450_106

assert int(
    basic_df.loc[0, "events"]
) == 646_957

assert int(
    basic_df.loc[0, "non_events"]
) == 19_508_312

assert set(
    target_values["target_6m"].tolist()
).issubset({0, 1})

print("\n" + "=" * 70)
print("SETUP RESULT: PASS")
print("=" * 70)

MODEL DEVELOPMENT — ENVIRONMENT SETUP

Project directory:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning

Final modelling dataset:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_final_modelling_features.parquet

Dataset exists: True

DuckDB version:
v1.5.5

DuckDB configuration:
Memory limit : 4GB
Threads      : 2
Temp dir     : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\modeling_duckdb_temp

Dataset schema:
Total columns: 66


,column_name,column_type,null,key,default,extra
0,loan_id,BIGINT,YES,None,None,None
1,reporting_date,TIMESTAMP,YES,None,None,None
2,target_6m,INTEGER,YES,None,None,None
3,channel,VARCHAR,YES,None,None,None
4,seller_name,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
61,interest_rate_change_1m,DOUBLE,YES,None,None,None
62,prior_month_loan_age,DOUBLE,YES,None,None,None
63,loan_age_change,DOUBLE,YES,None,None,None
64,prior_month_remaining_maturity,DOUBLE,YES,None,None,None



Basic dataset summary:


,observations,unique_loans,reporting_months,min_reporting_date,max_reporting_date,events,non_events
0,20155269,450106,93,2018-01-01,2025-09-01,646957.0,19508312.0



Target values:


,target_6m
0,0
1,1



SETUP RESULT: PASS


## Temporal Split Analysis — Monthly Event Rates

Before training any model, the distribution of the six-month early-warning
target is examined across reporting months.

A chronological split is required because the objective is to evaluate
whether information available at time t can predict credit deterioration
during the subsequent six months.

The monthly analysis is used to:

1. Measure observations and events by reporting month.
2. Examine the monthly event rate.
3. Identify changes in the target distribution over time.
4. Identify months with unusually sparse event counts.
5. Support a defensible chronological train/validation/test split.

No observations are randomly sampled and no model is trained in this cell.

In [2]:
# ============================================================
# CELL 2 — MONTHLY EVENT-RATE ANALYSIS
# ============================================================

print("=" * 70)
print("TEMPORAL SPLIT ANALYSIS — MONTHLY EVENT RATES")
print("=" * 70)

# ------------------------------------------------------------
# Monthly observation and target distribution
# ------------------------------------------------------------

monthly_query = f"""
SELECT
    reporting_date,

    COUNT(*) AS observations,

    SUM(
        CASE
            WHEN target_6m = 1
            THEN 1
            ELSE 0
        END
    ) AS events,

    SUM(
        CASE
            WHEN target_6m = 0
            THEN 1
            ELSE 0
        END
    ) AS non_events,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN target_6m = 1
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        4
    ) AS event_rate_pct

FROM read_parquet(
    '{FINAL_FEATURE_FILE}'
)

GROUP BY reporting_date

ORDER BY reporting_date
"""

monthly_df = model_con.execute(
    monthly_query
).fetchdf()

print("\nMonthly target distribution:")
display(monthly_df)

# ------------------------------------------------------------
# Summary statistics of monthly event rates
# ------------------------------------------------------------

monthly_rate_summary = monthly_df[
    "event_rate_pct"
].describe()

print("\nMonthly event-rate summary:")
display(
    monthly_rate_summary.to_frame(
        name="event_rate_pct"
    )
)

# ------------------------------------------------------------
# Minimum and maximum monthly event rates
# ------------------------------------------------------------

min_rate_row = monthly_df.loc[
    monthly_df["event_rate_pct"].idxmin()
]

max_rate_row = monthly_df.loc[
    monthly_df["event_rate_pct"].idxmax()
]

print("\nLowest monthly event rate:")
print(
    f"Date       : {min_rate_row['reporting_date']}"
)
print(
    f"Observations: {int(min_rate_row['observations']):,}"
)
print(
    f"Events      : {int(min_rate_row['events']):,}"
)
print(
    f"Event rate  : {min_rate_row['event_rate_pct']:.4f}%"
)

print("\nHighest monthly event rate:")
print(
    f"Date       : {max_rate_row['reporting_date']}"
)
print(
    f"Observations: {int(max_rate_row['observations']):,}"
)
print(
    f"Events      : {int(max_rate_row['events']):,}"
)
print(
    f"Event rate  : {max_rate_row['event_rate_pct']:.4f}%"
)

# ------------------------------------------------------------
# Lowest-event months
# ------------------------------------------------------------

print("\nMonths with the fewest events:")
display(
    monthly_df
    .sort_values("events")
    .head(10)
)

# ------------------------------------------------------------
# Highest-event months
# ------------------------------------------------------------

print("\nMonths with the most events:")
display(
    monthly_df
    .sort_values(
        "events",
        ascending=False
    )
    .head(10)
)

# ------------------------------------------------------------
# Overall event-rate consistency
# ------------------------------------------------------------

overall_observations = int(
    monthly_df["observations"].sum()
)

overall_events = int(
    monthly_df["events"].sum()
)

overall_non_events = int(
    monthly_df["non_events"].sum()
)

overall_event_rate = (
    100.0 *
    overall_events /
    overall_observations
)

print("\nOverall monthly aggregation check:")
print(
    f"Observations : {overall_observations:,}"
)
print(
    f"Events       : {overall_events:,}"
)
print(
    f"Non-events   : {overall_non_events:,}"
)
print(
    f"Event rate   : {overall_event_rate:.4f}%"
)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert len(monthly_df) == 93

assert (
    overall_observations
    == 20_155_269
)

assert (
    overall_events
    == 646_957
)

assert (
    overall_non_events
    == 19_508_312
)

assert (
    monthly_df["event_rate_pct"]
    .notna()
    .all()
)

assert (
    monthly_df["reporting_date"]
    .is_monotonic_increasing
)

print("\n" + "=" * 70)
print("MONTHLY TEMPORAL ANALYSIS: PASS")
print("=" * 70)

TEMPORAL SPLIT ANALYSIS — MONTHLY EVENT RATES

Monthly target distribution:


,reporting_date,observations,events,non_events,event_rate_pct
0,2018-01-01,162574,2933.0,159641.0,1.8041
1,2018-02-01,309000,4733.0,304267.0,1.5317
2,2018-03-01,446502,6902.0,439600.0,1.5458
3,2018-04-01,443985,6807.0,437178.0,1.5332
4,2018-05-01,442095,6958.0,435137.0,1.5739
...,...,...,...,...,...
88,2025-05-01,113882,3605.0,110277.0,3.1656
89,2025-06-01,113108,3535.0,109573.0,3.1253
90,2025-07-01,112510,3615.0,108895.0,3.2130
91,2025-08-01,111702,3577.0,108125.0,3.2023



Monthly event-rate summary:


,event_rate_pct
count,93.000000
mean,3.250785
std,1.539504
min,1.531700
25%,2.761400
50%,3.135400
75%,3.292400
max,9.506500



Lowest monthly event rate:
Date       : 2018-02-01 00:00:00
Observations: 309,000
Events      : 4,733
Event rate  : 1.5317%

Highest monthly event rate:
Date       : 2020-03-01 00:00:00
Observations: 302,626
Events      : 28,769
Event rate  : 9.5065%

Months with the fewest events:


,reporting_date,observations,events,non_events,event_rate_pct
0,2018-01-01,162574,2933.0,159641.0,1.8041
83,2024-12-01,117260,3196.0,114064.0,2.7256
85,2025-02-01,115839,3217.0,112622.0,2.7771
82,2024-11-01,117911,3256.0,114655.0,2.7614
87,2025-04-01,114624,3345.0,111279.0,2.9182
86,2025-03-01,115298,3361.0,111937.0,2.9151
84,2025-01-01,116784,3361.0,113423.0,2.8780
92,2025-09-01,111141,3531.0,107610.0,3.1770
89,2025-06-01,113108,3535.0,109573.0,3.1253
91,2025-08-01,111702,3577.0,108125.0,3.2023



Months with the most events:


,reporting_date,observations,events,non_events,event_rate_pct
24,2020-01-01,326949,29824.0,297125.0,9.1219
25,2020-02-01,314606,29608.0,284998.0,9.4111
23,2019-12-01,338918,29170.0,309748.0,8.6068
26,2020-03-01,302626,28769.0,273857.0,9.5065
22,2019-11-01,349957,27822.0,322135.0,7.9501
21,2019-10-01,361877,21302.0,340575.0,5.8865
27,2020-04-01,278676,16396.0,262280.0,5.8835
28,2020-05-01,260857,10004.0,250853.0,3.8351
29,2020-06-01,250453,8863.0,241590.0,3.5388
7,2018-08-01,435809,8825.0,426984.0,2.0250



Overall monthly aggregation check:
Observations : 20,155,269
Events       : 646,957
Non-events   : 19,508,312
Event rate   : 3.2099%

MONTHLY TEMPORAL ANALYSIS: PASS


## Temporal Train / Validation / Test Split

A chronological split is used instead of random sampling to preserve the
real-world forecasting structure of the credit-risk early-warning problem.

The final modelling dataset covers January 2018 through September 2025.

The proposed split is:

- Training: January 2018 – December 2022
- Validation: January 2023 – December 2023
- Test: January 2024 – September 2025

The training period contains the historical information used to estimate
model parameters.

The validation period is used for model selection, threshold selection,
and hyperparameter decisions.

The test period is held completely untouched until final model evaluation.

No future observations are allowed to influence model development.

The split is performed using reporting_date, not row order, because
reporting months contain different numbers of observations.

In [3]:
# ============================================================
# CELL 3 — TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

print("=" * 70)
print("TEMPORAL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# Define temporal windows
# ------------------------------------------------------------

TRAIN_START = "2018-01-01"
TRAIN_END   = "2022-12-01"

VALID_START = "2023-01-01"
VALID_END   = "2023-12-01"

TEST_START  = "2024-01-01"
TEST_END    = "2025-09-01"

# ------------------------------------------------------------
# Assign observations to temporal partitions
# ------------------------------------------------------------

split_query = f"""
SELECT
    CASE
        WHEN reporting_date BETWEEN
            DATE '{TRAIN_START}'
            AND DATE '{TRAIN_END}'
            THEN 'train'

        WHEN reporting_date BETWEEN
            DATE '{VALID_START}'
            AND DATE '{VALID_END}'
            THEN 'validation'

        WHEN reporting_date BETWEEN
            DATE '{TEST_START}'
            AND DATE '{TEST_END}'
            THEN 'test'

        ELSE 'outside'
    END AS dataset_split,

    COUNT(*) AS observations,

    COUNT(DISTINCT loan_id)
        AS unique_loans,

    SUM(
        CASE
            WHEN target_6m = 1
            THEN 1
            ELSE 0
        END
    ) AS events,

    SUM(
        CASE
            WHEN target_6m = 0
            THEN 1
            ELSE 0
        END
    ) AS non_events,

    COUNT(DISTINCT reporting_date)
        AS reporting_months,

    MIN(reporting_date)
        AS min_reporting_date,

    MAX(reporting_date)
        AS max_reporting_date,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN target_6m = 1
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        4
    ) AS event_rate_pct

FROM read_parquet(
    '{FINAL_FEATURE_FILE}'
)

GROUP BY dataset_split

ORDER BY
    CASE dataset_split
        WHEN 'train' THEN 1
        WHEN 'validation' THEN 2
        WHEN 'test' THEN 3
        ELSE 4
    END
"""

split_df = model_con.execute(
    split_query
).fetchdf()

print("\nTemporal split summary:")
display(split_df)

# ------------------------------------------------------------
# Check that every observation belongs to exactly one
# modelling partition
# ------------------------------------------------------------

partition_check = model_con.execute(
    f"""
    SELECT
        COUNT(*) AS total_observations,

        SUM(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{TRAIN_START}'
                    AND DATE '{TRAIN_END}'
                THEN 1
                ELSE 0
            END
        ) AS train_observations,

        SUM(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{VALID_START}'
                    AND DATE '{VALID_END}'
                THEN 1
                ELSE 0
            END
        ) AS validation_observations,

        SUM(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{TEST_START}'
                    AND DATE '{TEST_END}'
                THEN 1
                ELSE 0
            END
        ) AS test_observations

    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    """
).fetchdf()

print("\nPartition coverage:")
display(partition_check)

# ------------------------------------------------------------
# Check for outside observations
# ------------------------------------------------------------

outside_count = int(
    split_df.loc[
        split_df["dataset_split"] == "outside",
        "observations"
    ].sum()
)

print(
    "\nObservations outside defined windows:",
    f"{outside_count:,}"
)

# ------------------------------------------------------------
# Check chronological ordering
# ------------------------------------------------------------

chronology_df = model_con.execute(
    f"""
    SELECT

        MAX(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{TRAIN_START}'
                    AND DATE '{TRAIN_END}'
                THEN reporting_date
            END
        ) AS train_last_date,

        MIN(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{VALID_START}'
                    AND DATE '{VALID_END}'
                THEN reporting_date
            END
        ) AS validation_first_date,

        MAX(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{VALID_START}'
                    AND DATE '{VALID_END}'
                THEN reporting_date
            END
        ) AS validation_last_date,

        MIN(
            CASE
                WHEN reporting_date BETWEEN
                    DATE '{TEST_START}'
                    AND DATE '{TEST_END}'
                THEN reporting_date
            END
        ) AS test_first_date

    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    """
).fetchdf()

print("\nChronological ordering:")
display(chronology_df)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert outside_count == 0

assert (
    chronology_df.loc[0, "train_last_date"]
    < chronology_df.loc[0, "validation_first_date"]
)

assert (
    chronology_df.loc[0, "validation_last_date"]
    < chronology_df.loc[0, "test_first_date"]
)

assert (
    int(partition_check.loc[0, "total_observations"])
    == 20_155_269
)

assert (
    int(partition_check.loc[0, "train_observations"])
    + int(partition_check.loc[0, "validation_observations"])
    + int(partition_check.loc[0, "test_observations"])
    == 20_155_269
)

assert (
    int(split_df.loc[
        split_df["dataset_split"] == "train",
        "reporting_months"
    ].iloc[0])
    == 60
)

assert (
    int(split_df.loc[
        split_df["dataset_split"] == "validation",
        "reporting_months"
    ].iloc[0])
    == 12
)

assert (
    int(split_df.loc[
        split_df["dataset_split"] == "test",
        "reporting_months"
    ].iloc[0])
    == 21
)

print("\n" + "=" * 70)
print("TEMPORAL SPLIT RESULT: PASS")
print("=" * 70)

print("\nFinal split:")
print("Training   : Jan 2018 – Dec 2022")
print("Validation : Jan 2023 – Dec 2023")
print("Test       : Jan 2024 – Sep 2025")

TEMPORAL TRAIN / VALIDATION / TEST SPLIT

Temporal split summary:


,dataset_split,observations,unique_loans,events,non_events,reporting_months,min_reporting_date,max_reporting_date,event_rate_pct
0,train,16101651,450106,519144.0,15582507.0,60,2018-01-01,2022-12-01,3.2242
1,validation,1569879,138201,50952.0,1518927.0,12,2023-01-01,2023-12-01,3.2456
2,test,2483739,128906,76861.0,2406878.0,21,2024-01-01,2025-09-01,3.0946



Partition coverage:


,total_observations,train_observations,validation_observations,test_observations
0,20155269,16101651.0,1569879.0,2483739.0



Observations outside defined windows: 0

Chronological ordering:


,train_last_date,validation_first_date,validation_last_date,test_first_date
0,2022-12-01,2023-01-01,2023-12-01,2024-01-01



TEMPORAL SPLIT RESULT: PASS

Final split:
Training   : Jan 2018 – Dec 2022
Validation : Jan 2023 – Dec 2023
Test       : Jan 2024 – Sep 2025


## Feature Inventory and Modelling Variable Classification — Final

The final modelling dataset contains source variables together with engineered
features required for the credit-risk modelling framework.

Each column is explicitly classified as:

- identifier,
- time index,
- target,
- numeric predictor,
- categorical predictor,
- source date predictor,
- behavioural engineered feature,
- financial engineered feature,
- calendar engineered feature,
- temporal/dynamic engineered feature,
- source-only/excluded variable.

The raw payment-history string is retained only as a source/audit field because
its information has already been transformed into leakage-controlled behavioural
features.

Maturity date is retained for dataset traceability but excluded from modelling
because earlier source validation identified maturity-date anomalies and the
project does not require it as a direct predictor.

No target-based encoding or model preprocessing is performed in this step.

In [6]:
# ============================================================
# CELL 4 — FINAL FEATURE INVENTORY & CLASSIFICATION
# ============================================================

print("=" * 70)
print("FINAL FEATURE INVENTORY & MODELLING VARIABLE CLASSIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# Read actual schema
# ------------------------------------------------------------

schema_df = model_con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    """
).fetchdf()

available_columns = schema_df["column_name"].tolist()

print("\nTotal dataset columns:", len(available_columns))

# ------------------------------------------------------------
# 1. Identifier
# ------------------------------------------------------------

ID_COLUMNS = [
    "loan_id"
]

# ------------------------------------------------------------
# 2. Time index
# ------------------------------------------------------------

TIME_COLUMNS = [
    "reporting_date"
]

# ------------------------------------------------------------
# 3. Target
# ------------------------------------------------------------

TARGET_COLUMNS = [
    "target_6m"
]

# ------------------------------------------------------------
# 4. Source categorical predictors
# ------------------------------------------------------------

CATEGORICAL_FEATURES = [
    "channel",
    "seller_name",
    "servicer_name",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag"
]

# ------------------------------------------------------------
# 5. Source numeric predictors
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "original_loan_term",
    "loan_age",
    "remaining_months_legal_maturity",
    "remaining_months_maturity",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score",
    "co_borrower_credit_score",
    "number_of_units",
    "mortgage_insurance_percentage"
]

# ------------------------------------------------------------
# 6. Source date predictors
# ------------------------------------------------------------

DATE_FEATURES = [
    "origination_date",
    "first_payment_date"
]

# ------------------------------------------------------------
# 7. Engineered payment-behaviour features
# ------------------------------------------------------------

BEHAVIOURAL_FEATURES = [
    "ph_missing",
    "ph_prior_3m_delinquency",
    "ph_prior_6m_delinquency",
    "ph_prior_12m_delinquency",
    "ph_prior_24m_delinquency",
    "ph_prior_max_delinquency",
    "ph_prior_delinquency_count_6m",
    "ph_prior_delinquency_count_12m",
    "ph_prior_delinquency_count_24m"
]

# ------------------------------------------------------------
# 8. Engineered financial features
# ------------------------------------------------------------

FINANCIAL_FEATURES = [
    "balance_utilization",
    "interest_rate_change",
    "ltv_cltv_gap",
    "loan_age_ratio",
    "remaining_term_ratio",
    "co_borrower_score_gap",
    "borrower_credit_score_missing",
    "co_borrower_credit_score_missing",
    "mortgage_insurance_missing"
]

# ------------------------------------------------------------
# 9. Engineered calendar features
# ------------------------------------------------------------

CALENDAR_FEATURES = [
    "reporting_year",
    "reporting_month"
]

# ------------------------------------------------------------
# 10. Engineered temporal / dynamic features
# ------------------------------------------------------------

TEMPORAL_FEATURES = [
    "prior_month_balance",
    "balance_change",
    "balance_change_pct",
    "prior_month_interest_rate",
    "interest_rate_change_1m",
    "prior_month_loan_age",
    "loan_age_change",
    "prior_month_remaining_maturity",
    "remaining_maturity_change",
    "has_valid_prior_month"
]

# ------------------------------------------------------------
# 11. Source-only / explicitly excluded fields
# ------------------------------------------------------------

SOURCE_ONLY_FEATURES = [
    "loan_payment_history",
    "maturity_date"
]

# ------------------------------------------------------------
# Classification groups
# ------------------------------------------------------------

classification_groups = {
    "identifier": ID_COLUMNS,
    "time": TIME_COLUMNS,
    "target": TARGET_COLUMNS,
    "categorical": CATEGORICAL_FEATURES,
    "numeric": NUMERIC_FEATURES,
    "date": DATE_FEATURES,
    "behavioural_engineered": BEHAVIOURAL_FEATURES,
    "financial_engineered": FINANCIAL_FEATURES,
    "calendar_engineered": CALENDAR_FEATURES,
    "temporal_engineered": TEMPORAL_FEATURES,
    "source_only_excluded": SOURCE_ONLY_FEATURES
}

# ------------------------------------------------------------
# Flatten classifications
# ------------------------------------------------------------

classified_columns = []

for group, columns in classification_groups.items():
    classified_columns.extend(columns)

classification_series = pd.Series(
    classified_columns
)

duplicate_classifications = (
    classification_series[
        classification_series.duplicated(
            keep=False
        )
    ]
    .drop_duplicates()
    .tolist()
)

# ------------------------------------------------------------
# Missing expected columns
# ------------------------------------------------------------

missing_expected_columns = [
    col
    for col in classified_columns
    if col not in available_columns
]

# ------------------------------------------------------------
# Unclassified dataset columns
# ------------------------------------------------------------

classified_set = set(classified_columns)

unclassified_columns = [
    col
    for col in available_columns
    if col not in classified_set
]

# ------------------------------------------------------------
# Print classification summary
# ------------------------------------------------------------

print("\nClassification summary:")

for group, columns in classification_groups.items():

    present = [
        col
        for col in columns
        if col in available_columns
    ]

    missing = [
        col
        for col in columns
        if col not in available_columns
    ]

    print(
        f"{group:28s}: "
        f"{len(present):2d} present"
        f" | {len(missing):2d} missing"
    )

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("\nColumns not classified:")

if unclassified_columns:
    for col in unclassified_columns:
        print(" -", col)
else:
    print(" None")

print("\nExpected columns missing from dataset:")

if missing_expected_columns:
    for col in missing_expected_columns:
        print(" -", col)
else:
    print(" None")

print("\nColumns assigned to multiple groups:")

if duplicate_classifications:
    for col in duplicate_classifications:
        print(" -", col)
else:
    print(" None")

# ------------------------------------------------------------
# Modelling feature count
# ------------------------------------------------------------

MODELLING_FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + DATE_FEATURES
    + BEHAVIOURAL_FEATURES
    + FINANCIAL_FEATURES
    + CALENDAR_FEATURES
    + TEMPORAL_FEATURES
)

print("\n" + "=" * 70)
print("MODELLING FEATURE SUMMARY")
print("=" * 70)

print(
    "Categorical predictors       :",
    len(CATEGORICAL_FEATURES)
)

print(
    "Numeric predictors           :",
    len(NUMERIC_FEATURES)
)

print(
    "Date predictors              :",
    len(DATE_FEATURES)
)

print(
    "Behavioural engineered       :",
    len(BEHAVIOURAL_FEATURES)
)

print(
    "Financial engineered         :",
    len(FINANCIAL_FEATURES)
)

print(
    "Calendar engineered          :",
    len(CALENDAR_FEATURES)
)

print(
    "Temporal engineered          :",
    len(TEMPORAL_FEATURES)
)

print(
    "Total modelling features     :",
    len(MODELLING_FEATURES)
)

print(
    "Source-only/excluded fields  :",
    len(SOURCE_ONLY_FEATURES)
)

# ------------------------------------------------------------
# Actual schema
# ------------------------------------------------------------

print("\nActual schema data types:")

display(
    schema_df[
        ["column_name", "column_type", "null"]
    ]
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

classification_ok = (
    len(unclassified_columns) == 0
    and
    len(missing_expected_columns) == 0
    and
    len(duplicate_classifications) == 0
)

assert classification_ok

assert "loan_payment_history" in SOURCE_ONLY_FEATURES
assert "maturity_date" in SOURCE_ONLY_FEATURES

assert "target_6m" not in MODELLING_FEATURES
assert "loan_id" not in MODELLING_FEATURES
assert "reporting_date" not in MODELLING_FEATURES

print("\n" + "=" * 70)
print("FEATURE CLASSIFICATION RESULT: PASS")
print("=" * 70)

FINAL FEATURE INVENTORY & MODELLING VARIABLE CLASSIFICATION

Total dataset columns: 66

Classification summary:
identifier                  :  1 present |  0 missing
time                        :  1 present |  0 missing
target                      :  1 present |  0 missing
categorical                 : 14 present |  0 missing
numeric                     : 16 present |  0 missing
date                        :  2 present |  0 missing
behavioural_engineered      :  9 present |  0 missing
financial_engineered        :  9 present |  0 missing
calendar_engineered         :  2 present |  0 missing
temporal_engineered         : 10 present |  0 missing
source_only_excluded        :  1 present |  1 missing

Columns not classified:
 None

Expected columns missing from dataset:
 - loan_payment_history

Columns assigned to multiple groups:
 None

MODELLING FEATURE SUMMARY
Categorical predictors       : 14
Numeric predictors           : 16
Date predictors              : 2
Behavioural engineered     

,column_name,column_type,null
0,loan_id,BIGINT,YES
1,reporting_date,TIMESTAMP,YES
2,target_6m,INTEGER,YES
3,channel,VARCHAR,YES
4,seller_name,VARCHAR,YES
...,...,...,...
61,interest_rate_change_1m,DOUBLE,YES
62,prior_month_loan_age,DOUBLE,YES
63,loan_age_change,DOUBLE,YES
64,prior_month_remaining_maturity,DOUBLE,YES


AssertionError: 

In [7]:
# ============================================================
# CELL 4A — FINAL CLASSIFICATION CORRECTION
# ============================================================

print("=" * 70)
print("FINAL FEATURE CLASSIFICATION CORRECTION")
print("=" * 70)

# loan_payment_history is no longer present in the final dataset.
# Its information has already been transformed into ph_* features.
SOURCE_ONLY_FEATURES = [
    "maturity_date"
]

# Rebuild the classification list
classified_columns = (
    ID_COLUMNS
    + TIME_COLUMNS
    + TARGET_COLUMNS
    + CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + DATE_FEATURES
    + BEHAVIOURAL_FEATURES
    + FINANCIAL_FEATURES
    + CALENDAR_FEATURES
    + TEMPORAL_FEATURES
    + SOURCE_ONLY_FEATURES
)

classified_set = set(classified_columns)

unclassified_columns = [
    col
    for col in available_columns
    if col not in classified_set
]

missing_expected_columns = [
    col
    for col in classified_columns
    if col not in available_columns
]

classification_series = pd.Series(
    classified_columns
)

duplicate_classifications = (
    classification_series[
        classification_series.duplicated(keep=False)
    ]
    .drop_duplicates()
    .tolist()
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\nFinal classification audit:")

print(
    "Dataset columns              :",
    len(available_columns)
)

print(
    "Classified columns           :",
    len(set(classified_columns))
)

print(
    "Unclassified columns         :",
    len(unclassified_columns)
)

print(
    "Missing expected columns     :",
    len(missing_expected_columns)
)

print(
    "Duplicate classifications    :",
    len(duplicate_classifications)
)

print("\nUnclassified columns:")

if unclassified_columns:
    for col in unclassified_columns:
        print(" -", col)
else:
    print(" None")

print("\nMissing expected columns:")

if missing_expected_columns:
    for col in missing_expected_columns:
        print(" -", col)
else:
    print(" None")

# ------------------------------------------------------------
# Final modelling feature list
# ------------------------------------------------------------

MODELLING_FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + DATE_FEATURES
    + BEHAVIOURAL_FEATURES
    + FINANCIAL_FEATURES
    + CALENDAR_FEATURES
    + TEMPORAL_FEATURES
)

print("\n" + "=" * 70)
print("MODELLING FEATURE SUMMARY")
print("=" * 70)

print(
    "Categorical predictors       :",
    len(CATEGORICAL_FEATURES)
)

print(
    "Numeric predictors           :",
    len(NUMERIC_FEATURES)
)

print(
    "Date predictors              :",
    len(DATE_FEATURES)
)

print(
    "Behavioural engineered       :",
    len(BEHAVIOURAL_FEATURES)
)

print(
    "Financial engineered         :",
    len(FINANCIAL_FEATURES)
)

print(
    "Calendar engineered          :",
    len(CALENDAR_FEATURES)
)

print(
    "Temporal engineered          :",
    len(TEMPORAL_FEATURES)
)

print(
    "Total modelling features     :",
    len(MODELLING_FEATURES)
)

print(
    "Source-only fields           :",
    len(SOURCE_ONLY_FEATURES)
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(unclassified_columns) == 0
assert len(missing_expected_columns) == 0
assert len(duplicate_classifications) == 0

assert "maturity_date" in SOURCE_ONLY_FEATURES
assert "loan_payment_history" not in available_columns

assert "target_6m" not in MODELLING_FEATURES
assert "loan_id" not in MODELLING_FEATURES
assert "reporting_date" not in MODELLING_FEATURES

print("\n" + "=" * 70)
print("FEATURE CLASSIFICATION RESULT: PASS")
print("=" * 70)

FINAL FEATURE CLASSIFICATION CORRECTION

Final classification audit:
Dataset columns              : 66
Classified columns           : 66
Unclassified columns         : 0
Missing expected columns     : 0
Duplicate classifications    : 0

Unclassified columns:
 None

Missing expected columns:
 None

MODELLING FEATURE SUMMARY
Categorical predictors       : 14
Numeric predictors           : 16
Date predictors              : 2
Behavioural engineered       : 9
Financial engineered         : 9
Calendar engineered          : 2
Temporal engineered          : 10
Total modelling features     : 62
Source-only fields           : 1

FEATURE CLASSIFICATION RESULT: PASS


## Training-Only Preprocessing Audit

Before fitting any preprocessing transformation, the training, validation,
and test partitions are examined separately.

The purpose is to determine:

1. numeric missingness,
2. categorical missingness,
3. categorical cardinality,
4. high-cardinality variables requiring special treatment,
5. whether feature distributions differ materially across time periods.

All preprocessing decisions will subsequently be fitted using the training
partition only.

The validation and test partitions remain strictly out-of-sample.
No target-based encoding or imputation statistics are learned from them.

In [8]:
# ============================================================
# CELL 5 — TRAINING-ONLY PREPROCESSING AUDIT
# ============================================================

print("=" * 70)
print("TRAINING-ONLY PREPROCESSING AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Dataset split definitions
# ------------------------------------------------------------

TRAIN_END = "2022-12-01"
VALIDATION_END = "2023-12-01"
TEST_END = "2025-09-01"

# ------------------------------------------------------------
# Feature groups used for preprocessing
# ------------------------------------------------------------

PREPROCESS_NUMERIC_FEATURES = (
    NUMERIC_FEATURES
    + BEHAVIOURAL_FEATURES
    + FINANCIAL_FEATURES
    + CALENDAR_FEATURES
    + TEMPORAL_FEATURES
)

PREPROCESS_CATEGORICAL_FEATURES = (
    CATEGORICAL_FEATURES
)

PREPROCESS_DATE_FEATURES = (
    DATE_FEATURES
)

print("\nPreprocessing feature counts:")
print(
    "Numeric features     :",
    len(PREPROCESS_NUMERIC_FEATURES)
)
print(
    "Categorical features :",
    len(PREPROCESS_CATEGORICAL_FEATURES)
)
print(
    "Date features        :",
    len(PREPROCESS_DATE_FEATURES)
)

# ------------------------------------------------------------
# Build split labels
# ------------------------------------------------------------

split_query = f"""
SELECT
    CASE
        WHEN reporting_date <= DATE '2022-12-01'
            THEN 'train'
        WHEN reporting_date <= DATE '2023-12-01'
            THEN 'validation'
        WHEN reporting_date <= DATE '2025-09-01'
            THEN 'test'
        ELSE 'outside'
    END AS dataset_split,
    COUNT(*) AS observations,
    SUM(target_6m) AS events,
    SUM(1 - target_6m) AS non_events
FROM read_parquet(
    '{FINAL_FEATURE_FILE}'
)
GROUP BY 1
ORDER BY
    CASE dataset_split
        WHEN 'train' THEN 1
        WHEN 'validation' THEN 2
        WHEN 'test' THEN 3
        ELSE 4
    END
"""

split_summary_df = model_con.execute(
    split_query
).fetchdf()

print("\nSplit summary:")
display(split_summary_df)

# ------------------------------------------------------------
# Numeric missingness by split
# ------------------------------------------------------------

numeric_missing_records = []

for col in PREPROCESS_NUMERIC_FEATURES:

    query = f"""
    SELECT
        CASE
            WHEN reporting_date <= DATE '2022-12-01'
                THEN 'train'
            WHEN reporting_date <= DATE '2023-12-01'
                THEN 'validation'
            WHEN reporting_date <= DATE '2025-09-01'
                THEN 'test'
            ELSE 'outside'
        END AS dataset_split,

        COUNT(*) AS observations,

        COUNT(*) - COUNT("{col}") AS null_count

    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    GROUP BY 1
    """

    temp_df = model_con.execute(query).fetchdf()

    for _, row in temp_df.iterrows():

        observations = int(row["observations"])
        null_count = int(row["null_count"])

        numeric_missing_records.append({
            "feature": col,
            "dataset_split": row["dataset_split"],
            "observations": observations,
            "null_count": null_count,
            "null_percentage":
                100.0 * null_count / observations
        })

numeric_missing_df = pd.DataFrame(
    numeric_missing_records
)

print("\nNumeric missingness:")
display(
    numeric_missing_df[
        numeric_missing_df["null_count"] > 0
    ]
    .sort_values(
        ["feature", "dataset_split"]
    )
)

# ------------------------------------------------------------
# Categorical cardinality and missingness
# ------------------------------------------------------------

categorical_records = []

for col in PREPROCESS_CATEGORICAL_FEATURES:

    query = f"""
    SELECT
        CASE
            WHEN reporting_date <= DATE '2022-12-01'
                THEN 'train'
            WHEN reporting_date <= DATE '2023-12-01'
                THEN 'validation'
            WHEN reporting_date <= DATE '2025-09-01'
                THEN 'test'
            ELSE 'outside'
        END AS dataset_split,

        COUNT(*) AS observations,

        COUNT("{col}") AS non_null,

        COUNT(DISTINCT "{col}") AS distinct_values

    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    GROUP BY 1
    """

    temp_df = model_con.execute(query).fetchdf()

    for _, row in temp_df.iterrows():

        observations = int(row["observations"])
        non_null = int(row["non_null"])
        distinct_values = int(row["distinct_values"])

        null_count = (
            observations -
            non_null
        )

        categorical_records.append({
            "feature": col,
            "dataset_split": row["dataset_split"],
            "observations": observations,
            "distinct_values": distinct_values,
            "null_count": null_count,
            "null_percentage":
                100.0 * null_count / observations
        })

categorical_audit_df = pd.DataFrame(
    categorical_records
)

print("\nCategorical cardinality and missingness:")
display(
    categorical_audit_df
    .sort_values(
        ["distinct_values", "feature"],
        ascending=[False, True]
    )
)

# ------------------------------------------------------------
# Identify high-cardinality variables
# ------------------------------------------------------------

TRAIN_CARDINALITY_THRESHOLD = 50

high_cardinality_df = (
    categorical_audit_df[
        categorical_audit_df["dataset_split"] == "train"
    ]
    .loc[
        lambda x:
        x["distinct_values"]
        >= TRAIN_CARDINALITY_THRESHOLD
    ]
    .sort_values(
        "distinct_values",
        ascending=False
    )
)

print(
    "\nHigh-cardinality categorical features "
    f"(training distinct values >= {TRAIN_CARDINALITY_THRESHOLD}):"
)

display(
    high_cardinality_df[
        [
            "feature",
            "distinct_values",
            "null_count",
            "null_percentage"
        ]
    ]
)

# ------------------------------------------------------------
# Low-cardinality categorical features
# ------------------------------------------------------------

low_cardinality_df = (
    categorical_audit_df[
        (categorical_audit_df["dataset_split"] == "train")
        &
        (
            categorical_audit_df["distinct_values"]
            < TRAIN_CARDINALITY_THRESHOLD
        )
    ]
    .sort_values(
        "distinct_values",
        ascending=False
    )
)

print("\nLow-cardinality categorical features:")

display(
    low_cardinality_df[
        [
            "feature",
            "distinct_values",
            "null_count",
            "null_percentage"
        ]
    ]
)

# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

outside_rows = int(
    split_summary_df.loc[
        split_summary_df["dataset_split"] == "outside",
        "observations"
    ].sum()
)

assert outside_rows == 0

assert (
    set(
        split_summary_df["dataset_split"]
    )
    >= {"train", "validation", "test"}
)

print("\n" + "=" * 70)
print("PREPROCESSING AUDIT RESULT: PASS")
print("=" * 70)

print(
    "\nImportant: no preprocessing has been fitted yet."
)
print(
    "All imputation/encoding decisions will be based on TRAIN only."
)

TRAINING-ONLY PREPROCESSING AUDIT

Preprocessing feature counts:
Numeric features     : 46
Categorical features : 14
Date features        : 2

Split summary:


,dataset_split,observations,events,non_events
0,train,16101651,519144.0,15582507.0
1,validation,1569879,50952.0,1518927.0
2,test,2483739,76861.0,2406878.0



Numeric missingness:


,feature,dataset_split,observations,null_count,null_percentage
111,balance_change,test,2483739,18497,0.744724
112,balance_change,train,16101651,539741,3.352085
113,balance_change,validation,1569879,12210,0.777767
114,balance_change_pct,test,2483739,18497,0.744724
115,balance_change_pct,train,16101651,3179296,19.745155
116,balance_change_pct,validation,1569879,12210,0.777767
36,borrower_credit_score,test,2483739,1990,0.080121
37,borrower_credit_score,train,16101651,12832,0.079694
38,borrower_credit_score,validation,1569879,1283,0.081726
39,co_borrower_credit_score,test,2483739,1440256,57.987413



Categorical cardinality and missingness:


,feature,dataset_split,observations,distinct_values,null_count,null_percentage
28,zip_code_short,train,16101651,888,40,0.000248
27,zip_code_short,test,2483739,884,60,0.002416
29,zip_code_short,validation,1569879,884,17,0.001083
24,msa,test,2483739,409,0,0.000000
25,msa,train,16101651,395,0,0.000000
26,msa,validation,1569879,393,0,0.000000
7,servicer_name,train,16101651,82,0,0.000000
21,property_state,test,2483739,54,0,0.000000
22,property_state,train,16101651,54,0,0.000000
23,property_state,validation,1569879,54,0,0.000000



High-cardinality categorical features (training distinct values >= 50):


,feature,distinct_values,null_count,null_percentage
28,zip_code_short,888,40,0.000248
25,msa,395,0,0.000000
7,servicer_name,82,0,0.000000
22,property_state,54,0,0.000000



Low-cardinality categorical features:


,feature,distinct_values,null_count,null_percentage
4,seller_name,46,0,0.000000
16,property_type,5,0,0.000000
1,channel,3,0,0.000000
13,loan_purpose,3,0,0.000000
19,occupancy_status,3,0,0.000000
10,first_time_home_buyer,2,26,0.000161
40,modification_flag,2,0,0.000000
31,amortization_type,1,0,0.000000
34,prepayment_penalty_indicator,1,0,0.000000
37,interest_only_indicator,1,0,0.000000



PREPROCESSING AUDIT RESULT: PASS

Important: no preprocessing has been fitted yet.
All imputation/encoding decisions will be based on TRAIN only.


## Modelling Library and Preprocessing Strategy Audit

This cell verifies the availability and versions of the main modelling
libraries before constructing the production preprocessing pipeline.

The modelling framework will use:

1. A transparent baseline model for benchmarking.
2. A nonlinear tree-based model for predictive performance.
3. SHAP-based explanations for borrower-level risk attribution.

Preprocessing will respect the temporal split:

- Training statistics are fitted using TRAIN data only.
- Validation data is used only for model selection and threshold tuning.
- Test data remains untouched until final out-of-time evaluation.

Categorical variables will be separated into low-cardinality and
high-cardinality groups. High-cardinality variables will not be target-encoded
using validation or test observations.

In [10]:
# ============================================================
# CELL 6 — MODELLING LIBRARY & PREPROCESSING STRATEGY AUDIT
# ============================================================

print("=" * 70)
print("MODELLING LIBRARY & PREPROCESSING STRATEGY AUDIT")
print("=" * 70)

import importlib
import sys

# ------------------------------------------------------------
# Check required / candidate packages
# ------------------------------------------------------------

PACKAGE_LIST = [
    "pandas",
    "numpy",
    "sklearn",
    "duckdb",
    "shap",
    "xgboost",
    "lightgbm",
    "catboost"
]

package_results = []

for package_name in PACKAGE_LIST:

    try:
        module = importlib.import_module(package_name)

        version = getattr(
            module,
            "__version__",
            "version unavailable"
        )

        package_results.append({
            "package": package_name,
            "available": True,
            "version": version
        })

    except Exception as e:

        package_results.append({
            "package": package_name,
            "available": False,
            "version": str(e)
        })

package_df = pd.DataFrame(package_results)

print("\nPackage availability:")
display(package_df)

# ------------------------------------------------------------
# Define preprocessing groups
# ------------------------------------------------------------

HIGH_CARDINALITY_THRESHOLD = 50

HIGH_CARDINALITY_FEATURES = [
    "zip_code_short",
    "msa",
    "servicer_name",
    "property_state"
]

LOW_CARDINALITY_FEATURES = [
    col
    for col in CATEGORICAL_FEATURES
    if col not in HIGH_CARDINALITY_FEATURES
]

print("\nCategorical preprocessing groups:")
print(
    "High-cardinality categorical features :",
    len(HIGH_CARDINALITY_FEATURES)
)
print(
    "Low-cardinality categorical features  :",
    len(LOW_CARDINALITY_FEATURES)
)

print("\nHigh-cardinality features:")
for col in HIGH_CARDINALITY_FEATURES:
    print(" -", col)

print("\nLow-cardinality features:")
for col in LOW_CARDINALITY_FEATURES:
    print(" -", col)

# ------------------------------------------------------------
# Confirm all categorical variables are assigned exactly once
# ------------------------------------------------------------

all_categorical_assigned = (
    HIGH_CARDINALITY_FEATURES
    + LOW_CARDINALITY_FEATURES
)

assert set(all_categorical_assigned) == set(
    CATEGORICAL_FEATURES
)

assert len(all_categorical_assigned) == len(
    set(all_categorical_assigned)
)

# ------------------------------------------------------------
# Confirm preprocessing will not use target
# ------------------------------------------------------------

TARGET_EXCLUDED_FROM_PREPROCESSING = (
    "target_6m" not in MODELLING_FEATURES
)

assert TARGET_EXCLUDED_FROM_PREPROCESSING

# ------------------------------------------------------------
# Check core modelling packages
# ------------------------------------------------------------

available_packages = set(
    package_df.loc[
        package_df["available"],
        "package"
    ]
)

required_core = {
    "pandas",
    "numpy",
    "sklearn",
    "duckdb"
}

missing_core = (
    required_core -
    available_packages
)

assert len(missing_core) == 0

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREPROCESSING STRATEGY")
print("=" * 70)

print("""
Numeric features:
    - Preserve numeric meaning.
    - Missing values will be handled using TRAIN-only statistics.
    - Missingness indicators will be retained where semantically useful.

Categorical features:
    - Low-cardinality variables will use one-hot encoding.
    - High-cardinality variables will use a leakage-safe encoding strategy
      determined after reviewing available tree-model libraries.

Date variables:
    - Date columns will not be passed directly as raw timestamps.
    - Appropriate calendar/age information has already been engineered.

Target:
    - target_6m is excluded from all preprocessing transformations.
""")

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("=" * 70)
print("LIBRARY & PREPROCESSING AUDIT RESULT: PASS")
print("=" * 70)

print(
    "\nAvailable modelling packages:",
    ", ".join(sorted(available_packages))
)

print(
    "Missing core packages:",
    "None" if len(missing_core) == 0 else
    ", ".join(sorted(missing_core))
)

MODELLING LIBRARY & PREPROCESSING STRATEGY AUDIT

Package availability:


,package,available,version
0,pandas,True,3.0.3
1,numpy,True,2.4.6
2,sklearn,True,1.9.0
3,duckdb,True,1.5.5
4,shap,False,No module named 'shap'
5,xgboost,False,No module named 'xgboost'
6,lightgbm,False,No module named 'lightgbm'
7,catboost,False,No module named 'catboost'



Categorical preprocessing groups:
High-cardinality categorical features : 4
Low-cardinality categorical features  : 10

High-cardinality features:
 - zip_code_short
 - msa
 - servicer_name
 - property_state

Low-cardinality features:
 - channel
 - seller_name
 - first_time_home_buyer
 - loan_purpose
 - property_type
 - occupancy_status
 - amortization_type
 - prepayment_penalty_indicator
 - interest_only_indicator
 - modification_flag

PREPROCESSING STRATEGY

Numeric features:
    - Preserve numeric meaning.
    - Missing values will be handled using TRAIN-only statistics.
    - Missingness indicators will be retained where semantically useful.

Categorical features:
    - Low-cardinality variables will use one-hot encoding.
    - High-cardinality variables will use a leakage-safe encoding strategy
      determined after reviewing available tree-model libraries.

Date variables:
    - Date columns will not be passed directly as raw timestamps.
    - Appropriate calendar/age informatio

## Training-Only Preprocessing Parameter Construction

This cell constructs preprocessing parameters using the TRAINING
period only (January 2018 to December 2022).

The following strategy is used:

1. Numeric features:
   - Missing values are imputed using the training-set median.
   - Validation and test data will never influence these medians.

2. Low-cardinality categorical features:
   - One-hot encoding will be used.
   - Unknown categories appearing later will be handled safely.

3. High-cardinality categorical features:
   - Frequency encoding will be used.
   - Category frequencies are learned from TRAIN only.
   - Validation/test categories not observed during training receive
     a frequency of zero.
   - No target information is used.

4. Date features:
   - Dates are retained for subsequent deterministic date transformation.
   - No statistics are estimated from validation or test periods.

This cell only creates and validates preprocessing parameters.
It does not transform the full 20M-row dataset.

In [11]:
# ============================================================
# CELL 7 — TRAINING-ONLY PREPROCESSING PARAMETERS
# ============================================================

print("=" * 70)
print("TRAINING-ONLY PREPROCESSING PARAMETER CONSTRUCTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Training-only source
# ------------------------------------------------------------

TRAIN_FILTER = """
reporting_date >= DATE '2018-01-01'
AND reporting_date <= DATE '2022-12-01'
"""

# ------------------------------------------------------------
# 2. Numeric training medians
# ------------------------------------------------------------

numeric_median_records = []

print("\nCalculating TRAIN-only numeric medians...")

for col in PREPROCESS_NUMERIC_FEATURES:

    query = f"""
    SELECT
        '{col}' AS feature,
        MEDIAN("{col}") AS train_median,
        COUNT(*) AS train_observations,
        COUNT("{col}") AS train_non_null,
        COUNT(*) - COUNT("{col}") AS train_null_count
    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    WHERE {TRAIN_FILTER}
    """

    result = model_con.execute(query).fetchdf()

    numeric_median_records.append(
        result.iloc[0].to_dict()
    )

NUMERIC_PREPROCESSING_PARAMS = pd.DataFrame(
    numeric_median_records
)

NUMERIC_PREPROCESSING_PARAMS[
    "train_null_percentage"
] = (
    NUMERIC_PREPROCESSING_PARAMS["train_null_count"]
    /
    NUMERIC_PREPROCESSING_PARAMS["train_observations"]
    * 100
)

print("\nTraining numeric preprocessing parameters:")

display(
    NUMERIC_PREPROCESSING_PARAMS[
        [
            "feature",
            "train_median",
            "train_null_count",
            "train_null_percentage"
        ]
    ]
    .sort_values(
        "train_null_percentage",
        ascending=False
    )
)

# ------------------------------------------------------------
# 3. High-cardinality frequency maps
# ------------------------------------------------------------

print("\nBuilding TRAIN-only frequency maps...")

HIGH_CARDINALITY_FREQUENCY_MAPS = {}

for col in HIGH_CARDINALITY_FEATURES:

    query = f"""
    SELECT
        "{col}" AS category,
        COUNT(*) AS category_count,
        COUNT(*) * 1.0 /
            SUM(COUNT(*)) OVER () AS category_frequency
    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    WHERE {TRAIN_FILTER}
      AND "{col}" IS NOT NULL
    GROUP BY "{col}"
    ORDER BY category_frequency DESC
    """

    freq_df = model_con.execute(query).fetchdf()

    HIGH_CARDINALITY_FREQUENCY_MAPS[col] = freq_df

    print(
        f"  {col:<20} "
        f"{len(freq_df):>5} training categories"
    )

# ------------------------------------------------------------
# 4. Frequency-map audit
# ------------------------------------------------------------

print("\nFrequency-map audit:")

frequency_audit_records = []

for col, freq_df in HIGH_CARDINALITY_FREQUENCY_MAPS.items():

    frequency_sum = (
        freq_df["category_frequency"]
        .sum()
    )

    frequency_audit_records.append({
        "feature": col,
        "training_categories": len(freq_df),
        "frequency_sum": frequency_sum,
        "minimum_frequency":
            freq_df["category_frequency"].min(),
        "maximum_frequency":
            freq_df["category_frequency"].max()
    })

frequency_audit_df = pd.DataFrame(
    frequency_audit_records
)

display(frequency_audit_df)

# ------------------------------------------------------------
# 5. Low-cardinality category audit
# ------------------------------------------------------------

print("\nLow-cardinality training categories:")

low_cardinality_records = []

for col in LOW_CARDINALITY_FEATURES:

    query = f"""
    SELECT
        "{col}" AS category,
        COUNT(*) AS category_count
    FROM read_parquet(
        '{FINAL_FEATURE_FILE}'
    )
    WHERE {TRAIN_FILTER}
    GROUP BY "{col}"
    ORDER BY category_count DESC
    """

    category_df = model_con.execute(query).fetchdf()

    low_cardinality_records.append({
        "feature": col,
        "training_categories":
            int(category_df["category"].notna().sum()),
        "training_null_count":
            int(
                category_df.loc[
                    category_df["category"].isna(),
                    "category_count"
                ].sum()
            )
            if category_df["category"].isna().any()
            else 0
    })

LOW_CARDINALITY_AUDIT = pd.DataFrame(
    low_cardinality_records
)

display(LOW_CARDINALITY_AUDIT)

# ------------------------------------------------------------
# 6. Verify training-only parameter completeness
# ------------------------------------------------------------

print("\nParameter completeness checks:")

assert (
    set(
        NUMERIC_PREPROCESSING_PARAMS["feature"]
    )
    == set(PREPROCESS_NUMERIC_FEATURES)
)

assert (
    set(
        HIGH_CARDINALITY_FREQUENCY_MAPS.keys()
    )
    == set(HIGH_CARDINALITY_FEATURES)
)

assert (
    set(LOW_CARDINALITY_AUDIT["feature"])
    == set(LOW_CARDINALITY_FEATURES)
)

# Every numeric feature must have a valid median unless the
# entire training feature is missing.
numeric_missing_medians = (
    NUMERIC_PREPROCESSING_PARAMS[
        NUMERIC_PREPROCESSING_PARAMS["train_median"].isna()
    ]
)

print(
    "Numeric features with missing training median:",
    len(numeric_missing_medians)
)

if len(numeric_missing_medians) > 0:
    display(numeric_missing_medians)

# Frequency sums should be approximately 1.
frequency_sum_ok = np.allclose(
    frequency_audit_df["frequency_sum"].values,
    1.0,
    atol=1e-8
)

assert frequency_sum_ok

# ------------------------------------------------------------
# 7. Explicit leakage-control statement
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE CONTROL")
print("=" * 70)

print("""
Preprocessing parameters were calculated using TRAINING observations only.

Validation period:
    January 2023 – December 2023
    NOT used to calculate medians or category frequencies.

Test period:
    January 2024 – September 2025
    NOT used to calculate medians or category frequencies.

Target variable:
    target_6m was NOT used in any preprocessing calculation.
""")

# ------------------------------------------------------------
# 8. Final result
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING-ONLY PREPROCESSING PARAMETER AUDIT: PASS")
print("=" * 70)

print(
    f"Numeric parameters created    : "
    f"{len(NUMERIC_PREPROCESSING_PARAMS)}"
)

print(
    f"High-cardinality maps created : "
    f"{len(HIGH_CARDINALITY_FREQUENCY_MAPS)}"
)

print(
    f"Low-cardinality groups        : "
    f"{len(LOW_CARDINALITY_FEATURES)}"
)

TRAINING-ONLY PREPROCESSING PARAMETER CONSTRUCTION

Calculating TRAIN-only numeric medians...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (19 outputs left out)

,feature,training_categories,frequency_sum,minimum_frequency,maximum_frequency
0,zip_code_short,888,1.0,1.180006e-06,0.011894
1,msa,395,1.0,2.384849e-05,0.093655
2,servicer_name,82,1.0,6.210543e-08,0.360016
3,property_state,54,1.0,3.148745e-05,0.124384



Low-cardinality training categories:


,feature,training_categories,training_null_count
0,channel,3,0
1,seller_name,46,0
2,first_time_home_buyer,2,26
3,loan_purpose,3,0
4,property_type,5,0
5,occupancy_status,3,0
6,amortization_type,1,0
7,prepayment_penalty_indicator,1,0
8,interest_only_indicator,1,0
9,modification_flag,2,0



Parameter completeness checks:
Numeric features with missing training median: 0

LEAKAGE CONTROL

Preprocessing parameters were calculated using TRAINING observations only.

Validation period:
    January 2023 – December 2023
    NOT used to calculate medians or category frequencies.

Test period:
    January 2024 – September 2025
    NOT used to calculate medians or category frequencies.

Target variable:
    target_6m was NOT used in any preprocessing calculation.

TRAINING-ONLY PREPROCESSING PARAMETER AUDIT: PASS
Numeric parameters created    : 46
High-cardinality maps created : 4
Low-cardinality groups        : 10


## Model Preprocessing Transformation Test

This cell performs a small-scale transformation test on 10,000
observations from the modelling dataset.

The transformation uses the preprocessing parameters learned from
the TRAINING period only:

- Numeric variables → training-set median imputation
- Low-cardinality categorical variables → one-hot encoding
- High-cardinality categorical variables → training-only frequency encoding
- Engineered/date variables → retained according to the modelling feature design
- Target → kept separately and never used during preprocessing

The purpose is to verify:

1. All expected predictors are available.
2. Numeric missing values are handled correctly.
3. Categorical encoding produces valid values.
4. Unseen/high-cardinality categories can be handled safely.
5. No non-finite values are introduced.
6. The resulting feature matrix has the expected dimensions.

Only after this test passes will the full training/validation/test
transformation be performed.

In [12]:
# ============================================================
# CELL 8 — SMALL-SCALE MODEL PREPROCESSING TEST
# ============================================================

print("=" * 70)
print("MODEL PREPROCESSING TRANSFORMATION TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Read a small sample
# ------------------------------------------------------------

SAMPLE_SIZE = 10_000

sample_query = f"""
SELECT *
FROM read_parquet('{FINAL_FEATURE_FILE}')
ORDER BY reporting_date, loan_id
LIMIT {SAMPLE_SIZE}
"""

sample_df = model_con.execute(sample_query).fetchdf()

print(f"\nSample observations: {len(sample_df):,}")
print(f"Sample columns     : {len(sample_df.columns)}")

assert len(sample_df) == SAMPLE_SIZE
assert "target_6m" in sample_df.columns

# ------------------------------------------------------------
# 2. Separate target and identifiers
# ------------------------------------------------------------

y_sample = sample_df["target_6m"].astype("int8").copy()

sample_ids = sample_df[
    ["loan_id", "reporting_date"]
].copy()

X_sample = sample_df.drop(
    columns=["target_6m", "loan_id", "reporting_date"],
    errors="ignore"
).copy()

# ------------------------------------------------------------
# 3. Verify feature groups
# ------------------------------------------------------------

all_predictor_features = (
    PREPROCESS_NUMERIC_FEATURES
    + LOW_CARDINALITY_FEATURES
    + HIGH_CARDINALITY_FEATURES
)

missing_predictors = [
    c for c in all_predictor_features
    if c not in X_sample.columns
]

print("\nPredictor availability:")
print(f"Expected predictors : {len(all_predictor_features)}")
print(f"Missing predictors  : {len(missing_predictors)}")

assert len(missing_predictors) == 0

# ------------------------------------------------------------
# 4. Numeric preprocessing
# ------------------------------------------------------------

X_numeric = pd.DataFrame(index=X_sample.index)

for _, row in NUMERIC_PREPROCESSING_PARAMS.iterrows():

    col = row["feature"]
    median_value = row["train_median"]

    values = pd.to_numeric(
        X_sample[col],
        errors="coerce"
    )

    X_numeric[col] = values.fillna(median_value)

# ------------------------------------------------------------
# 5. Low-cardinality categorical preprocessing
# ------------------------------------------------------------

X_lowcat = pd.DataFrame(index=X_sample.index)

for col in LOW_CARDINALITY_FEATURES:

    values = (
        X_sample[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    # Training categories
    training_categories_query = f"""
    SELECT DISTINCT "{col}" AS category
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    WHERE {TRAIN_FILTER}
      AND "{col}" IS NOT NULL
    """

    training_categories_df = model_con.execute(
        training_categories_query
    ).fetchdf()

    training_categories = set(
        training_categories_df["category"]
        .dropna()
        .astype(str)
        .tolist()
    )

    # Explicit missing category
    training_categories.add("__MISSING__")

    # Map unseen values to explicit unknown category
    values = values.apply(
        lambda x:
        x if str(x) in training_categories
        else "__UNKNOWN__"
    )

    X_lowcat[col] = values

# ------------------------------------------------------------
# 6. One-hot encode low-cardinality variables
# ------------------------------------------------------------

X_lowcat_encoded = pd.get_dummies(
    X_lowcat,
    columns=LOW_CARDINALITY_FEATURES,
    dtype=np.int8
)

# ------------------------------------------------------------
# 7. High-cardinality frequency encoding
# ------------------------------------------------------------

X_highcat = pd.DataFrame(index=X_sample.index)

for col in HIGH_CARDINALITY_FEATURES:

    freq_map = (
        HIGH_CARDINALITY_FREQUENCY_MAPS[col]
        .set_index("category")["category_frequency"]
        .to_dict()
    )

    values = X_sample[col]

    X_highcat[col + "_frequency"] = (
        values.map(freq_map)
        .fillna(0.0)
        .astype("float32")
    )

# ------------------------------------------------------------
# 8. Combine transformed features
# ------------------------------------------------------------

X_transformed = pd.concat(
    [
        X_numeric,
        X_lowcat_encoded,
        X_highcat
    ],
    axis=1
)

# ------------------------------------------------------------
# 9. Validate target
# ------------------------------------------------------------

assert set(y_sample.unique()).issubset({0, 1})
assert len(y_sample) == len(X_transformed)

# ------------------------------------------------------------
# 10. Validate numerical finiteness
# ------------------------------------------------------------

numeric_matrix = X_transformed.select_dtypes(
    include=[np.number]
)

nonfinite_count = (
    ~np.isfinite(
        numeric_matrix.to_numpy(
            dtype=np.float64,
            na_value=np.nan
        )
    )
).sum()

null_count = (
    X_transformed.isna()
    .sum()
    .sum()
)

print("\nTransformation summary:")
print(f"Original predictors       : {len(all_predictor_features)}")
print(f"Transformed feature count : {X_transformed.shape[1]:,}")
print(f"Numeric features          : {X_numeric.shape[1]}")
print(f"One-hot features          : {X_lowcat_encoded.shape[1]}")
print(f"Frequency features        : {X_highcat.shape[1]}")

print("\nOutput validation:")
print(f"Rows preserved            : {len(X_transformed):,}")
print(f"NULL values               : {int(null_count):,}")
print(f"Non-finite values         : {int(nonfinite_count):,}")

# ------------------------------------------------------------
# 11. Check that frequency features are valid probabilities
# ------------------------------------------------------------

frequency_columns = [
    c for c in X_transformed.columns
    if c.endswith("_frequency")
]

frequency_min = (
    X_transformed[frequency_columns]
    .min()
    .min()
)

frequency_max = (
    X_transformed[frequency_columns]
    .max()
    .max()
)

print("\nFrequency encoding range:")
print(f"Minimum frequency         : {frequency_min:.10f}")
print(f"Maximum frequency         : {frequency_max:.10f}")

# ------------------------------------------------------------
# 12. Assertions
# ------------------------------------------------------------

assert null_count == 0
assert nonfinite_count == 0

assert frequency_min >= 0
assert frequency_max <= 1

assert len(X_transformed) == SAMPLE_SIZE
assert len(y_sample) == SAMPLE_SIZE

# Ensure categorical one-hot values are binary
one_hot_columns = [
    c for c in X_lowcat_encoded.columns
]

if one_hot_columns:
    one_hot_values = set(
        np.unique(
            X_lowcat_encoded.to_numpy()
        )
    )

    assert one_hot_values.issubset({0, 1})

print("\n" + "=" * 70)
print("MODEL PREPROCESSING TEST: PASS")
print("=" * 70)

MODEL PREPROCESSING TRANSFORMATION TEST


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample observations: 10,000
Sample columns     : 66

Predictor availability:
Expected predictors : 60
Missing predictors  : 0

Transformation summary:
Original predictors       : 60
Transformed feature count : 100
Numeric features          : 46
One-hot features          : 50
Frequency features        : 4

Output validation:
Rows preserved            : 10,000
NULL values               : 0
Non-finite values         : 0

Frequency encoding range:
Minimum frequency         : 0.0000003726
Maximum frequency         : 0.3600157499

MODEL PREPROCESSING TEST: PASS


## Temporal Split Preprocessing Test

This cell verifies that the same preprocessing parameters learned
from the training period can be applied consistently to the
validation and test periods.

Temporal partitions:

- Training: January 2018 – December 2022
- Validation: January 2023 – December 2023
- Test: January 2024 – September 2025

Important leakage-control rule:

All imputation statistics and high-cardinality frequency maps were
estimated exclusively from the training period.

Validation and test observations are transformed using those fixed
training parameters. No validation or test information is used to
refit preprocessing.

The cell performs a small sample transformation from each temporal
partition and verifies:

1. Correct temporal partitioning.
2. Expected predictors are available.
3. Identical transformed feature structure across all splits.
4. No NULL values after preprocessing.
5. No non-finite numerical values.
6. Frequency-encoded variables remain within [0, 1].
7. Target remains binary and separate from the feature matrix.

Only after this test passes will the full datasets be transformed.

In [14]:
# ============================================================
# CELL 9 — CORRECTED TEMPORAL PREPROCESSING TEST
# ============================================================

print("=" * 70)
print("CORRECTED TEMPORAL SPLIT PREPROCESSING TEST")
print("=" * 70)

SPLIT_SAMPLE_SIZE = 5_000

# ------------------------------------------------------------
# 1. Temporal split definitions
# ------------------------------------------------------------

SPLIT_DEFINITIONS = {
    "train": """
        reporting_date >= DATE '2018-01-01'
        AND reporting_date <= DATE '2022-12-01'
    """,

    "validation": """
        reporting_date >= DATE '2023-01-01'
        AND reporting_date <= DATE '2023-12-01'
    """,

    "test": """
        reporting_date >= DATE '2024-01-01'
        AND reporting_date <= DATE '2025-09-01'
    """
}

# ------------------------------------------------------------
# 2. Build FIXED training-derived one-hot schema
# ------------------------------------------------------------

print("\nBuilding fixed training-derived categorical schema...")

TRAINING_CATEGORY_VALUES = {}
FIXED_ONEHOT_COLUMNS = []

for col in LOW_CARDINALITY_FEATURES:

    category_query = f"""
    SELECT DISTINCT "{col}" AS category
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    WHERE {TRAIN_FILTER}
      AND "{col}" IS NOT NULL
    ORDER BY "{col}"
    """

    category_df = model_con.execute(
        category_query
    ).fetchdf()

    categories = (
        category_df["category"]
        .dropna()
        .astype(str)
        .tolist()
    )

    # Explicit missing and unknown categories
    categories = (
        ["__MISSING__", "__UNKNOWN__"]
        + categories
    )

    TRAINING_CATEGORY_VALUES[col] = categories

    for category in categories:

        FIXED_ONEHOT_COLUMNS.append(
            f"{col}_{category}"
        )

    print(
        f"  {col:<30} "
        f"{len(categories):>4} encoded categories"
    )

print(
    "\nTotal fixed one-hot columns:",
    len(FIXED_ONEHOT_COLUMNS)
)

# ------------------------------------------------------------
# 3. Transformation function
# ------------------------------------------------------------

def transform_temporal_sample(df):

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    y = df["target_6m"].astype("int8").copy()

    # --------------------------------------------------------
    # Predictor data
    # --------------------------------------------------------

    X = df.drop(
        columns=[
            "target_6m",
            "loan_id",
            "reporting_date"
        ],
        errors="ignore"
    ).copy()

    # --------------------------------------------------------
    # Numeric features
    # --------------------------------------------------------

    X_numeric = pd.DataFrame(
        index=X.index
    )

    for _, row in NUMERIC_PREPROCESSING_PARAMS.iterrows():

        col = row["feature"]
        median_value = row["train_median"]

        values = pd.to_numeric(
            X[col],
            errors="coerce"
        )

        X_numeric[col] = values.fillna(
            median_value
        )

    # --------------------------------------------------------
    # Low-cardinality categorical features
    # --------------------------------------------------------

    X_lowcat = pd.DataFrame(
        index=X.index
    )

    for col in LOW_CARDINALITY_FEATURES:

        values = (
            X[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

        training_categories = set(
            TRAINING_CATEGORY_VALUES[col]
        )

        values = values.apply(
            lambda x:
            x
            if x in training_categories
            else "__UNKNOWN__"
        )

        X_lowcat[col] = values

    # --------------------------------------------------------
    # One-hot encoding
    # --------------------------------------------------------

    X_lowcat_encoded = pd.get_dummies(
        X_lowcat,
        columns=LOW_CARDINALITY_FEATURES,
        dtype=np.int8
    )

    # --------------------------------------------------------
    # Force EXACT same one-hot schema
    # --------------------------------------------------------

    X_lowcat_encoded = (
        X_lowcat_encoded
        .reindex(
            columns=FIXED_ONEHOT_COLUMNS,
            fill_value=0
        )
    )

    # --------------------------------------------------------
    # High-cardinality frequency encoding
    # --------------------------------------------------------

    X_highcat = pd.DataFrame(
        index=X.index
    )

    for col in HIGH_CARDINALITY_FEATURES:

        freq_map = (
            HIGH_CARDINALITY_FREQUENCY_MAPS[col]
            .set_index("category")[
                "category_frequency"
            ]
            .to_dict()
        )

        X_highcat[
            col + "_frequency"
        ] = (
            X[col]
            .map(freq_map)
            .fillna(0.0)
            .astype("float32")
        )

    # --------------------------------------------------------
    # Combine all transformed predictors
    # --------------------------------------------------------

    X_transformed = pd.concat(
        [
            X_numeric,
            X_lowcat_encoded,
            X_highcat
        ],
        axis=1
    )

    return X_transformed, y


# ------------------------------------------------------------
# 4. Transform samples from all temporal partitions
# ------------------------------------------------------------

split_results = {}

for split_name, split_filter in SPLIT_DEFINITIONS.items():

    print(
        f"\nProcessing {split_name.upper()} sample..."
    )

    query = f"""
    SELECT *
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    WHERE {split_filter}
    ORDER BY reporting_date, loan_id
    LIMIT {SPLIT_SAMPLE_SIZE}
    """

    split_df = model_con.execute(
        query
    ).fetchdf()

    assert len(split_df) == SPLIT_SAMPLE_SIZE

    X_split, y_split = transform_temporal_sample(
        split_df
    )

    split_results[split_name] = (
        X_split,
        y_split
    )

    print(
        f"  Observations : {len(split_df):,}"
    )

    print(
        f"  Features     : {X_split.shape[1]:,}"
    )

    print(
        f"  Target values: "
        f"{sorted(y_split.unique().tolist())}"
    )


# ------------------------------------------------------------
# 5. Feature-schema comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE SCHEMA CHECK")
print("=" * 70)

train_columns = list(
    split_results["train"][0].columns
)

validation_columns = list(
    split_results["validation"][0].columns
)

test_columns = list(
    split_results["test"][0].columns
)

print(
    f"Train transformed features      : "
    f"{len(train_columns)}"
)

print(
    f"Validation transformed features : "
    f"{len(validation_columns)}"
)

print(
    f"Test transformed features       : "
    f"{len(test_columns)}"
)

schema_match_validation = (
    train_columns == validation_columns
)

schema_match_test = (
    train_columns == test_columns
)

print(
    f"\nTrain = Validation schema      : "
    f"{'PASS' if schema_match_validation else 'FAIL'}"
)

print(
    f"Train = Test schema             : "
    f"{'PASS' if schema_match_test else 'FAIL'}"
)

# ------------------------------------------------------------
# 6. Numerical validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMERICAL OUTPUT VALIDATION")
print("=" * 70)

for split_name, (X_split, y_split) in split_results.items():

    numeric_values = (
        X_split
        .select_dtypes(
            include=[np.number]
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    null_count = int(
        X_split.isna()
        .sum()
        .sum()
    )

    nonfinite_count = int(
        (~np.isfinite(numeric_values))
        .sum()
    )

    print(
        f"{split_name:<12} "
        f"NULL={null_count:,} | "
        f"Non-finite={nonfinite_count:,}"
    )

    assert null_count == 0
    assert nonfinite_count == 0


# ------------------------------------------------------------
# 7. Frequency encoding validation
# ------------------------------------------------------------

frequency_columns = [
    c
    for c in train_columns
    if c.endswith("_frequency")
]

print("\nFrequency features:")
print(frequency_columns)

for split_name, (X_split, _) in split_results.items():

    freq_min = (
        X_split[frequency_columns]
        .min()
        .min()
    )

    freq_max = (
        X_split[frequency_columns]
        .max()
        .max()
    )

    print(
        f"{split_name:<12} "
        f"min={freq_min:.10f} | "
        f"max={freq_max:.10f}"
    )

    assert freq_min >= 0
    assert freq_max <= 1


# ------------------------------------------------------------
# 8. Target validation
# ------------------------------------------------------------

print("\nTarget validation:")

for split_name, (_, y_split) in split_results.items():

    target_values = sorted(
        y_split.unique().tolist()
    )

    print(
        f"{split_name:<12} "
        f"{target_values}"
    )

    assert set(
        target_values
    ).issubset({0, 1})


# ------------------------------------------------------------
# 9. Row preservation
# ------------------------------------------------------------

print("\nRow-preservation validation:")

for split_name, (X_split, y_split) in split_results.items():

    print(
        f"{split_name:<12} "
        f"X={len(X_split):,} | "
        f"y={len(y_split):,}"
    )

    assert len(X_split) == SPLIT_SAMPLE_SIZE
    assert len(y_split) == SPLIT_SAMPLE_SIZE


# ------------------------------------------------------------
# 10. Final assertions
# ------------------------------------------------------------

assert schema_match_validation
assert schema_match_test

assert len(train_columns) == (
    len(PREPROCESS_NUMERIC_FEATURES)
    + len(FIXED_ONEHOT_COLUMNS)
    + len(HIGH_CARDINALITY_FEATURES)
)

# ------------------------------------------------------------
# 11. Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED TEMPORAL PREPROCESSING TEST: PASS")
print("=" * 70)

print(
    f"Numeric features       : "
    f"{len(PREPROCESS_NUMERIC_FEATURES)}"
)

print(
    f"Fixed one-hot features : "
    f"{len(FIXED_ONEHOT_COLUMNS)}"
)

print(
    f"Frequency features     : "
    f"{len(HIGH_CARDINALITY_FEATURES)}"
)

print(
    f"Final transformed features: "
    f"{len(train_columns)}"
)

print(
    "Identical train/validation/test schema : PASS"
)

print(
    "Training-only preprocessing             : PASS"
)

print(
    "No NULL values                          : PASS"
)

print(
    "No non-finite values                    : PASS"
)

CORRECTED TEMPORAL SPLIT PREPROCESSING TEST

Building fixed training-derived categorical schema...
  channel                           5 encoded categories
  seller_name                      48 encoded categories
  first_time_home_buyer             4 encoded categories
  loan_purpose                      5 encoded categories
  property_type                     7 encoded categories
  occupancy_status                  5 encoded categories
  amortization_type                 3 encoded categories
  prepayment_penalty_indicator      3 encoded categories
  interest_only_indicator           3 encoded categories
  modification_flag                 4 encoded categories

Total fixed one-hot columns: 87

Processing TRAIN sample...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Observations : 5,000
  Features     : 137
  Target values: [0, 1]

Processing VALIDATION sample...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Observations : 5,000
  Features     : 137
  Target values: [0, 1]

Processing TEST sample...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Observations : 5,000
  Features     : 137
  Target values: [0, 1]

FEATURE SCHEMA CHECK
Train transformed features      : 137
Validation transformed features : 137
Test transformed features       : 137

Train = Validation schema      : PASS
Train = Test schema             : PASS

NUMERICAL OUTPUT VALIDATION
train        NULL=0 | Non-finite=0
validation   NULL=0 | Non-finite=0
test         NULL=0 | Non-finite=0

Frequency features:
['zip_code_short_frequency', 'msa_frequency', 'servicer_name_frequency', 'property_state_frequency']
train        min=0.0000003726 | max=0.3600157499
validation   min=0.0000151538 | max=0.3600157499
test         min=0.0000000000 | max=0.3600157499

Target validation:
train        [0, 1]
validation   [0, 1]
test         [0, 1]

Row-preservation validation:
train        X=5,000 | y=5,000
validation   X=5,000 | y=5,000
test         X=5,000 | y=5,000

CORRECTED TEMPORAL PREPROCESSING TEST: PASS
Numeric features       : 46
Fixed one-hot features : 87
Frequency f

## CELL 15 — TRAINING-ONLY PREPROCESSING PARAMETERS

This cell constructs the preprocessing parameters using the training
period only (January 2018 to December 2022).

Low-cardinality categorical variables are represented using fixed
training-derived one-hot categories.

High-cardinality categorical variables are represented using
training-derived frequency encoding:

- zip_code_short
- msa
- servicer_name
- property_state

Numeric missing values are handled using training-period medians.

All preprocessing parameters are estimated exclusively from the training
period and will be frozen before being applied to validation and test data.

This prevents temporal information leakage from validation or test periods.

In [16]:
# ================================================================
# CELL 15 — TRAINING-ONLY PREPROCESSING PARAMETERS
# ================================================================

print("=" * 70)
print("TRAINING-ONLY PREPROCESSING PARAMETER CONSTRUCTION")
print("=" * 70)

import pandas as pd
import numpy as np
import duckdb
import json
import gc

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

FINAL_FEATURE_FILE = (
    PROCESSED_DIR
    / "feature_engineering"
    / "fannie_2018Q1_final_modelling_features.parquet"
)

MODEL_DIR = (
    PROCESSED_DIR
    / "feature_engineering"
    / "model_preprocessing"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

PARAMETER_FILE = (
    MODEL_DIR / "training_preprocessing_parameters.json"
)

print("\nSource:")
print(FINAL_FEATURE_FILE)

print("\nParameter output:")
print(PARAMETER_FILE)

# ------------------------------------------------------------
# 2. Temporal windows
# ------------------------------------------------------------

TRAIN_START = "2018-01-01"
TRAIN_END   = "2022-12-01"

VALID_START = "2023-01-01"
VALID_END   = "2023-12-01"

TEST_START  = "2024-01-01"
TEST_END    = "2025-09-01"

# ------------------------------------------------------------
# 3. Categorical feature design
# ------------------------------------------------------------

# Low-cardinality variables -> one-hot encoding
LOW_CARDINALITY_FEATURES = [
    "channel",
    "seller_name",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag",
]

# High-cardinality variables -> frequency encoding
HIGH_CARDINALITY_FEATURES = [
    "zip_code_short",
    "msa",
    "servicer_name",
    "property_state",
]

CATEGORICAL_FEATURES = (
    LOW_CARDINALITY_FEATURES
    + HIGH_CARDINALITY_FEATURES
)

# ------------------------------------------------------------
# 4. Date features
# ------------------------------------------------------------

DATE_FEATURES = [
    "origination_date",
    "first_payment_date",
]

# ------------------------------------------------------------
# 5. Numerical modelling features
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    # Original numerical predictors
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "original_loan_term",
    "loan_age",
    "remaining_months_legal_maturity",
    "remaining_months_maturity",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score",
    "co_borrower_credit_score",
    "number_of_units",
    "mortgage_insurance_percentage",

    # Payment-history features
    "ph_missing",
    "ph_prior_3m_delinquency",
    "ph_prior_6m_delinquency",
    "ph_prior_12m_delinquency",
    "ph_prior_24m_delinquency",
    "ph_prior_max_delinquency",
    "ph_prior_delinquency_count_6m",
    "ph_prior_delinquency_count_12m",
    "ph_prior_delinquency_count_24m",

    # Financial / loan-derived features
    "balance_utilization",
    "interest_rate_change",
    "ltv_cltv_gap",
    "loan_age_ratio",
    "remaining_term_ratio",
    "co_borrower_score_gap",
    "borrower_credit_score_missing",
    "co_borrower_credit_score_missing",
    "mortgage_insurance_missing",

    # Calendar features
    "reporting_year",
    "reporting_month",

    # Temporal / dynamic features
    "has_valid_prior_month",
    "prior_month_balance",
    "balance_change",
    "balance_change_pct",
    "prior_month_interest_rate",
    "interest_rate_change_1m",
    "prior_month_loan_age",
    "loan_age_change",
    "prior_month_remaining_maturity",
    "remaining_maturity_change",
]

print("\nFeature groups:")
print(f"  Low-cardinality categorical : {len(LOW_CARDINALITY_FEATURES)}")
print(f"  High-cardinality categorical: {len(HIGH_CARDINALITY_FEATURES)}")
print(f"  Numeric features             : {len(NUMERIC_FEATURES)}")
print(f"  Date fields                  : {len(DATE_FEATURES)}")

assert len(LOW_CARDINALITY_FEATURES) == 10
assert len(HIGH_CARDINALITY_FEATURES) == 4
assert len(NUMERIC_FEATURES) == 46

# ------------------------------------------------------------
# 6. Fresh DuckDB connection
# ------------------------------------------------------------

parameter_con = duckdb.connect()

parameter_con.execute("SET memory_limit = '4GB'")
parameter_con.execute("SET threads = 2")
parameter_con.execute("SET preserve_insertion_order = false")

PARAM_TEMP_DIR = MODEL_DIR / "parameter_duckdb_temp"
PARAM_TEMP_DIR.mkdir(parents=True, exist_ok=True)

parameter_con.execute(
    f"SET temp_directory = '{PARAM_TEMP_DIR}'"
)

# ------------------------------------------------------------
# 7. Training row count
# ------------------------------------------------------------

train_count = parameter_con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    WHERE reporting_date >= DATE '{TRAIN_START}'
      AND reporting_date <= DATE '{TRAIN_END}'
    """
).fetchone()[0]

print(f"\nTraining observations: {train_count:,}")

assert train_count == 16_101_651

# ------------------------------------------------------------
# 8. Training numeric medians
# ------------------------------------------------------------

numeric_medians = {}

for col in NUMERIC_FEATURES:

    value = parameter_con.execute(
        f"""
        SELECT median(CAST("{col}" AS DOUBLE))
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        WHERE reporting_date >= DATE '{TRAIN_START}'
          AND reporting_date <= DATE '{TRAIN_END}'
        """
    ).fetchone()[0]

    if value is None or not np.isfinite(float(value)):
        raise ValueError(
            f"Invalid training median for feature: {col}"
        )

    numeric_medians[col] = float(value)

print("\nTraining numeric medians created:")
print(f"  {len(numeric_medians)} features")

# ------------------------------------------------------------
# 9. Training-only LOW-cardinality one-hot schemas
# ------------------------------------------------------------

categorical_levels = {}

print("\nTraining one-hot schemas:")

for col in LOW_CARDINALITY_FEATURES:

    levels_df = parameter_con.execute(
        f"""
        SELECT DISTINCT
            COALESCE(CAST("{col}" AS VARCHAR), '__UNKNOWN__') AS category
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        WHERE reporting_date >= DATE '{TRAIN_START}'
          AND reporting_date <= DATE '{TRAIN_END}'
        ORDER BY category
        """
    ).fetchdf()

    levels = levels_df["category"].astype(str).tolist()

    if "__UNKNOWN__" not in levels:
        levels.append("__UNKNOWN__")

    categorical_levels[col] = levels

    print(
        f"  {col:<35} "
        f"{len(levels):>3} encoded categories"
    )

# ------------------------------------------------------------
# 10. Training-only frequency mappings
# ------------------------------------------------------------

frequency_maps = {}

print("\nTraining frequency mappings:")

for col in HIGH_CARDINALITY_FEATURES:

    freq_df = parameter_con.execute(
        f"""
        SELECT
            COALESCE(CAST("{col}" AS VARCHAR), '__UNKNOWN__') AS category,
            COUNT(*)::DOUBLE / {train_count} AS frequency
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        WHERE reporting_date >= DATE '{TRAIN_START}'
          AND reporting_date <= DATE '{TRAIN_END}'
        GROUP BY 1
        ORDER BY 1
        """
    ).fetchdf()

    frequency_maps[col] = {
        str(row["category"]): float(row["frequency"])
        for _, row in freq_df.iterrows()
    }

    print(
        f"  {col:<35} "
        f"{len(frequency_maps[col]):>3} categories"
    )

# ------------------------------------------------------------
# 11. Expected transformed feature count
# ------------------------------------------------------------

fixed_one_hot_count = sum(
    len(categorical_levels[col])
    for col in LOW_CARDINALITY_FEATURES
)

frequency_feature_count = len(HIGH_CARDINALITY_FEATURES)

expected_transformed_features = (
    len(NUMERIC_FEATURES)
    + fixed_one_hot_count
    + frequency_feature_count
)

print("\nPreprocessing feature structure:")
print(f"  Numeric features       : {len(NUMERIC_FEATURES)}")
print(f"  Fixed one-hot features : {fixed_one_hot_count}")
print(f"  Frequency features     : {frequency_feature_count}")
print(
    f"  Expected transformed   : "
    f"{expected_transformed_features}"
)

# ------------------------------------------------------------
# 12. Save parameters
# ------------------------------------------------------------

parameters = {
    "train_start": TRAIN_START,
    "train_end": TRAIN_END,
    "valid_start": VALID_START,
    "valid_end": VALID_END,
    "test_start": TEST_START,
    "test_end": TEST_END,

    "low_cardinality_features": LOW_CARDINALITY_FEATURES,
    "high_cardinality_features": HIGH_CARDINALITY_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "date_features": DATE_FEATURES,

    "numeric_medians": numeric_medians,
    "categorical_levels": categorical_levels,
    "frequency_maps": frequency_maps,

    "fixed_one_hot_count": fixed_one_hot_count,
    "frequency_feature_count": frequency_feature_count,
    "expected_transformed_features": expected_transformed_features,
}

with open(PARAMETER_FILE, "w", encoding="utf-8") as f:
    json.dump(parameters, f, indent=2)

print("\nParameter file:")
print(f"  Exists: {PARAMETER_FILE.exists()}")
print(
    f"  Size: "
    f"{PARAMETER_FILE.stat().st_size / 1024:.2f} KB"
)

assert PARAMETER_FILE.exists()

# ------------------------------------------------------------
# 13. Final audit
# ------------------------------------------------------------

assert len(numeric_medians) == 46
assert len(categorical_levels) == 10
assert len(frequency_maps) == 4

for col in HIGH_CARDINALITY_FEATURES:
    assert len(frequency_maps[col]) > 0

for col in LOW_CARDINALITY_FEATURES:
    assert len(categorical_levels[col]) > 0

assert expected_transformed_features > 0

print("\n" + "=" * 70)
print("TRAINING-ONLY PARAMETER AUDIT: PASS")
print("=" * 70)

TRAINING-ONLY PREPROCESSING PARAMETER CONSTRUCTION

Source:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_final_modelling_features.parquet

Parameter output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\model_preprocessing\training_preprocessing_parameters.json

Feature groups:
  Low-cardinality categorical : 10
  High-cardinality categorical: 4
  Numeric features             : 46
  Date fields                  : 2

Training observations: 16,101,651


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (11 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Training numeric medians created:
  46 features

Training one-hot schemas:
  channel                               4 encoded categories
  seller_name                          47 encoded categories
  first_time_home_buyer                 3 encoded categories
  loan_purpose                          4 encoded categories
  property_type                         6 encoded categories
  occupancy_status                      4 encoded categories
  amortization_type                     2 encoded categories
  prepayment_penalty_indicator          2 encoded categories
  interest_only_indicator               2 encoded categories
  modification_flag                     3 encoded categories

Training frequency mappings:
  zip_code_short                      889 categories
  msa                                 395 categories
  servicer_name                        82 categories
  property_state                       54 categories

Preprocessing feature structure:
  Numeric features       : 46
  Fixed 

## Training-Only Preprocessing Parameter Recovery

The final modelling dataset has been successfully created and validated.
However, the training-only preprocessing parameter file is not currently
available at the expected path.

This cell first searches the project for an existing parameter file.
If none is found, it reconstructs the training-only preprocessing parameters
from the TRAINING period only (January 2018–December 2022).

No validation or test information is used to estimate preprocessing
parameters.

The resulting parameter file will be saved to the model_preprocessing folder
and validated before continuing.

In [19]:
# ============================================================
# CELL 18 — TRAINING-ONLY PREPROCESSING PARAMETER RECOVERY
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
import duckdb
import gc

print("=" * 70)
print("TRAINING-ONLY PREPROCESSING PARAMETER RECOVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FEATURE_DIR = PROCESSED_DIR / "feature_engineering"
MODEL_DIR = PROCESSED_DIR / "model_preprocessing"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_FEATURE_FILE = (
    FEATURE_DIR / "fannie_2018Q1_final_modelling_features.parquet"
)

PARAMETER_FILE = (
    MODEL_DIR / "training_preprocessing_parameters.json"
)

print("\nFinal feature dataset:")
print(FINAL_FEATURE_FILE)
print("Exists:", FINAL_FEATURE_FILE.exists())

# ------------------------------------------------------------
# 2. SEARCH FOR EXISTING PARAMETER FILE
# ------------------------------------------------------------

print("\nSearching for existing preprocessing parameter files...")

parameter_candidates = list(
    PROJECT_DIR.rglob("*preprocessing*parameter*.json")
)

if parameter_candidates:

    print("\nParameter files found:")

    for p in parameter_candidates:
        print(" -", p)

else:
    print("No existing preprocessing parameter JSON found.")

# ------------------------------------------------------------
# 3. IF EXACT FILE EXISTS, VALIDATE IT
# ------------------------------------------------------------

if PARAMETER_FILE.exists():

    print("\nExpected parameter file already exists.")
    print(PARAMETER_FILE)

    with open(PARAMETER_FILE, "r", encoding="utf-8") as f:
        params = json.load(f)

    print("\nExisting parameter file loaded successfully.")

# ------------------------------------------------------------
# 4. OTHERWISE REBUILD TRAINING-ONLY PARAMETERS
# ------------------------------------------------------------

else:

    print("\nParameter file is missing.")
    print("Rebuilding from TRAINING data only...")

    # --------------------------------------------------------
    # DuckDB setup
    # --------------------------------------------------------

    TEMP_DIR = MODEL_DIR / "parameter_rebuild_temp"
    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()

    con.execute("SET memory_limit = '4GB'")
    con.execute("SET threads = 2")
    con.execute("SET preserve_insertion_order = false")
    con.execute(
        f"SET temp_directory = '{TEMP_DIR}'"
    )

    # --------------------------------------------------------
    # Identify modelling columns
    # --------------------------------------------------------

    schema_df = con.execute(
        f"""
        DESCRIBE SELECT *
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        """
    ).fetchdf()

    all_columns = schema_df["column_name"].tolist()

    print("\nTotal source columns:", len(all_columns))

    # --------------------------------------------------------
    # Feature groups
    # --------------------------------------------------------

    categorical_features = [
        "channel",
        "seller_name",
        "first_time_home_buyer",
        "loan_purpose",
        "property_type",
        "occupancy_status",
        "amortization_type",
        "prepayment_penalty_indicator",
        "interest_only_indicator",
        "modification_flag",
        "zip_code_short",
        "msa",
        "servicer_name",
        "property_state"
    ]

    numeric_features = [
        c for c in all_columns
        if c not in categorical_features
        and c not in [
            "loan_id",
            "reporting_date",
            "target_6m"
        ]
        and not (
            c.endswith("_frequency")
        )
        and c != "maturity_date"
    ]

    # Remove date-derived fields from numeric list if present
    numeric_features = [
        c for c in numeric_features
        if c not in [
            "origination_date",
            "first_payment_date"
        ]
    ]

    high_cardinality_features = [
        "zip_code_short",
        "msa",
        "servicer_name",
        "property_state"
    ]

    low_cardinality_features = [
        c for c in categorical_features
        if c not in high_cardinality_features
    ]

    print("\nFeature groups:")
    print("Categorical features :", len(categorical_features))
    print("Numeric features     :", len(numeric_features))
    print("High-cardinality     :", len(high_cardinality_features))
    print("Low-cardinality      :", len(low_cardinality_features))

    # --------------------------------------------------------
    # Training-only medians
    # --------------------------------------------------------

    print("\nCalculating TRAINING-only numeric medians...")

    median_expr = ",\n".join(
        [
            f"MEDIAN({c}) AS {c}"
            for c in numeric_features
        ]
    )

    median_query = f"""
    SELECT
        {median_expr}
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    WHERE reporting_date BETWEEN DATE '2018-01-01'
                              AND DATE '2022-12-01'
    """

    median_df = con.execute(median_query).fetchdf()

    numeric_medians = {}

    for c in numeric_features:

        value = median_df.loc[0, c]

        if pd.isna(value):
            numeric_medians[c] = None
        else:
            numeric_medians[c] = float(value)

    print(
        "Training numeric medians created:",
        len(numeric_medians)
    )

    # --------------------------------------------------------
    # Training-only categorical schemas
    # --------------------------------------------------------

    print("\nBuilding TRAINING-only categorical schemas...")

    categorical_schema = {}

    for c in categorical_features:

        query = f"""
        SELECT DISTINCT
            CAST({c} AS VARCHAR) AS value
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        WHERE reporting_date BETWEEN DATE '2018-01-01'
                                  AND DATE '2022-12-01'
          AND {c} IS NOT NULL
        ORDER BY value
        """

        values_df = con.execute(query).fetchdf()

        values = values_df["value"].tolist()

        # Reserve an explicit unknown category
        if "__UNKNOWN__" not in values:
            values.append("__UNKNOWN__")

        categorical_schema[c] = values

        print(
            f"  {c}: {len(values)} encoded categories"
        )

    # --------------------------------------------------------
    # Training-only frequency mappings
    # --------------------------------------------------------

    print("\nBuilding TRAINING-only frequency mappings...")

    frequency_mappings = {}

    for c in high_cardinality_features:

        query = f"""
        SELECT
            CAST({c} AS VARCHAR) AS value,
            COUNT(*)::DOUBLE /
            SUM(COUNT(*)) OVER () AS frequency
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        WHERE reporting_date BETWEEN DATE '2018-01-01'
                                  AND DATE '2022-12-01'
          AND {c} IS NOT NULL
        GROUP BY {c}
        ORDER BY value
        """

        freq_df = con.execute(query).fetchdf()

        mapping = {}

        for _, row in freq_df.iterrows():
            mapping[str(row["value"])] = float(row["frequency"])

        # Unknown/unseen category frequency
        mapping["__UNKNOWN__"] = 0.0

        frequency_mappings[c] = mapping

        print(
            f"  {c}: {len(mapping)} frequency values"
        )

    # --------------------------------------------------------
    # Expected transformed structure
    # --------------------------------------------------------

    numeric_output_features = len(numeric_features)

    one_hot_features = sum(
        len(categorical_schema[c])
        for c in low_cardinality_features
    )

    frequency_features = len(high_cardinality_features)

    expected_transformed_features = (
        numeric_output_features
        + one_hot_features
        + frequency_features
    )

    print("\nPreprocessing feature structure:")
    print("Numeric features       :", numeric_output_features)
    print("One-hot features       :", one_hot_features)
    print("Frequency features     :", frequency_features)
    print(
        "Expected transformed features:",
        expected_transformed_features
    )

    # --------------------------------------------------------
    # Parameter object
    # --------------------------------------------------------

    params = {
        "metadata": {
            "parameter_source": "training_only",
            "training_start": "2018-01-01",
            "training_end": "2022-12-01",
            "validation_start": "2023-01-01",
            "validation_end": "2023-12-01",
            "test_start": "2024-01-01",
            "test_end": "2025-09-01"
        },

        "feature_groups": {
            "categorical_features": categorical_features,
            "numeric_features": numeric_features,
            "high_cardinality_features": high_cardinality_features,
            "low_cardinality_features": low_cardinality_features
        },

        "numeric_medians": numeric_medians,

        "categorical_schema": categorical_schema,

        "frequency_mappings": frequency_mappings,

        "expected_structure": {
            "numeric_features": numeric_output_features,
            "one_hot_features": one_hot_features,
            "frequency_features": frequency_features,
            "expected_transformed_features":
                expected_transformed_features
        }
    }

    # --------------------------------------------------------
    # Save parameter file
    # --------------------------------------------------------

    with open(
        PARAMETER_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            params,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\nParameter file saved:")
    print(PARAMETER_FILE)

    con.close()
    gc.collect()

# ------------------------------------------------------------
# 5. FINAL PARAMETER AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PARAMETER FILE AUDIT")
print("=" * 70)

assert PARAMETER_FILE.exists(), (
    "Parameter file was not successfully created."
)

with open(
    PARAMETER_FILE,
    "r",
    encoding="utf-8"
) as f:

    params = json.load(f)

required_groups = [
    "metadata",
    "feature_groups",
    "numeric_medians",
    "categorical_schema",
    "frequency_mappings",
    "expected_structure"
]

missing_groups = [
    g for g in required_groups
    if g not in params
]

print("Parameter file exists :", PARAMETER_FILE.exists())
print("Parameter groups      :", len(params))
print("Missing groups        :", len(missing_groups))

if missing_groups:
    print("Missing:", missing_groups)

assert len(missing_groups) == 0

print("\nParameter file size:")
print(
    f"{PARAMETER_FILE.stat().st_size / 1024:.2f} KB"
)

print("\n" + "=" * 70)
print("TRAINING-ONLY PARAMETER RECOVERY: PASS")
print("=" * 70)

TRAINING-ONLY PREPROCESSING PARAMETER RECOVERY

Final feature dataset:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_final_modelling_features.parquet
Exists: True

Searching for existing preprocessing parameter files...

Parameter files found:
 - C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\model_preprocessing\training_preprocessing_parameters.json

Parameter file is missing.
Rebuilding from TRAINING data only...

Total source columns: 66

Feature groups:
Categorical features : 14
Numeric features     : 46
High-cardinality     : 4
Low-cardinality      : 10

Calculating TRAINING-only numeric medians...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training numeric medians created: 46

Building TRAINING-only categorical schemas...
  channel: 4 encoded categories
  seller_name: 47 encoded categories
  first_time_home_buyer: 3 encoded categories
  loan_purpose: 4 encoded categories
  property_type: 6 encoded categories
  occupancy_status: 4 encoded categories
  amortization_type: 2 encoded categories
  prepayment_penalty_indicator: 2 encoded categories
  interest_only_indicator: 2 encoded categories
  modification_flag: 3 encoded categories
  zip_code_short: 889 encoded categories
  msa: 396 encoded categories
  servicer_name: 83 encoded categories
  property_state: 55 encoded categories

Building TRAINING-only frequency mappings...
  zip_code_short: 889 frequency values
  msa: 396 frequency values
  servicer_name: 83 frequency values
  property_state: 55 frequency values

Preprocessing feature structure:
Numeric features       : 46
One-hot features       : 77
Frequency features     : 4
Expected transformed features: 127

Parameter

## Full Temporal Model Preprocessing

This cell applies the previously constructed TRAINING-ONLY preprocessing parameters
to the complete final modelling dataset.

The preprocessing parameters were estimated exclusively from the training period
(January 2018–December 2022). The same fixed parameters are applied unchanged to
the validation and test periods to prevent temporal information leakage.

The transformation produces the same 127-feature schema for all three temporal
partitions:

- 46 numerical features
- 77 fixed one-hot categorical features
- 4 frequency-encoded high-cardinality features

Outputs:
1. Training transformed dataset
2. Validation transformed dataset
3. Test transformed dataset

No target-derived preprocessing is performed.

In [20]:
# ================================================================
# FULL TEMPORAL MODEL PREPROCESSING
# ================================================================

from pathlib import Path
import json
import duckdb
import pandas as pd
import numpy as np
import gc

print("=" * 70)
print("FULL TEMPORAL MODEL PREPROCESSING")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

FINAL_FEATURE_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "feature_engineering"
    / "fannie_2018Q1_final_modelling_features.parquet"
)

MODEL_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
)

PARAMETER_FILE = MODEL_DIR / "training_preprocessing_parameters.json"

OUTPUT_DIR = MODEL_DIR / "transformed_datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_OUTPUT = OUTPUT_DIR / "train_transformed.parquet"
VALIDATION_OUTPUT = OUTPUT_DIR / "validation_transformed.parquet"
TEST_OUTPUT = OUTPUT_DIR / "test_transformed.parquet"

assert FINAL_FEATURE_FILE.exists(), (
    f"Final feature dataset not found:\n{FINAL_FEATURE_FILE}"
)

assert PARAMETER_FILE.exists(), (
    f"Training preprocessing parameter file not found:\n{PARAMETER_FILE}"
)

print(f"\nSource dataset:")
print(FINAL_FEATURE_FILE)

print(f"\nTraining-only parameters:")
print(PARAMETER_FILE)

print(f"\nOutput directory:")
print(OUTPUT_DIR)


# ------------------------------------------------
# 2. Load training-only parameters
# ------------------------------------------------

with open(PARAMETER_FILE, "r", encoding="utf-8") as f:
    params = json.load(f)

print("\nTraining-only parameter file loaded.")

print("Parameter groups:")
for key in params.keys():
    print(f"  - {key}")


# ------------------------------------------------
# 3. Inspect parameter structure
# ------------------------------------------------

print("\nParameter structure validation:")

assert isinstance(params, dict), "Parameter file must contain a dictionary."

print(f"Top-level parameter groups: {len(params)}")


# ------------------------------------------------
# 4. Connect DuckDB
# ------------------------------------------------

model_con = duckdb.connect()

model_temp_dir = MODEL_DIR / "full_preprocessing_duckdb_temp"
model_temp_dir.mkdir(parents=True, exist_ok=True)

model_con.execute("SET memory_limit = '4GB'")
model_con.execute("SET threads = 2")
model_con.execute("SET preserve_insertion_order = false")
model_con.execute(
    f"SET temp_directory = '{model_temp_dir}'"
)

print("\nDuckDB configured:")
print("  Memory limit : 4GB")
print("  Threads      : 2")
print(f"  Temp dir     : {model_temp_dir}")


# ------------------------------------------------
# 5. Verify source dataset
# ------------------------------------------------

source_check = model_con.execute(
    f"""
    SELECT
        COUNT(*) AS observations,
        COUNT(DISTINCT loan_id) AS unique_loans,
        SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END) AS events,
        SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
        MIN(reporting_date) AS min_date,
        MAX(reporting_date) AS max_date
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    """
).fetchdf()

print("\nSource dataset check:")
display(source_check)


# ------------------------------------------------
# 6. Define temporal partitions
# ------------------------------------------------

splits = {
    "train": {
        "start": "2018-01-01",
        "end": "2022-12-01",
        "output": TRAIN_OUTPUT
    },
    "validation": {
        "start": "2023-01-01",
        "end": "2023-12-01",
        "output": VALIDATION_OUTPUT
    },
    "test": {
        "start": "2024-01-01",
        "end": "2025-09-01",
        "output": TEST_OUTPUT
    }
}

print("\nTemporal partitions:")
for name, spec in splits.items():
    print(
        f"  {name:<12}: "
        f"{spec['start']} to {spec['end']}"
    )


# ------------------------------------------------
# 7. IMPORTANT
# ------------------------------------------------
# The parameter file was constructed from the training data.
# The actual transformation implementation must use the exact
# parameter schema recovered above.
#
# Before writing 20M rows, inspect the saved parameter structure.
# This prevents another expensive run if a parameter-key mismatch exists.
# ------------------------------------------------

print("\nParameter keys and object sizes:")

for key, value in params.items():

    if isinstance(value, dict):
        print(
            f"  {key:<30} "
            f"dict with {len(value)} entries"
        )

    elif isinstance(value, list):
        print(
            f"  {key:<30} "
            f"list with {len(value)} entries"
        )

    else:
        print(
            f"  {key:<30} "
            f"{type(value).__name__}"
        )


print("\n" + "=" * 70)
print("PARAMETER STRUCTURE CHECK COMPLETE")
print("=" * 70)
print(
    "\nDo NOT start the 20M-row transformation yet.\n"
    "Send me this output first so I can give you the exact "
    "transformation cell matching the recovered JSON structure."
)

FULL TEMPORAL MODEL PREPROCESSING

Source dataset:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_final_modelling_features.parquet

Training-only parameters:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\training_preprocessing_parameters.json

Output directory:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets

Training-only parameter file loaded.
Parameter groups:
  - metadata
  - feature_groups
  - numeric_medians
  - categorical_schema
  - frequency_mappings
  - expected_structure

Parameter structure validation:
Top-level parameter groups: 6

DuckDB configured:
  Memory limit : 4GB
  Threads      : 2
  Temp dir     : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\full_preprocessing_duckdb_temp

Source dataset check:


,observations,unique_loans,events,non_events,min_date,max_date
0,20155269,450106,646957.0,19508312.0,2018-01-01,2025-09-01



Temporal partitions:
  train       : 2018-01-01 to 2022-12-01
  validation  : 2023-01-01 to 2023-12-01
  test        : 2024-01-01 to 2025-09-01

Parameter keys and object sizes:
  metadata                       dict with 7 entries
  feature_groups                 dict with 4 entries
  numeric_medians                dict with 46 entries
  categorical_schema             dict with 14 entries
  frequency_mappings             dict with 4 entries
  expected_structure             dict with 4 entries

PARAMETER STRUCTURE CHECK COMPLETE

Do NOT start the 20M-row transformation yet.
Send me this output first so I can give you the exact transformation cell matching the recovered JSON structure.


## FULL TEMPORAL MODEL PREPROCESSING

This cell applies the validated TRAINING-ONLY preprocessing parameters
to the complete final modelling dataset.

The transformation is applied consistently across:
- Training: Jan 2018 – Dec 2022
- Validation: Jan 2023 – Dec 2023
- Test: Jan 2024 – Sep 2025

Preprocessing rules:
1. Numeric missing values use training-set medians only.
2. Low-cardinality categorical variables use the fixed training one-hot schema.
3. High-cardinality categorical variables use training-derived frequency mappings.
4. Date fields are represented through engineered calendar information.
5. The target variable is preserved separately and is never used as a preprocessing input.
6. The resulting train/validation/test datasets must have identical feature schemas.
7. Row counts and chronological partitions must remain unchanged.

The transformation is executed with DuckDB disk spilling to avoid loading
the complete ~20.15M-row dataset into pandas memory.

In [21]:
# ================================================================
# CELL 23 — FULL TEMPORAL MODEL PREPROCESSING
# ================================================================

from pathlib import Path
import json
import duckdb
import pandas as pd
import numpy as np
import gc

print("=" * 70)
print("FULL TEMPORAL MODEL PREPROCESSING")
print("=" * 70)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

FINAL_FEATURE_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "feature_engineering"
    / "fannie_2018Q1_final_modelling_features.parquet"
)

PARAMETER_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "training_preprocessing_parameters.json"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "transformed_datasets"
)

TEMP_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "full_preprocessing_duckdb_temp"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_OUTPUT = OUTPUT_DIR / "train_transformed.parquet"
VALIDATION_OUTPUT = OUTPUT_DIR / "validation_transformed.parquet"
TEST_OUTPUT = OUTPUT_DIR / "test_transformed.parquet"

print(f"\nSource dataset:")
print(FINAL_FEATURE_FILE)

print(f"\nTraining-only parameters:")
print(PARAMETER_FILE)

print(f"\nOutput directory:")
print(OUTPUT_DIR)

# ------------------------------------------------------------
# 2. EXISTENCE CHECKS
# ------------------------------------------------------------

assert FINAL_FEATURE_FILE.exists(), (
    f"Final feature dataset not found:\n{FINAL_FEATURE_FILE}"
)

assert PARAMETER_FILE.exists(), (
    f"Training preprocessing parameters not found:\n{PARAMETER_FILE}"
)

print("\nRequired files:")
print(f"  Final dataset exists     : {FINAL_FEATURE_FILE.exists()}")
print(f"  Parameter file exists    : {PARAMETER_FILE.exists()}")

# ------------------------------------------------------------
# 3. LOAD TRAINING-ONLY PARAMETERS
# ------------------------------------------------------------

with open(PARAMETER_FILE, "r", encoding="utf-8") as f:
    params = json.load(f)

print("\nParameter groups loaded:")
for key in params.keys():
    print(f"  - {key}")

assert "numeric_medians" in params
assert "categorical_schema" in params
assert "frequency_mappings" in params
assert "expected_structure" in params

# ------------------------------------------------------------
# 4. READ FEATURE GROUP DEFINITIONS
# ------------------------------------------------------------

feature_groups = params.get("feature_groups", {})

numeric_features = feature_groups.get(
    "numeric_features",
    []
)

categorical_features = feature_groups.get(
    "categorical_features",
    []
)

date_features = feature_groups.get(
    "date_features",
    []
)

high_cardinality_features = feature_groups.get(
    "high_cardinality_features",
    []
)

low_cardinality_features = [
    c for c in categorical_features
    if c not in high_cardinality_features
]

print("\nFeature groups:")
print(f"  Numeric features              : {len(numeric_features)}")
print(f"  Low-cardinality categorical   : {len(low_cardinality_features)}")
print(f"  High-cardinality categorical  : {len(high_cardinality_features)}")
print(f"  Date features                 : {len(date_features)}")

# ------------------------------------------------------------
# 5. FIXED TRAINING CATEGORICAL SCHEMA
# ------------------------------------------------------------

categorical_schema = params["categorical_schema"]

print("\nTraining-derived categorical schemas:")

for col, categories in categorical_schema.items():
    print(f"  {col}: {len(categories)} encoded categories")

# ------------------------------------------------------------
# 6. TRAINING NUMERIC MEDIANS
# ------------------------------------------------------------

numeric_medians = params["numeric_medians"]

assert len(numeric_medians) == len(numeric_features), (
    "Mismatch between numeric feature list and training medians."
)

print(
    f"\nTraining numeric medians available: "
    f"{len(numeric_medians)}"
)

# ------------------------------------------------------------
# 7. TRAINING FREQUENCY MAPPINGS
# ------------------------------------------------------------

frequency_mappings = params["frequency_mappings"]

print("\nTraining-derived frequency mappings:")

for col, mapping in frequency_mappings.items():
    print(f"  {col}: {len(mapping)} categories")

# ------------------------------------------------------------
# 8. IDENTIFY TARGET / CORE FIELDS
# ------------------------------------------------------------

TARGET_COLUMN = "target_6m"
ID_COLUMN = "loan_id"
DATE_COLUMN = "reporting_date"

# ------------------------------------------------------------
# 9. CREATE FRESH DUCKDB CONNECTION
# ------------------------------------------------------------

# Important:
# Use a fresh connection before setting temp_directory.

model_con = duckdb.connect()

model_con.execute("SET memory_limit = '4GB'")
model_con.execute("SET threads = 2")
model_con.execute("SET preserve_insertion_order = false")
model_con.execute(
    f"SET temp_directory = '{TEMP_DIR}'"
)

print("\nDuckDB configuration:")
print("  Memory limit : 4GB")
print("  Threads      : 2")
print(f"  Temp dir     : {TEMP_DIR}")

# ------------------------------------------------------------
# 10. SOURCE DATASET VALIDATION
# ------------------------------------------------------------

source_check = model_con.execute(f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    SUM(target_6m) AS events,
    SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
    COUNT(DISTINCT reporting_date) AS reporting_months,
    MIN(reporting_date) AS min_date,
    MAX(reporting_date) AS max_date
FROM read_parquet('{FINAL_FEATURE_FILE}')
""").fetchdf()

print("\nSource dataset:")
display(source_check)

# ------------------------------------------------------------
# 11. DEFINE TEMPORAL WINDOWS
# ------------------------------------------------------------

split_case = f"""
CASE
    WHEN reporting_date >= DATE '2018-01-01'
     AND reporting_date <  DATE '2023-01-01'
        THEN 'train'

    WHEN reporting_date >= DATE '2023-01-01'
     AND reporting_date <  DATE '2024-01-01'
        THEN 'validation'

    WHEN reporting_date >= DATE '2024-01-01'
     AND reporting_date <= DATE '2025-09-01'
        THEN 'test'

    ELSE NULL
END
"""

# ------------------------------------------------------------
# 12. BUILD NUMERIC TRANSFORM EXPRESSIONS
# ------------------------------------------------------------

numeric_exprs = []

for col in numeric_features:

    median_value = numeric_medians[col]

    if median_value is None or not np.isfinite(float(median_value)):
        raise ValueError(
            f"Invalid training median for numeric feature: {col}"
        )

    numeric_exprs.append(
        f"COALESCE(CAST({col} AS DOUBLE), {float(median_value)}) "
        f'AS "{col}"'
    )

# ------------------------------------------------------------
# 13. BUILD LOW-CARDINALITY ONE-HOT EXPRESSIONS
# ------------------------------------------------------------

onehot_exprs = []

for col in low_cardinality_features:

    categories = categorical_schema[col]

    for category in categories:

        # Convert parameter category safely to string.
        category_str = str(category)

        # Escape SQL single quotes.
        escaped = category_str.replace("'", "''")

        feature_name = (
            f"{col}_{category_str}"
            .replace(" ", "_")
        )

        # Keep the original category in the comparison.
        onehot_exprs.append(
            f"""
            CASE
                WHEN CAST({col} AS VARCHAR) = '{escaped}'
                THEN 1
                ELSE 0
            END AS "{feature_name}"
            """
        )

# ------------------------------------------------------------
# 14. BUILD HIGH-CARDINALITY FREQUENCY FEATURES
# ------------------------------------------------------------

frequency_exprs = []

for col in high_cardinality_features:

    mapping = frequency_mappings[col]

    # Create a CASE expression from training-only frequencies.
    cases = []

    for category, frequency in mapping.items():

        category_str = str(category)
        escaped = category_str.replace("'", "''")

        if frequency is None:
            continue

        cases.append(
            f"WHEN CAST({col} AS VARCHAR) = '{escaped}' "
            f"THEN {float(frequency)}"
        )

    if cases:
        expr = (
            "CASE "
            + " ".join(cases)
            + " ELSE 0.0 END "
            + f'AS "{col}_frequency"'
        )
    else:
        expr = f"0.0 AS \"{col}_frequency\""

    frequency_exprs.append(expr)

# ------------------------------------------------------------
# 15. DATE FEATURES
# ------------------------------------------------------------

date_exprs = []

# reporting_date is the primary temporal index.
# Convert it into numeric calendar information rather than
# passing the raw timestamp to the model.

date_exprs.extend([
    'EXTRACT(YEAR FROM reporting_date)::DOUBLE AS reporting_year',
    'EXTRACT(MONTH FROM reporting_date)::DOUBLE AS reporting_month'
])

# ------------------------------------------------------------
# 16. TARGET AND IDENTIFIER
# ------------------------------------------------------------

core_exprs = [
    'loan_id',
    'reporting_date',
    'target_6m'
]

# ------------------------------------------------------------
# 17. COMBINE TRANSFORMED FEATURES
# ------------------------------------------------------------

all_feature_exprs = (
    core_exprs
    + numeric_exprs
    + onehot_exprs
    + frequency_exprs
    + date_exprs
)

print("\nTransformation structure:")
print(f"  Core fields                  : {len(core_exprs)}")
print(f"  Numeric transformed features: {len(numeric_exprs)}")
print(f"  One-hot features             : {len(onehot_exprs)}")
print(f"  Frequency features           : {len(frequency_exprs)}")
print(f"  Date features                : {len(date_exprs)}")
print(
    f"  Total output columns         : "
    f"{len(all_feature_exprs)}"
)

# ------------------------------------------------------------
# 18. BUILD SELECT CLAUSE
# ------------------------------------------------------------

select_clause = ",\n        ".join(all_feature_exprs)

# ------------------------------------------------------------
# 19. PROCESS EACH TEMPORAL SPLIT
# ------------------------------------------------------------

split_definitions = [
    (
        "train",
        TRAIN_OUTPUT,
        "reporting_date >= DATE '2018-01-01' "
        "AND reporting_date <= DATE '2022-12-01'"
    ),
    (
        "validation",
        VALIDATION_OUTPUT,
        "reporting_date >= DATE '2023-01-01' "
        "AND reporting_date <= DATE '2023-12-01'"
    ),
    (
        "test",
        TEST_OUTPUT,
        "reporting_date >= DATE '2024-01-01' "
        "AND reporting_date <= DATE '2025-09-01'"
    )
]

print("\n" + "=" * 70)
print("STARTING FULL TEMPORAL TRANSFORMATION")
print("=" * 70)

for split_name, output_file, date_condition in split_definitions:

    print(f"\nProcessing {split_name.upper()}...")
    print(f"Output: {output_file}")

    if output_file.exists():
        output_file.unlink()

    query = f"""
    COPY (
        SELECT
            {select_clause}
        FROM read_parquet('{FINAL_FEATURE_FILE}')
        WHERE {date_condition}
    )
    TO '{output_file}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """

    model_con.execute(query)

    print(f"  Completed: {output_file.exists()}")

# ------------------------------------------------------------
# 20. VALIDATE OUTPUT FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRANSFORMED DATASET VALIDATION")
print("=" * 70)

output_files = {
    "train": TRAIN_OUTPUT,
    "validation": VALIDATION_OUTPUT,
    "test": TEST_OUTPUT
}

for split_name, output_file in output_files.items():

    assert output_file.exists(), (
        f"{split_name} output file was not created."
    )

    validation = model_con.execute(f"""
    SELECT
        COUNT(*) AS observations,
        COUNT(DISTINCT loan_id) AS unique_loans,
        SUM(target_6m) AS events,
        SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END)
            AS non_events,
        COUNT(DISTINCT reporting_date) AS reporting_months,
        MIN(reporting_date) AS min_date,
        MAX(reporting_date) AS max_date,
        COUNT(*) FILTER (
            WHERE target_6m IS NULL
        ) AS null_target
    FROM read_parquet('{output_file}')
    """).fetchdf()

    print(f"\n{split_name.upper()}:")
    display(validation)

# ------------------------------------------------------------
# 21. CHECK IDENTICAL SCHEMAS
# ------------------------------------------------------------

schema_queries = {}

for split_name, output_file in output_files.items():

    schema_df = model_con.execute(f"""
    SELECT column_name, column_type
    FROM (
        DESCRIBE SELECT *
        FROM read_parquet('{output_file}')
    )
    ORDER BY column_name
    """).fetchdf()

    schema_queries[split_name] = schema_df

train_schema = schema_queries["train"]
validation_schema = schema_queries["validation"]
test_schema = schema_queries["test"]

train_schema_check = train_schema.equals(validation_schema)
train_test_schema_check = train_schema.equals(test_schema)

print("\nSchema consistency:")
print(f"  Train = Validation : {train_schema_check}")
print(f"  Train = Test       : {train_test_schema_check}")

# ------------------------------------------------------------
# 22. NUMERIC NULL / NON-FINITE CHECK
# ------------------------------------------------------------

for split_name, output_file in output_files.items():

    null_check = model_con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        SUM(
            CASE
                WHEN target_6m IS NULL THEN 1
                ELSE 0
            END
        ) AS null_target
    FROM read_parquet('{output_file}')
    """).fetchdf()

    print(f"\n{split_name.upper()} null check:")
    display(null_check)

# ------------------------------------------------------------
# 23. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FULL TEMPORAL MODEL PREPROCESSING RESULT")
print("=" * 70)

all_outputs_exist = all(
    path.exists()
    for path in output_files.values()
)

assert all_outputs_exist
assert train_schema_check
assert train_test_schema_check

print("Output files exist             : PASS")
print("Train/validation schema        : PASS")
print("Train/test schema              : PASS")

print("\nTraining output:")
print(TRAIN_OUTPUT)

print("\nValidation output:")
print(VALIDATION_OUTPUT)

print("\nTest output:")
print(TEST_OUTPUT)

print("\n" + "=" * 70)
print("OVERALL: PASS")
print("=" * 70)

# Close connection only after all validation is complete.
model_con.close()

gc.collect()

FULL TEMPORAL MODEL PREPROCESSING

Source dataset:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_final_modelling_features.parquet

Training-only parameters:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\training_preprocessing_parameters.json

Output directory:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets

Required files:
  Final dataset exists     : True
  Parameter file exists    : True

Parameter groups loaded:
  - metadata
  - feature_groups
  - numeric_medians
  - categorical_schema
  - frequency_mappings
  - expected_structure

Feature groups:
  Numeric features              : 46
  Low-cardinality categorical   : 10
  High-cardinality categorical  : 4
  Date features                 : 0

Training-derived categorical schemas:
  channel: 4 encoded categories
  seller_name: 47 encoded cate

,observations,unique_loans,events,non_events,reporting_months,min_date,max_date
0,20155269,450106,646957.0,19508312.0,93,2018-01-01,2025-09-01



Transformation structure:
  Core fields                  : 3
  Numeric transformed features: 46
  One-hot features             : 77
  Frequency features           : 4
  Date features                : 2
  Total output columns         : 132

STARTING FULL TEMPORAL TRANSFORMATION

Processing TRAIN...
Output: C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets\train_transformed.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Completed: True

Processing VALIDATION...
Output: C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets\validation_transformed.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (12 outputs left out)


TEST null check:


,rows,null_target
0,2483739,0.0



FULL TEMPORAL MODEL PREPROCESSING RESULT
Output files exist             : PASS
Train/validation schema        : PASS
Train/test schema              : PASS

Training output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets\train_transformed.parquet

Validation output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets\validation_transformed.parquet

Test output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets\test_transformed.parquet

OVERALL: PASS


0

## FINAL PREPROCESSED DATASET AUDIT

This cell performs the final integrity audit of the fully transformed
training, validation, and test datasets before model development.

The audit verifies:

1. Exact row counts against the validated temporal split.
2. Exact event and non-event counts.
3. Correct temporal coverage.
4. Identical schemas across train, validation, and test.
5. Binary target integrity.
6. No NULL targets.
7. No NULL or non-finite predictor values.
8. No target or identifier leakage into the predictor matrix.
9. No raw categorical predictor columns remain after preprocessing.
10. Expected transformed feature count.

No model fitting or parameter estimation is performed in this cell.

In [22]:
# ================================================================
# CELL 24 — FINAL PREPROCESSED DATASET AUDIT
# ================================================================

from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL PREPROCESSED DATASET AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

TRANSFORMED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "transformed_datasets"
)

TRAIN_FILE = TRANSFORMED_DIR / "train_transformed.parquet"
VALIDATION_FILE = TRANSFORMED_DIR / "validation_transformed.parquet"
TEST_FILE = TRANSFORMED_DIR / "test_transformed.parquet"

FILES = {
    "train": TRAIN_FILE,
    "validation": VALIDATION_FILE,
    "test": TEST_FILE
}

# ------------------------------------------------------------
# 2. EXPECTED SPLIT VALUES FROM VALIDATED TEMPORAL SPLIT
# ------------------------------------------------------------

EXPECTED = {
    "train": {
        "observations": 16_101_651,
        "unique_loans": 450_106,
        "events": 519_144,
        "non_events": 15_582_507,
        "months": 60,
        "min_date": "2018-01-01",
        "max_date": "2022-12-01"
    },
    "validation": {
        "observations": 1_569_879,
        "unique_loans": 138_201,
        "events": 50_952,
        "non_events": 1_518_927,
        "months": 12,
        "min_date": "2023-01-01",
        "max_date": "2023-12-01"
    },
    "test": {
        "observations": 2_483_739,
        "unique_loans": 128_906,
        "events": 76_861,
        "non_events": 2_406_878,
        "months": 21,
        "min_date": "2024-01-01",
        "max_date": "2025-09-01"
    }
}

# ------------------------------------------------------------
# 3. FILE EXISTENCE
# ------------------------------------------------------------

print("\nChecking transformed files...")

for split, path in FILES.items():
    print(f"{split:12s}: {path.exists()}")
    assert path.exists(), f"Missing transformed dataset: {path}"

# ------------------------------------------------------------
# 4. FRESH DUCKDB CONNECTION
# ------------------------------------------------------------

audit_temp_dir = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "audit_duckdb_temp"
)

audit_temp_dir.mkdir(parents=True, exist_ok=True)

audit_con = duckdb.connect()

audit_con.execute("SET memory_limit = '4GB'")
audit_con.execute("SET threads = 2")
audit_con.execute("SET preserve_insertion_order = false")
audit_con.execute(
    f"SET temp_directory = '{audit_temp_dir}'"
)

# ------------------------------------------------------------
# 5. BASIC DATASET AUDIT
# ------------------------------------------------------------

audit_results = []

for split, path in FILES.items():

    print(f"\nAuditing {split.upper()}...")

    result = audit_con.execute(f"""
        SELECT
            COUNT(*) AS observations,
            COUNT(DISTINCT loan_id) AS unique_loans,
            SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END)
                AS events,
            SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END)
                AS non_events,
            COUNT(DISTINCT reporting_date)
                AS reporting_months,
            MIN(reporting_date) AS min_date,
            MAX(reporting_date) AS max_date,
            SUM(CASE WHEN loan_id IS NULL THEN 1 ELSE 0 END)
                AS null_loan_id,
            SUM(CASE WHEN reporting_date IS NULL THEN 1 ELSE 0 END)
                AS null_reporting_date,
            SUM(CASE WHEN target_6m IS NULL THEN 1 ELSE 0 END)
                AS null_target
        FROM read_parquet('{path}')
    """).fetchdf()

    result.insert(0, "split", split)
    audit_results.append(result)

audit_summary = pd.concat(
    audit_results,
    ignore_index=True
)

display(audit_summary)

# ------------------------------------------------------------
# 6. EXACT SPLIT VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXACT SPLIT VALIDATION")
print("=" * 70)

split_validation = []

for _, row in audit_summary.iterrows():

    split = row["split"]
    exp = EXPECTED[split]

    checks = {
        "observations": int(row["observations"]) == exp["observations"],
        "unique_loans": int(row["unique_loans"]) == exp["unique_loans"],
        "events": int(row["events"]) == exp["events"],
        "non_events": int(row["non_events"]) == exp["non_events"],
        "reporting_months": int(row["reporting_months"]) == exp["months"],
        "min_date": str(row["min_date"])[:10] == exp["min_date"],
        "max_date": str(row["max_date"])[:10] == exp["max_date"],
        "null_loan_id": int(row["null_loan_id"]) == 0,
        "null_reporting_date": int(row["null_reporting_date"]) == 0,
        "null_target": int(row["null_target"]) == 0
    }

    for check_name, passed in checks.items():
        split_validation.append({
            "split": split,
            "check": check_name,
            "status": "PASS" if passed else "FAIL"
        })

split_validation_df = pd.DataFrame(split_validation)

display(
    split_validation_df.pivot(
        index="split",
        columns="check",
        values="status"
    )
)

assert (
    split_validation_df["status"] == "PASS"
).all()

print("\nExact split counts/ranges: PASS")

# ------------------------------------------------------------
# 7. TOTAL ROW / TARGET RECONCILIATION
# ------------------------------------------------------------

total_summary = audit_con.execute(f"""
SELECT
    (
        SELECT COUNT(*)
        FROM read_parquet('{TRAIN_FILE}')
    )
    +
    (
        SELECT COUNT(*)
        FROM read_parquet('{VALIDATION_FILE}')
    )
    +
    (
        SELECT COUNT(*)
        FROM read_parquet('{TEST_FILE}')
    ) AS total_observations,

    (
        SELECT SUM(target_6m)
        FROM read_parquet('{TRAIN_FILE}')
    )
    +
    (
        SELECT SUM(target_6m)
        FROM read_parquet('{VALIDATION_FILE}')
    )
    +
    (
        SELECT SUM(target_6m)
        FROM read_parquet('{TEST_FILE}')
    ) AS total_events
""").fetchdf()

print("\nOverall transformed dataset:")
display(total_summary)

assert int(total_summary.loc[0, "total_observations"]) == 20_155_269
assert int(total_summary.loc[0, "total_events"]) == 646_957

print("Overall row/event reconciliation: PASS")

# ------------------------------------------------------------
# 8. SCHEMA AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCHEMA AUDIT")
print("=" * 70)

schema_dict = {}

for split, path in FILES.items():

    schema = audit_con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{path}')
    """).fetchdf()

    schema_dict[split] = schema

    print(
        f"{split:12s}: "
        f"{len(schema)} columns"
    )

train_columns = schema_dict["train"]["column_name"].tolist()
validation_columns = schema_dict["validation"]["column_name"].tolist()
test_columns = schema_dict["test"]["column_name"].tolist()

assert train_columns == validation_columns
assert train_columns == test_columns

print("\nTrain = Validation schema : PASS")
print("Train = Test schema       : PASS")

# ------------------------------------------------------------
# 9. CORE COLUMN CHECK
# ------------------------------------------------------------

required_core_columns = [
    "loan_id",
    "reporting_date",
    "target_6m"
]

for col in required_core_columns:
    assert col in train_columns

print("Required core columns     : PASS")

# ------------------------------------------------------------
# 10. TARGET INTEGRITY
# ------------------------------------------------------------

target_values = audit_con.execute(f"""
SELECT DISTINCT target_6m
FROM read_parquet('{TRAIN_FILE}')
ORDER BY target_6m
""").fetchdf()["target_6m"].tolist()

print(f"\nTarget values: {target_values}")

assert set(target_values) == {0, 1}

for split, path in FILES.items():

    invalid_targets = audit_con.execute(f"""
        SELECT COUNT(*) AS invalid_targets
        FROM read_parquet('{path}')
        WHERE target_6m NOT IN (0, 1)
           OR target_6m IS NULL
    """).fetchone()[0]

    assert invalid_targets == 0

print("Binary target integrity    : PASS")

# ------------------------------------------------------------
# 11. LEAKAGE AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE / PREDICTOR AUDIT")
print("=" * 70)

# These must never be modelling predictors.
forbidden_predictor_columns = [
    "loan_id",
    "target_6m"
]

# Current delinquency and explicitly excluded raw fields should
# already have been removed during feature engineering.
leakage_sensitive_columns = [
    "current_delinquency_status",
    "zero_balance_code",
    "zero_balance_effective_date",
    "mi_cancellation_indicator",
    "interest_only_first_pi_date",
    "months_to_amortization"
]

# loan_id and target are expected to exist as metadata/core fields,
# but must be excluded when the actual predictor matrix is built.

unexpected_leakage_columns = [
    col
    for col in train_columns
    if col in leakage_sensitive_columns
]

print("Leakage-sensitive columns remaining:")
print(unexpected_leakage_columns)

assert len(unexpected_leakage_columns) == 0

print("Leakage-sensitive columns : PASS")

# ------------------------------------------------------------
# 12. RAW CATEGORICAL COLUMN AUDIT
# ------------------------------------------------------------

raw_categorical_columns = [
    "channel",
    "seller_name",
    "servicer_name",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag",
    "zip_code_short",
    "msa",
    "property_state"
]

remaining_raw_categoricals = [
    col
    for col in raw_categorical_columns
    if col in train_columns
]

print("\nRaw categorical columns remaining:")
print(remaining_raw_categoricals)

assert len(remaining_raw_categoricals) == 0

print("Raw categorical removal    : PASS")

# ------------------------------------------------------------
# 13. NON-FINITE NUMERIC AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMERIC VALUE AUDIT")
print("=" * 70)

numeric_schema = audit_con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{TRAIN_FILE}')
""").fetchdf()

numeric_columns = numeric_schema[
    numeric_schema["column_type"].str.upper().str.contains(
        "DOUBLE|FLOAT|DECIMAL|INTEGER|BIGINT|SMALLINT|HUGEINT"
    )
]["column_name"].tolist()

print(f"Numeric columns detected: {len(numeric_columns)}")

# Check all transformed files using DuckDB isfinite().
nonfinite_results = []

for split, path in FILES.items():

    for col in numeric_columns:

        count = audit_con.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{path}')
            WHERE "{col}" IS NOT NULL
              AND NOT isfinite(CAST("{col}" AS DOUBLE))
        """).fetchone()[0]

        if count > 0:
            nonfinite_results.append({
                "split": split,
                "column": col,
                "nonfinite_count": count
            })

if nonfinite_results:
    nonfinite_df = pd.DataFrame(nonfinite_results)
    display(nonfinite_df)
    raise AssertionError(
        "Non-finite numeric values detected."
    )

print("Non-finite numeric values : PASS")

# ------------------------------------------------------------
# 14. PREDICTOR COUNT
# ------------------------------------------------------------

# Remove core metadata fields.
predictor_columns = [
    col
    for col in train_columns
    if col not in [
        "loan_id",
        "reporting_date",
        "target_6m"
    ]
]

print("\nPredictor matrix:")
print(f"  Total transformed columns : {len(train_columns)}")
print(f"  Core fields excluded      : 3")
print(f"  Predictor features        : {len(predictor_columns)}")

# 46 numeric + 77 one-hot + 4 frequency + 2 calendar = 129
# predictors in this final output.
#
# Note:
# Some numeric columns originate from engineered features and
# remain numeric after preprocessing.

assert len(predictor_columns) == 129

print("Predictor feature count    : PASS")

# ------------------------------------------------------------
# 15. TEMPORAL ORDERING AUDIT
# ------------------------------------------------------------

temporal_check = audit_con.execute(f"""
SELECT
    (SELECT MAX(reporting_date)
     FROM read_parquet('{TRAIN_FILE}')) <
    (SELECT MIN(reporting_date)
     FROM read_parquet('{VALIDATION_FILE}'))
    AS train_before_validation,

    (SELECT MAX(reporting_date)
     FROM read_parquet('{VALIDATION_FILE}')) <
    (SELECT MIN(reporting_date)
     FROM read_parquet('{TEST_FILE}'))
    AS validation_before_test
""").fetchdf()

display(temporal_check)

assert bool(
    temporal_check.loc[0, "train_before_validation"]
)

assert bool(
    temporal_check.loc[0, "validation_before_test"]
)

print("Temporal ordering          : PASS")

# ------------------------------------------------------------
# 16. FINAL AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL PREPROCESSED DATASET AUDIT RESULT")
print("=" * 70)

print("Exact split counts         : PASS")
print("Overall row reconciliation : PASS")
print("Overall event reconciliation: PASS")
print("Schema consistency         : PASS")
print("Core columns               : PASS")
print("Binary target              : PASS")
print("Leakage-sensitive columns  : PASS")
print("Raw categorical removal    : PASS")
print("Non-finite numeric values  : PASS")
print("Predictor feature count    : PASS")
print("Temporal ordering          : PASS")

print("\n" + "=" * 70)
print("OVERALL: PASS")
print("=" * 70)

audit_con.close()

gc.collect()

FINAL PREPROCESSED DATASET AUDIT

Checking transformed files...
train       : True
validation  : True
test        : True

Auditing TRAIN...

Auditing VALIDATION...

Auditing TEST...


,split,observations,unique_loans,events,non_events,reporting_months,min_date,max_date,null_loan_id,null_reporting_date,null_target
0,train,16101651,450106,519144.0,15582507.0,60,2018-01-01,2022-12-01,0.0,0.0,0.0
1,validation,1569879,138201,50952.0,1518927.0,12,2023-01-01,2023-12-01,0.0,0.0,0.0
2,test,2483739,128906,76861.0,2406878.0,21,2024-01-01,2025-09-01,0.0,0.0,0.0



EXACT SPLIT VALIDATION


check,events,max_date,min_date,non_events,null_loan_id,null_reporting_date,null_target,observations,reporting_months,unique_loans
split,,,,,,,,,,
test,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
train,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
validation,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS



Exact split counts/ranges: PASS

Overall transformed dataset:


,total_observations,total_events
0,20155269,646957.0


Overall row/event reconciliation: PASS

SCHEMA AUDIT
train       : 132 columns
validation  : 132 columns
test        : 132 columns

Train = Validation schema : PASS
Train = Test schema       : PASS
Required core columns     : PASS

Target values: [0, 1]
Binary target integrity    : PASS

LEAKAGE / PREDICTOR AUDIT
Leakage-sensitive columns remaining:
[]
Leakage-sensitive columns : PASS

Raw categorical columns remaining:
[]
Raw categorical removal    : PASS

NUMERIC VALUE AUDIT
Numeric columns detected: 131
Non-finite numeric values : PASS

Predictor matrix:
  Total transformed columns : 132
  Core fields excluded      : 3
  Predictor features        : 129
Predictor feature count    : PASS


,train_before_validation,validation_before_test
0,True,True


Temporal ordering          : PASS

FINAL PREPROCESSED DATASET AUDIT RESULT
Exact split counts         : PASS
Overall row reconciliation : PASS
Overall event reconciliation: PASS
Schema consistency         : PASS
Core columns               : PASS
Binary target              : PASS
Leakage-sensitive columns  : PASS
Raw categorical removal    : PASS
Non-finite numeric values  : PASS
Predictor feature count    : PASS
Temporal ordering          : PASS

OVERALL: PASS


4

# Baseline Model Development — Memory-Efficient Regularized Logistic Regression

The training dataset contains approximately **16.1 million observations and 129 predictor features**. Loading the complete training matrix into pandas simultaneously is unnecessary and may exceed available system memory.

Therefore, the baseline logistic model is trained using **mini-batch incremental learning** with `SGDClassifier.partial_fit()`.

The complete training dataset is still used. It is simply processed in manageable batches rather than loaded into memory at once.

Class weights are calculated from the complete training-set event distribution and supplied explicitly to each batch. This avoids relying on batch-specific class proportions.

The validation and test datasets are also evaluated in batches, ensuring that the full temporal datasets are used without creating a large in-memory prediction matrix.

The model remains a regularized logistic-loss classifier and serves as the benchmark for subsequent nonlinear models.

In [24]:
# ================================================================
# CELL 4 — MEMORY-EFFICIENT BASELINE LOGISTIC MODEL
# ================================================================

import pandas as pd
import numpy as np
import duckdb
import joblib
import time
import gc

from pathlib import Path
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("=" * 70)
print("MEMORY-EFFICIENT BASELINE REGULARIZED LOGISTIC MODEL")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = (
    PROJECT_DIR / "data" / "processed" / "model_preprocessing"
    / "transformed_datasets" / "train_transformed.parquet"
)

VALIDATION_FILE = (
    PROJECT_DIR / "data" / "processed" / "model_preprocessing"
    / "transformed_datasets" / "validation_transformed.parquet"
)

TEST_FILE = (
    PROJECT_DIR / "data" / "processed" / "model_preprocessing"
    / "transformed_datasets" / "test_transformed.parquet"
)

MODEL_FILE = MODEL_DIR / "baseline_logistic_sgd.joblib"
METRICS_FILE = MODEL_DIR / "baseline_logistic_metrics.csv"

BATCH_SIZE = 100_000

# ------------------------------------------------
# 2. Dataset configuration
# ------------------------------------------------

CORE_COLUMNS = [
    "loan_id",
    "reporting_date",
    "target_6m"
]

# ------------------------------------------------
# 3. Read schema only
# ------------------------------------------------

print("\nReading transformed dataset schema...")

schema_con = duckdb.connect()
schema_con.execute("SET memory_limit = '2GB'")
schema_con.execute("SET threads = 2")

schema_df = schema_con.execute(
    f"""
    SELECT *
    FROM read_parquet('{TRAIN_FILE}')
    LIMIT 0
    """
).fetchdf()

schema_con.close()

FEATURE_COLUMNS = [
    c for c in schema_df.columns
    if c not in CORE_COLUMNS
]

assert len(FEATURE_COLUMNS) == 129

print(f"Total dataset columns : {len(schema_df.columns)}")
print(f"Predictor features    : {len(FEATURE_COLUMNS)}")
print(f"Batch size            : {BATCH_SIZE:,}")

# ------------------------------------------------
# 4. Expected observations
# ------------------------------------------------

EXPECTED_ROWS = {
    "train": 16_101_651,
    "validation": 1_569_879,
    "test": 2_483_739
}

EXPECTED_EVENTS = {
    "train": 519_144,
    "validation": 50_952,
    "test": 76_861
}

EXPECTED_NON_EVENTS = {
    "train": 15_582_507,
    "validation": 1_518_927,
    "test": 2_406_878
}

print("\nExpected temporal datasets:")
print(f"Train      : {EXPECTED_ROWS['train']:,}")
print(f"Validation : {EXPECTED_ROWS['validation']:,}")
print(f"Test       : {EXPECTED_ROWS['test']:,}")

# ------------------------------------------------
# 5. Calculate fixed training class weights
# ------------------------------------------------
#
# These weights are calculated from the COMPLETE
# training distribution rather than individual batches.
# ------------------------------------------------

n_train = EXPECTED_ROWS["train"]
n_event = EXPECTED_EVENTS["train"]
n_non_event = EXPECTED_NON_EVENTS["train"]

class_weight_0 = n_train / (2.0 * n_non_event)
class_weight_1 = n_train / (2.0 * n_event)

CLASS_WEIGHTS = {
    0: class_weight_0,
    1: class_weight_1
}

print("\nTraining class weights:")
print(f"Class 0 weight : {class_weight_0:.6f}")
print(f"Class 1 weight : {class_weight_1:.6f}")

# ------------------------------------------------
# 6. Helper: stream parquet batches
# ------------------------------------------------

def stream_batches(file_path, batch_size=BATCH_SIZE):

    con = duckdb.connect()
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 2")
    con.execute("SET preserve_insertion_order = false")

    feature_sql = ", ".join(
        f'"{c}"' for c in FEATURE_COLUMNS
    )

    query = f"""
        SELECT
            {feature_sql},
            target_6m
        FROM read_parquet('{file_path}')
    """

    reader = con.execute(query).fetch_record_batch(
        rows_per_batch=batch_size
    )

    try:
        for batch in reader:
            df = batch.to_pandas()

            X = df[FEATURE_COLUMNS].to_numpy(
                dtype=np.float32,
                copy=False
            )

            y = df["target_6m"].to_numpy(
                dtype=np.int8,
                copy=False
            )

            yield X, y

            del df, X, y
            gc.collect()

    finally:
        con.close()


# ------------------------------------------------
# 7. Create baseline model
# ------------------------------------------------

baseline_model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-5,
    random_state=42,
    average=True
)

# ------------------------------------------------
# 8. Incremental training
# ------------------------------------------------

print("\n" + "=" * 70)
print("STARTING BATCH-WISE TRAINING")
print("=" * 70)

start_time = time.time()

total_rows_seen = 0
total_events_seen = 0
batch_number = 0

for X_batch, y_batch in stream_batches(TRAIN_FILE):

    batch_number += 1

    sample_weights = np.where(
        y_batch == 1,
        class_weight_1,
        class_weight_0
    ).astype(np.float32)

    baseline_model.partial_fit(
        X_batch,
        y_batch,
        classes=np.array([0, 1], dtype=np.int8),
        sample_weight=sample_weights
    )

    total_rows_seen += len(y_batch)
    total_events_seen += int(y_batch.sum())

    if batch_number == 1 or batch_number % 20 == 0:
        print(
            f"Batch {batch_number:>4} | "
            f"Rows processed: {total_rows_seen:>10,} | "
            f"Events: {total_events_seen:>8,}"
        )

    del X_batch, y_batch, sample_weights
    gc.collect()

training_time = time.time() - start_time

print("\nTraining completed.")
print(f"Training batches : {batch_number}")
print(f"Rows processed   : {total_rows_seen:,}")
print(f"Events processed : {total_events_seen:,}")
print(f"Training time    : {training_time / 60:.2f} minutes")

# ------------------------------------------------
# 9. Verify complete training data was processed
# ------------------------------------------------

assert total_rows_seen == EXPECTED_ROWS["train"]
assert total_events_seen == EXPECTED_EVENTS["train"]

print("\nComplete training-set processing: PASS")

# ------------------------------------------------
# 10. Batch-wise probability evaluation
# ------------------------------------------------

def evaluate_streaming_model(model, file_path, split_name):

    print(f"\nEvaluating {split_name}...")

    auc_scores = []
    all_probabilities = []
    all_targets = []

    total_rows = 0

    for X_batch, y_batch in stream_batches(file_path):

        probabilities = model.predict_proba(X_batch)[:, 1]

        all_probabilities.append(probabilities)
        all_targets.append(y_batch)

        total_rows += len(y_batch)

        del X_batch, y_batch, probabilities
        gc.collect()

    y_true = np.concatenate(all_targets)
    probabilities = np.concatenate(all_probabilities)

    auc = roc_auc_score(
        y_true,
        probabilities
    )

    pr_auc = average_precision_score(
        y_true,
        probabilities
    )

    brier = brier_score_loss(
        y_true,
        probabilities
    )

    print(f"Rows       : {total_rows:,}")
    print(f"ROC-AUC    : {auc:.6f}")
    print(f"PR-AUC     : {pr_auc:.6f}")
    print(f"Brier Score: {brier:.6f}")

    return {
        "split": split_name,
        "roc_auc": auc,
        "pr_auc": pr_auc,
        "brier_score": brier,
        "observations": total_rows
    }


# ------------------------------------------------
# 11. Evaluate train / validation / test
# ------------------------------------------------

train_metrics = evaluate_streaming_model(
    baseline_model,
    TRAIN_FILE,
    "TRAIN"
)

validation_metrics = evaluate_streaming_model(
    baseline_model,
    VALIDATION_FILE,
    "VALIDATION"
)

test_metrics = evaluate_streaming_model(
    baseline_model,
    TEST_FILE,
    "TEST"
)

# ------------------------------------------------
# 12. Metrics table
# ------------------------------------------------

baseline_metrics = pd.DataFrame([
    train_metrics,
    validation_metrics,
    test_metrics
])

print("\n" + "=" * 70)
print("BASELINE LOGISTIC MODEL SUMMARY")
print("=" * 70)

print(
    baseline_metrics.to_string(index=False)
)

# ------------------------------------------------
# 13. Save model and metrics
# ------------------------------------------------

joblib.dump(
    baseline_model,
    MODEL_FILE
)

baseline_metrics.to_csv(
    METRICS_FILE,
    index=False
)

print("\nModel saved:")
print(MODEL_FILE)

print("\nMetrics saved:")
print(METRICS_FILE)

# ------------------------------------------------
# 14. Final validation
# ------------------------------------------------

assert MODEL_FILE.exists()
assert METRICS_FILE.exists()

assert total_rows_seen == EXPECTED_ROWS["train"]
assert total_events_seen == EXPECTED_EVENTS["train"]

assert train_metrics["observations"] == EXPECTED_ROWS["train"]
assert validation_metrics["observations"] == EXPECTED_ROWS["validation"]
assert test_metrics["observations"] == EXPECTED_ROWS["test"]

assert np.isfinite(
    baseline_metrics[
        ["roc_auc", "pr_auc", "brier_score"]
    ].to_numpy()
).all()

print("\n" + "=" * 70)
print("BASELINE LOGISTIC MODEL RESULT: PASS")
print("=" * 70)

MEMORY-EFFICIENT BASELINE REGULARIZED LOGISTIC MODEL

Reading transformed dataset schema...
Total dataset columns : 132
Predictor features    : 129
Batch size            : 100,000

Expected temporal datasets:
Train      : 16,101,651
Validation : 1,569,879
Test       : 2,483,739

Training class weights:
Class 0 weight : 0.516658
Class 1 weight : 15.507885

STARTING BATCH-WISE TRAINING


C:\Users\madhu\AppData\Local\Temp\ipykernel_13856\1952724361.py:165: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = con.execute(query).fetch_record_batch(


Batch    1 | Rows processed:    100,000 | Events:    3,463


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Batch   20 | Rows processed:  2,000,000 | Events:   64,378
Batch   40 | Rows processed:  4,000,000 | Events:  127,848
Batch   60 | Rows processed:  6,000,000 | Events:  191,681
Batch   80 | Rows processed:  8,000,000 | Events:  256,682
Batch  100 | Rows processed: 10,000,000 | Events:  320,785
Batch  120 | Rows processed: 12,000,000 | Events:  385,785
Batch  140 | Rows processed: 14,000,000 | Events:  450,780
Batch  160 | Rows processed: 16,000,000 | Events:  516,016

Training completed.
Training batches : 162
Rows processed   : 16,101,651
Events processed : 519,144
Training time    : 2.18 minutes

Complete training-set processing: PASS

Evaluating TRAIN...


C:\Users\madhu\AppData\Local\Temp\ipykernel_13856\1952724361.py:165: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = con.execute(query).fetch_record_batch(


... (4 outputs left out)

Rows       : 1,569,879
ROC-AUC    : 0.502889
PR-AUC     : 0.032638
Brier Score: 0.959384

Evaluating TEST...


C:\Users\madhu\AppData\Local\Temp\ipykernel_13856\1952724361.py:165: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = con.execute(query).fetch_record_batch(


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows       : 2,483,739
ROC-AUC    : 0.503883
PR-AUC     : 0.031180
Brier Score: 0.956208

BASELINE LOGISTIC MODEL SUMMARY
     split  roc_auc   pr_auc  brier_score  observations
     TRAIN 0.546637 0.035473     0.791905      16101651
VALIDATION 0.502889 0.032638     0.959384       1569879
      TEST 0.503883 0.031180     0.956208       2483739

Model saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\baseline_logistic_sgd.joblib

Metrics saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\baseline_logistic_metrics.csv

BASELINE LOGISTIC MODEL RESULT: PASS


## 22. Non-Linear Benchmark Model — Histogram Gradient Boosting

The baseline regularized logistic regression provides a linear benchmark for the
six-month credit-risk early-warning task. The next model introduces non-linear
relationships and feature interactions that may not be captured by a linear model.

A Histogram Gradient Boosting classifier is selected because the current
environment contains scikit-learn, while XGBoost, LightGBM, and CatBoost are not
available.

To maintain computational feasibility with the 16.1 million training
observations, model fitting will use a reproducible training sample while the
validation and test datasets will remain untouched and will be evaluated in
full.

The temporal split remains unchanged:

- Training: January 2018 – December 2022
- Validation: January 2023 – December 2023
- Test: January 2024 – September 2025

The target is `target_6m`, where:

- 0 = no 30+ day delinquency during the following six months
- 1 = 30+ day delinquency during the following six months

Model performance will be evaluated using:

- ROC-AUC
- PR-AUC
- Brier score

The validation set will be used for model comparison and model-selection
decisions. The test set will remain an out-of-time holdout for final
performance assessment.

This model is a benchmark for predictive comparison; it is not being claimed
as a novel algorithmic contribution.

In [25]:
# ================================================================
# CELL 23 — DETERMINISTIC TRAINING SAMPLE FOR NON-LINEAR MODEL
# ================================================================

from pathlib import Path
import duckdb
import pandas as pd
import gc

print("=" * 70)
print("NON-LINEAR MODEL — TRAINING SAMPLE CONSTRUCTION")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

MODEL_PREP_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
)

TRANSFORMED_DIR = MODEL_PREP_DIR / "transformed_datasets"
MODEL_DIR = PROJECT_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = TRANSFORMED_DIR / "train_transformed.parquet"

HGB_SAMPLE_FILE = (
    MODEL_PREP_DIR
    / "hgb_training_sample.parquet"
)

assert TRAIN_FILE.exists(), "Training transformed dataset not found."

print(f"\nSource training dataset:")
print(TRAIN_FILE)

print(f"\nSample output:")
print(HGB_SAMPLE_FILE)

# ------------------------------------------------
# 2. Fresh DuckDB connection
# ------------------------------------------------
sample_con = duckdb.connect()

sample_temp_dir = MODEL_PREP_DIR / "hgb_sampling_duckdb_temp"
sample_temp_dir.mkdir(parents=True, exist_ok=True)

sample_con.execute("SET memory_limit = '4GB'")
sample_con.execute("SET threads = 2")
sample_con.execute("SET preserve_insertion_order = false")
sample_con.execute(
    f"SET temp_directory = '{sample_temp_dir}'"
)

# ------------------------------------------------
# 3. Predictor columns
# ------------------------------------------------
schema_df = sample_con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{TRAIN_FILE}')
    """
).fetchdf()

all_columns = schema_df["column_name"].tolist()

CORE_COLUMNS = {
    "loan_id",
    "reporting_date",
    "target_6m"
}

PREDICTOR_COLUMNS = [
    c for c in all_columns
    if c not in CORE_COLUMNS
]

assert len(PREDICTOR_COLUMNS) == 129, (
    f"Expected 129 predictor features, "
    f"found {len(PREDICTOR_COLUMNS)}."
)

print(f"\nTotal source columns : {len(all_columns)}")
print(f"Predictor features   : {len(PREDICTOR_COLUMNS)}")

# ------------------------------------------------
# 4. Build deterministic loan-level sample
# ------------------------------------------------
#
# hash(loan_id) % 32 = 0
# approximately 1/32 (~3.125%) of loans
#
# Sampling by loan rather than individual rows ensures
# that all observations of a selected loan remain together.
# No validation/test observations are touched.
# ------------------------------------------------

predictor_sql = ",\n        ".join(
    [f'"{c}"' for c in PREDICTOR_COLUMNS]
)

sample_query = f"""
COPY (
    SELECT
        target_6m,
        {predictor_sql}
    FROM read_parquet('{TRAIN_FILE}')
    WHERE ABS(hash(loan_id)) % 32 = 0
)
TO '{HGB_SAMPLE_FILE}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

print("\nCreating deterministic loan-level training sample...")
print("Sampling rule: ABS(hash(loan_id)) % 32 = 0")
print("Approximately 3.125% of training loans will be retained.")
print("Validation and test datasets are untouched.")
print("\nPlease allow the cell to complete...")

sample_con.execute(sample_query)

print("\nSample construction completed.")
print(f"Output exists: {HGB_SAMPLE_FILE.exists()}")

# ------------------------------------------------
# 5. Validate sample
# ------------------------------------------------
sample_validation = sample_con.execute(
    f"""
    SELECT
        COUNT(*) AS observations,
        SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END) AS events,
        SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
        MIN(target_6m) AS min_target,
        MAX(target_6m) AS max_target
    FROM read_parquet('{HGB_SAMPLE_FILE}')
    """
).fetchdf()

print("\nTraining sample summary:")
display(sample_validation)

sample_rate = (
    sample_validation.loc[0, "events"]
    / sample_validation.loc[0, "observations"]
    * 100
)

print(f"\nSample event rate: {sample_rate:.4f}%")

# ------------------------------------------------
# 6. Schema validation
# ------------------------------------------------
sample_schema = sample_con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{HGB_SAMPLE_FILE}')
    """
).fetchdf()

sample_columns = sample_schema["column_name"].tolist()

assert sample_columns == ["target_6m"] + PREDICTOR_COLUMNS, (
    "Training sample schema does not match expected predictor structure."
)

assert sample_validation.loc[0, "min_target"] == 0
assert sample_validation.loc[0, "max_target"] == 1

print("\nSchema validation: PASS")
print("Binary target validation: PASS")
print("Predictor count validation: PASS")

print("\n" + "=" * 70)
print("CELL 23 RESULT: PASS")
print("=" * 70)

sample_con.close()
gc.collect()

NON-LINEAR MODEL — TRAINING SAMPLE CONSTRUCTION

Source training dataset:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\transformed_datasets\train_transformed.parquet

Sample output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\hgb_training_sample.parquet

Total source columns : 132
Predictor features   : 129

Creating deterministic loan-level training sample...
Sampling rule: ABS(hash(loan_id)) % 32 = 0
Approximately 3.125% of training loans will be retained.
Validation and test datasets are untouched.

Please allow the cell to complete...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample construction completed.
Output exists: True

Training sample summary:


,observations,events,non_events,min_target,max_target
0,499725,16257.0,483468.0,0,1



Sample event rate: 3.2532%

Schema validation: PASS
Binary target validation: PASS
Predictor count validation: PASS

CELL 23 RESULT: PASS


0

## 24. Non-Linear Benchmark — HistGradientBoosting Model

This section trains a non-linear gradient-boosting benchmark using the
deterministic training sample created in Cell 23.

The model is trained only on the training-period sample and is evaluated
separately on the complete validation and out-of-time test datasets.

Model choice:
- `HistGradientBoostingClassifier` from scikit-learn
- Captures non-linear relationships and feature interactions
- Uses histogram-based optimization for computational efficiency

The modelling design preserves the temporal validation framework:

- Training: January 2018 – December 2022
- Validation: January 2023 – December 2023
- Test: January 2024 – September 2025

The validation dataset is used for model assessment and later model selection.
The test dataset remains untouched as the final out-of-time evaluation set.

Primary evaluation metrics:
- ROC-AUC
- PR-AUC
- Brier score

The model will use class weighting through sample weights because the target is
imbalanced (~3.2% event rate).

No target-derived feature or future-period information is introduced during
model fitting.

In [26]:
# ================================================================
# CELL 25 — HISTGRADIENTBOOSTING MODEL TRAINING
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import gc
import time

from sklearn.ensemble import HistGradientBoostingClassifier

print("=" * 70)
print("NON-LINEAR BENCHMARK — HISTOGRAM GRADIENT BOOSTING")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

MODEL_PREP_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
)

HGB_SAMPLE_FILE = (
    MODEL_PREP_DIR
    / "hgb_training_sample.parquet"
)

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HGB_MODEL_FILE = (
    MODEL_DIR
    / "hist_gradient_boosting_baseline.joblib"
)

assert HGB_SAMPLE_FILE.exists(), (
    "HGB training sample not found. "
    "Run Cell 23 first."
)

print(f"\nTraining sample:")
print(HGB_SAMPLE_FILE)

print(f"\nModel output:")
print(HGB_MODEL_FILE)

# ------------------------------------------------
# 2. Load deterministic training sample
# ------------------------------------------------
print("\nLoading training sample...")

train_sample = pd.read_parquet(HGB_SAMPLE_FILE)

print(f"Rows loaded: {len(train_sample):,}")
print(f"Columns loaded: {train_sample.shape[1]:,}")

# ------------------------------------------------
# 3. Separate target and predictors
# ------------------------------------------------
assert "target_6m" in train_sample.columns

y_train = train_sample["target_6m"].astype(np.int8)

X_train = train_sample.drop(columns=["target_6m"])

# Ensure every predictor is numeric
non_numeric_columns = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

assert len(non_numeric_columns) == 0, (
    f"Non-numeric predictors detected: {non_numeric_columns}"
)

X_train = X_train.astype(np.float32)

print("\nTraining matrix:")
print(f"X shape: {X_train.shape}")
print(f"y shape: {y_train.shape}")

# ------------------------------------------------
# 4. Target distribution
# ------------------------------------------------
positive_count = int((y_train == 1).sum())
negative_count = int((y_train == 0).sum())

positive_rate = positive_count / len(y_train)

print("\nTarget distribution:")
print(f"Non-events : {negative_count:,}")
print(f"Events     : {positive_count:,}")
print(f"Event rate : {positive_rate * 100:.4f}%")

# ------------------------------------------------
# 5. Class-balanced sample weights
# ------------------------------------------------
#
# HistGradientBoostingClassifier does not expose a
# class_weight parameter in the same way as several
# other sklearn classifiers.
#
# Therefore, balanced observation weights are supplied
# explicitly through sample_weight.
#
# Weight formula:
#     n / (2 * class_count)
# ------------------------------------------------
weight_negative = len(y_train) / (2 * negative_count)
weight_positive = len(y_train) / (2 * positive_count)

sample_weight = np.where(
    y_train.to_numpy() == 1,
    weight_positive,
    weight_negative
).astype(np.float32)

print("\nClass weights:")
print(f"Class 0 weight: {weight_negative:.4f}")
print(f"Class 1 weight: {weight_positive:.4f}")

# ------------------------------------------------
# 6. Define model
# ------------------------------------------------
hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=31,
    min_samples_leaf=100,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15,
    random_state=42
)

print("\nModel configuration:")
print("Algorithm          : HistGradientBoostingClassifier")
print("Learning rate      : 0.05")
print("Maximum iterations : 200")
print("Maximum leaf nodes : 31")
print("Minimum leaf size  : 100")
print("L2 regularization  : 1.0")
print("Early stopping     : Enabled")
print("Random state       : 42")

# ------------------------------------------------
# 7. Train
# ------------------------------------------------
print("\n" + "=" * 70)
print("STARTING MODEL TRAINING")
print("=" * 70)

start_time = time.time()

hgb_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weight
)

elapsed_minutes = (time.time() - start_time) / 60

print("\nTraining completed.")
print(f"Training time: {elapsed_minutes:.2f} minutes")

# ------------------------------------------------
# 8. Training diagnostics
# ------------------------------------------------
print("\nTraining diagnostics:")
print(f"Iterations used: {hgb_model.n_iter_}")

if hasattr(hgb_model, "n_trees_per_iteration_"):
    print(
        f"Trees per iteration: "
        f"{hgb_model.n_trees_per_iteration_}"
    )

if hasattr(hgb_model, "do_early_stopping_"):
    print(
        f"Early stopping applied: "
        f"{hgb_model.do_early_stopping_}"
    )

# ------------------------------------------------
# 9. Save model
# ------------------------------------------------
joblib.dump(
    hgb_model,
    HGB_MODEL_FILE
)

print("\nModel saved:")
print(HGB_MODEL_FILE)

print(f"Model file exists: {HGB_MODEL_FILE.exists()}")

# ------------------------------------------------
# 10. Final structural checks
# ------------------------------------------------
assert HGB_MODEL_FILE.exists()
assert len(X_train.columns) == 129
assert hgb_model.n_features_in_ == 129

print("\nModel structure:")
print(f"Predictor features: {hgb_model.n_features_in_}")
print(f"Training rows     : {len(X_train):,}")

print("\n" + "=" * 70)
print("CELL 25 RESULT: PASS")
print("=" * 70)

# Free memory
del train_sample
del X_train
del y_train
del sample_weight
gc.collect()

NON-LINEAR BENCHMARK — HISTOGRAM GRADIENT BOOSTING

Training sample:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\model_preprocessing\hgb_training_sample.parquet

Model output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\hist_gradient_boosting_baseline.joblib

Loading training sample...
Rows loaded: 499,725
Columns loaded: 130

Training matrix:
X shape: (499725, 129)
y shape: (499725,)

Target distribution:
Non-events : 483,468
Events     : 16,257
Event rate : 3.2532%

Class weights:
Class 0 weight: 0.5168
Class 1 weight: 15.3695

Model configuration:
Algorithm          : HistGradientBoostingClassifier
Learning rate      : 0.05
Maximum iterations : 200
Maximum leaf nodes : 31
Minimum leaf size  : 100
L2 regularization  : 1.0
Early stopping     : Enabled
Random state       : 42

STARTING MODEL TRAINING

Training completed.
Training time: 1.30 minutes

Training diagnostics:
Iterations used: 200
Trees per iteration: 1
Early st

29

## 26. Validation Evaluation — Non-Linear Benchmark

The trained Histogram Gradient Boosting model is now evaluated on the complete
January–December 2023 validation dataset.

No model fitting or parameter estimation is performed on the validation data.
The validation set is used strictly for out-of-sample performance assessment.

Metrics:

- ROC-AUC — overall ranking/discrimination
- PR-AUC — performance under the imbalanced positive class
- Brier score — probabilistic prediction quality

The validation results will be compared against the baseline regularized
logistic regression model before proceeding to final out-of-time testing.

The test dataset (January 2024–September 2025) remains untouched at this stage.

In [28]:
# ================================================================
# CELL 27 — FULL VALIDATION EVALUATION (MEMORY-EFFICIENT)
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import joblib
import gc
import time

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("=" * 70)
print("NON-LINEAR BENCHMARK — FULL VALIDATION EVALUATION")
print("MEMORY-EFFICIENT BATCHED PREDICTION")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

MODEL_PREP_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
)

TRANSFORMED_DIR = MODEL_PREP_DIR / "transformed_datasets"
MODEL_DIR = PROJECT_DIR / "models"

MODEL_FILE = MODEL_DIR / "hist_gradient_boosting_baseline.joblib"
VALIDATION_FILE = TRANSFORMED_DIR / "validation_transformed.parquet"

METRICS_FILE = (
    MODEL_DIR
    / "hist_gradient_boosting_validation_metrics.csv"
)

assert MODEL_FILE.exists(), "HGB model not found."
assert VALIDATION_FILE.exists(), "Validation dataset not found."

# ------------------------------------------------
# 2. Load trained model
# ------------------------------------------------
print("\nLoading trained model...")

hgb_model = joblib.load(MODEL_FILE)

EXPECTED_FEATURE_COUNT = hgb_model.n_features_in_

assert EXPECTED_FEATURE_COUNT == 129

expected_features = list(hgb_model.feature_names_in_)

print(f"Model loaded successfully.")
print(f"Expected predictor features: {EXPECTED_FEATURE_COUNT}")

# ------------------------------------------------
# 3. Inspect validation dataset
# ------------------------------------------------
print("\nInspecting validation dataset...")

con = duckdb.connect()

con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

validation_summary = con.execute(
    f"""
    SELECT
        COUNT(*) AS observations,
        SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END) AS events,
        SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
        MIN(reporting_date) AS min_date,
        MAX(reporting_date) AS max_date
    FROM read_parquet('{VALIDATION_FILE}')
    """
).fetchdf()

display(validation_summary)

expected_rows = int(
    validation_summary.loc[0, "observations"]
)

events = int(
    validation_summary.loc[0, "events"]
)

non_events = int(
    validation_summary.loc[0, "non_events"]
)

print(f"\nValidation observations: {expected_rows:,}")
print(f"Validation events      : {events:,}")
print(f"Validation non-events  : {non_events:,}")
print(
    f"Validation event rate  : "
    f"{events / expected_rows * 100:.4f}%"
)

# ------------------------------------------------
# 4. Verify validation schema
# ------------------------------------------------
schema_df = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{VALIDATION_FILE}')
    """
).fetchdf()

validation_columns = schema_df["column_name"].tolist()

core_columns = {
    "loan_id",
    "reporting_date",
    "target_6m"
}

available_predictors = [
    c for c in validation_columns
    if c not in core_columns
]

missing_features = [
    c for c in expected_features
    if c not in available_predictors
]

extra_features = [
    c for c in available_predictors
    if c not in expected_features
]

assert len(missing_features) == 0, (
    f"Missing validation features: {missing_features}"
)

assert len(extra_features) == 0, (
    f"Unexpected validation features: {extra_features}"
)

print("\nPredictor schema alignment: PASS")
print(f"Validation predictors: {len(available_predictors)}")

# ------------------------------------------------
# 5. Batched prediction
# ------------------------------------------------
#
# IMPORTANT:
# The entire validation dataset is evaluated.
# Only the in-memory batch is limited to 50,000 rows.
#
# This avoids the previous ~1.51 GB allocation.
# ------------------------------------------------

BATCH_SIZE = 50_000

select_columns = ", ".join(
    [f'"{c}"' for c in expected_features]
)

query = f"""
SELECT
    target_6m,
    {select_columns}
FROM read_parquet('{VALIDATION_FILE}')
"""

print("\nStarting batched validation prediction...")
print(f"Batch size: {BATCH_SIZE:,}")
print(f"Total rows to evaluate: {expected_rows:,}")
print("All validation observations will be evaluated.")

reader = con.execute(query).fetch_record_batch(
    rows_per_batch=BATCH_SIZE
)

all_targets = []
all_probabilities = []

processed_rows = 0
batch_number = 0

start_time = time.time()

for batch in reader:

    batch_number += 1

    batch_df = batch.to_pandas()

    y_batch = batch_df["target_6m"].to_numpy(
        dtype=np.int8
    )

    X_batch = batch_df[
        expected_features
    ].astype(np.float32)

    # Generate probabilities for this batch only
    probability_batch = hgb_model.predict_proba(
        X_batch
    )[:, 1]

    assert len(probability_batch) == len(y_batch)
    assert np.isfinite(probability_batch).all()

    all_targets.append(y_batch)
    all_probabilities.append(probability_batch)

    processed_rows += len(batch_df)

    if batch_number == 1 or batch_number % 5 == 0:
        elapsed = time.time() - start_time

        print(
            f"Batch {batch_number:>3} | "
            f"Rows processed: {processed_rows:>10,} / "
            f"{expected_rows:,} | "
            f"Elapsed: {elapsed / 60:.2f} min"
        )

    del batch_df
    del X_batch
    del y_batch
    del probability_batch

    gc.collect()

# ------------------------------------------------
# 6. Combine predictions
# ------------------------------------------------
y_validation = np.concatenate(all_targets)
validation_probability = np.concatenate(
    all_probabilities
)

del all_targets
del all_probabilities

gc.collect()

assert len(y_validation) == expected_rows
assert len(validation_probability) == expected_rows

assert np.isfinite(validation_probability).all()
assert (
    (validation_probability >= 0).all()
    and
    (validation_probability <= 1).all()
)

print("\nPrediction completed.")
print(f"Rows evaluated: {len(y_validation):,}")
print(f"Batches used   : {batch_number}")

# ------------------------------------------------
# 7. Calculate validation metrics
# ------------------------------------------------
roc_auc = roc_auc_score(
    y_validation,
    validation_probability
)

pr_auc = average_precision_score(
    y_validation,
    validation_probability
)

brier = brier_score_loss(
    y_validation,
    validation_probability
)

# ------------------------------------------------
# 8. Display results
# ------------------------------------------------
print("\n" + "=" * 70)
print("FULL VALIDATION PERFORMANCE")
print("=" * 70)

print(f"ROC-AUC : {roc_auc:.6f}")
print(f"PR-AUC  : {pr_auc:.6f}")
print(f"Brier   : {brier:.6f}")

# ------------------------------------------------
# 9. Save metrics
# ------------------------------------------------
metrics_df = pd.DataFrame({
    "model": ["hist_gradient_boosting"],
    "split": ["validation"],
    "roc_auc": [roc_auc],
    "pr_auc": [pr_auc],
    "brier_score": [brier],
    "observations": [expected_rows],
    "events": [events],
    "non_events": [non_events]
})

metrics_df.to_csv(
    METRICS_FILE,
    index=False
)

print("\nMetrics saved:")
print(METRICS_FILE)

# ------------------------------------------------
# 10. Final validation checks
# ------------------------------------------------
assert METRICS_FILE.exists()
assert len(metrics_df) == 1

assert metrics_df.loc[0, "observations"] == expected_rows
assert metrics_df.loc[0, "events"] == events
assert metrics_df.loc[0, "non_events"] == non_events

assert np.isfinite(roc_auc)
assert np.isfinite(pr_auc)
assert np.isfinite(brier)

print("\nValidation row reconciliation: PASS")
print("Probability integrity: PASS")
print("Metric integrity: PASS")

print("\n" + "=" * 70)
print("CELL 27 RESULT: PASS")
print("=" * 70)

# ------------------------------------------------
# 11. Cleanup
# ------------------------------------------------
del y_validation
del validation_probability

gc.collect()

con.close()

NON-LINEAR BENCHMARK — FULL VALIDATION EVALUATION
MEMORY-EFFICIENT BATCHED PREDICTION

Loading trained model...
Model loaded successfully.
Expected predictor features: 129

Inspecting validation dataset...


,observations,events,non_events,min_date,max_date
0,1569879,50952.0,1518927.0,2023-01-01,2023-12-01



Validation observations: 1,569,879
Validation events      : 50,952
Validation non-events  : 1,518,927
Validation event rate  : 3.2456%

Predictor schema alignment: PASS
Validation predictors: 129

Starting batched validation prediction...
Batch size: 50,000
Total rows to evaluate: 1,569,879
All validation observations will be evaluated.


C:\Users\madhu\AppData\Local\Temp\ipykernel_13856\1535485101.py:186: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = con.execute(query).fetch_record_batch(


Batch   1 | Rows processed:     50,000 / 1,569,879 | Elapsed: 0.02 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Batch   5 | Rows processed:    250,000 / 1,569,879 | Elapsed: 0.35 min
Batch  10 | Rows processed:    500,000 / 1,569,879 | Elapsed: 0.43 min
Batch  15 | Rows processed:    750,000 / 1,569,879 | Elapsed: 0.52 min
Batch  20 | Rows processed:  1,000,000 / 1,569,879 | Elapsed: 0.60 min
Batch  25 | Rows processed:  1,250,000 / 1,569,879 | Elapsed: 0.66 min
Batch  30 | Rows processed:  1,500,000 / 1,569,879 | Elapsed: 0.72 min

Prediction completed.
Rows evaluated: 1,569,879
Batches used   : 32

FULL VALIDATION PERFORMANCE
ROC-AUC : 0.810745
PR-AUC  : 0.256778
Brier   : 0.103845

Metrics saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\hist_gradient_boosting_validation_metrics.csv

Validation row reconciliation: PASS
Probability integrity: PASS
Metric integrity: PASS

CELL 27 RESULT: PASS


## Full Test Evaluation — Histogram Gradient Boosting

This cell evaluates the trained Histogram Gradient Boosting benchmark on the
untouched temporal test set (January 2024 to September 2025).

The evaluation is performed in batches to avoid loading the full test predictor
matrix into memory.

Metrics reported:
- ROC-AUC
- PR-AUC
- Brier score

No model fitting or parameter tuning is performed on the test set.

In [29]:
# ================================================================
# CELL 28 — FULL TEST EVALUATION
# MEMORY-EFFICIENT BATCHED PREDICTION
# ================================================================

from pathlib import Path
import duckdb
import joblib
import numpy as np
import pandas as pd
import time

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

print("=" * 70)
print("NON-LINEAR BENCHMARK — FULL TEST EVALUATION")
print("MEMORY-EFFICIENT BATCHED PREDICTION")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

TEST_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "transformed_datasets"
    / "test_transformed.parquet"
)

MODEL_FILE = (
    PROJECT_DIR
    / "models"
    / "hist_gradient_boosting_baseline.joblib"
)

METRICS_FILE = (
    PROJECT_DIR
    / "models"
    / "hist_gradient_boosting_test_metrics.csv"
)

assert TEST_FILE.exists(), f"Test dataset not found: {TEST_FILE}"
assert MODEL_FILE.exists(), f"Model not found: {MODEL_FILE}"

# ------------------------------------------------
# 2. Load trained model
# ------------------------------------------------

print("\nLoading trained model...")

hgb_model = joblib.load(MODEL_FILE)

print("Model loaded successfully.")
print(f"Expected predictors: {hgb_model.n_features_in_}")

# ------------------------------------------------
# 3. Inspect test dataset
# ------------------------------------------------

print("\nInspecting test dataset...")

test_con = duckdb.connect()
test_con.execute("SET memory_limit = '4GB'")
test_con.execute("SET threads = 2")
test_con.execute(
    f"SET temp_directory = '{PROJECT_DIR / 'data' / 'processed' / 'model_preprocessing' / 'test_evaluation_duckdb_temp'}'"
)

test_summary = test_con.execute(
    f"""
    SELECT
        COUNT(*) AS observations,
        COUNT(DISTINCT loan_id) AS unique_loans,
        SUM(target_6m) AS events,
        SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
        MIN(reporting_date) AS min_date,
        MAX(reporting_date) AS max_date
    FROM read_parquet('{TEST_FILE}')
    """
).fetchdf()

print(test_summary)

TEST_ROWS = int(test_summary.loc[0, "observations"])
TEST_EVENTS = int(test_summary.loc[0, "events"])
TEST_NON_EVENTS = int(test_summary.loc[0, "non_events"])

print(f"\nTest observations : {TEST_ROWS:,}")
print(f"Test events       : {TEST_EVENTS:,}")
print(f"Test non-events   : {TEST_NON_EVENTS:,}")
print(
    f"Test event rate   : "
    f"{TEST_EVENTS / TEST_ROWS * 100:.4f}%"
)

# ------------------------------------------------
# 4. Predictor schema validation
# ------------------------------------------------

schema_df = test_con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{TEST_FILE}')
    """
).fetchdf()

test_columns = schema_df["column_name"].tolist()

CORE_COLUMNS = {
    "loan_id",
    "reporting_date",
    "target_6m"
}

PREDICTOR_COLUMNS = [
    c for c in test_columns
    if c not in CORE_COLUMNS
]

print("\nPredictor schema validation:")
print(f"Test predictor columns : {len(PREDICTOR_COLUMNS)}")
print(f"Model expected         : {hgb_model.n_features_in_}")

assert len(PREDICTOR_COLUMNS) == hgb_model.n_features_in_

print("Predictor schema alignment: PASS")

# ------------------------------------------------
# 5. Batched prediction
# ------------------------------------------------

BATCH_SIZE = 50_000

y_true_parts = []
y_prob_parts = []

print("\nStarting batched test prediction...")
print(f"Batch size : {BATCH_SIZE:,}")
print(f"Total rows : {TEST_ROWS:,}")

start_time = time.time()
rows_processed = 0
batch_number = 0

select_columns = ", ".join(
    [f'"{c}"' for c in PREDICTOR_COLUMNS]
    + ['"target_6m"']
)

reader = test_con.execute(
    f"""
    SELECT {select_columns}
    FROM read_parquet('{TEST_FILE}')
    """
)

while True:

    batch = reader.fetch_df_chunk(BATCH_SIZE)

    if batch.empty:
        break

    batch_number += 1

    y_batch = batch["target_6m"].to_numpy(dtype=np.int8)

    X_batch = batch[PREDICTOR_COLUMNS].to_numpy(
        dtype=np.float32
    )

    p_batch = hgb_model.predict_proba(X_batch)[:, 1]

    y_true_parts.append(y_batch)
    y_prob_parts.append(p_batch)

    rows_processed += len(batch)

    if (
        batch_number == 1
        or batch_number % 5 == 0
        or rows_processed == TEST_ROWS
    ):
        elapsed = (time.time() - start_time) / 60

        print(
            f"Batch {batch_number:>3} | "
            f"Rows processed: {rows_processed:,} / {TEST_ROWS:,} | "
            f"Elapsed: {elapsed:.2f} min"
        )

# ------------------------------------------------
# 6. Reconstruct prediction vectors
# ------------------------------------------------

y_test = np.concatenate(y_true_parts)
p_test = np.concatenate(y_prob_parts)

print("\nPrediction completed.")
print(f"Rows evaluated : {len(y_test):,}")
print(f"Batches used   : {batch_number}")

assert len(y_test) == TEST_ROWS
assert len(p_test) == TEST_ROWS

# ------------------------------------------------
# 7. Probability integrity
# ------------------------------------------------

assert np.isfinite(p_test).all()
assert np.all((p_test >= 0) & (p_test <= 1))

print("\nProbability integrity: PASS")

# ------------------------------------------------
# 8. Test metrics
# ------------------------------------------------

roc_auc = roc_auc_score(y_test, p_test)
pr_auc = average_precision_score(y_test, p_test)
brier = brier_score_loss(y_test, p_test)

print("\n" + "=" * 70)
print("FULL TEST PERFORMANCE")
print("=" * 70)

print(f"ROC-AUC : {roc_auc:.6f}")
print(f"PR-AUC  : {pr_auc:.6f}")
print(f"Brier   : {brier:.6f}")

# ------------------------------------------------
# 9. Save metrics
# ------------------------------------------------

metrics_df = pd.DataFrame(
    [{
        "model": "hist_gradient_boosting",
        "split": "test",
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "brier_score": brier,
        "observations": TEST_ROWS,
        "events": TEST_EVENTS,
        "non_events": TEST_NON_EVENTS,
        "event_rate_pct": TEST_EVENTS / TEST_ROWS * 100,
        "min_date": test_summary.loc[0, "min_date"],
        "max_date": test_summary.loc[0, "max_date"]
    }]
)

metrics_df.to_csv(METRICS_FILE, index=False)

print("\nMetrics saved:")
print(METRICS_FILE)

# ------------------------------------------------
# 10. Final reconciliation
# ------------------------------------------------

assert len(y_test) == TEST_ROWS
assert np.isfinite(p_test).all()
assert ((p_test >= 0) & (p_test <= 1)).all()
assert set(np.unique(y_test)).issubset({0, 1})

print("\n" + "=" * 70)
print("CELL 28 RESULT: PASS")
print("=" * 70)

test_con.close()

# Release large arrays
del y_true_parts
del y_prob_parts
del y_test
del p_test

NON-LINEAR BENCHMARK — FULL TEST EVALUATION
MEMORY-EFFICIENT BATCHED PREDICTION

Loading trained model...
Model loaded successfully.
Expected predictors: 129

Inspecting test dataset...
   observations  unique_loans   events  non_events   min_date   max_date
0       2483739        128906  76861.0   2406878.0 2024-01-01 2025-09-01

Test observations : 2,483,739
Test events       : 76,861
Test non-events   : 2,406,878
Test event rate   : 3.0946%

Predictor schema validation:
Test predictor columns : 129
Model expected         : 129
Predictor schema alignment: PASS

Starting batched test prediction...
Batch size : 50,000
Total rows : 2,483,739


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(


Batch   1 | Rows processed: 2,483,739 / 2,483,739 | Elapsed: 0.94 min

Prediction completed.
Rows evaluated : 2,483,739
Batches used   : 1

Probability integrity: PASS

FULL TEST PERFORMANCE
ROC-AUC : 0.828852
PR-AUC  : 0.289215
Brier   : 0.094350

Metrics saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\hist_gradient_boosting_test_metrics.csv

CELL 28 RESULT: PASS


## Model Performance Comparison

This cell compares the regularized Logistic Regression baseline with the
Histogram Gradient Boosting nonlinear benchmark across the validation and
out-of-time test periods.

The comparison uses:
- ROC-AUC
- PR-AUC
- Brier score

The test set is used only for final out-of-time performance comparison and
not for model tuning or parameter selection.

In [30]:
# ================================================================
# CELL 29 — MODEL PERFORMANCE COMPARISON
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

MODELS_DIR = PROJECT_DIR / "models"

LOGISTIC_METRICS = MODELS_DIR / "baseline_logistic_metrics.csv"
HGB_VALIDATION_METRICS = MODELS_DIR / "hist_gradient_boosting_validation_metrics.csv"
HGB_TEST_METRICS = MODELS_DIR / "hist_gradient_boosting_test_metrics.csv"

COMPARISON_FILE = MODELS_DIR / "model_performance_comparison.csv"

# ------------------------------------------------
# 1. Check required files
# ------------------------------------------------

print("\nChecking metric files...")

required_files = {
    "Logistic baseline": LOGISTIC_METRICS,
    "HGB validation": HGB_VALIDATION_METRICS,
    "HGB test": HGB_TEST_METRICS
}

for name, path in required_files.items():
    print(f"{name:25s}: {path.exists()}")
    assert path.exists(), f"Missing metric file: {path}"

# ------------------------------------------------
# 2. Load metrics
# ------------------------------------------------

logistic = pd.read_csv(LOGISTIC_METRICS)
hgb_val = pd.read_csv(HGB_VALIDATION_METRICS)
hgb_test = pd.read_csv(HGB_TEST_METRICS)

print("\nLoaded metric records:")
print(f"Logistic rows       : {len(logistic)}")
print(f"HGB validation rows : {len(hgb_val)}")
print(f"HGB test rows       : {len(hgb_test)}")

# ------------------------------------------------
# 3. Standardize model names
# ------------------------------------------------

logistic["model"] = "Regularized Logistic Regression"
hgb_val["model"] = "Histogram Gradient Boosting"
hgb_test["model"] = "Histogram Gradient Boosting"

# ------------------------------------------------
# 4. Combine validation + test
# ------------------------------------------------

comparison = pd.concat(
    [
        logistic,
        hgb_val,
        hgb_test
    ],
    ignore_index=True
)

required_metrics = [
    "model",
    "split",
    "roc_auc",
    "pr_auc",
    "brier_score"
]

missing_metrics = [
    c for c in required_metrics
    if c not in comparison.columns
]

assert len(missing_metrics) == 0, (
    f"Missing metric columns: {missing_metrics}"
)

comparison = comparison[required_metrics + [
    c for c in comparison.columns
    if c not in required_metrics
]]

# ------------------------------------------------
# 5. Display comparison
# ------------------------------------------------

print("\n" + "=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

display(
    comparison[
        [
            "model",
            "split",
            "roc_auc",
            "pr_auc",
            "brier_score"
        ]
    ].sort_values(
        ["split", "roc_auc"],
        ascending=[True, False]
    )
)

# ------------------------------------------------
# 6. Extract HGB vs Logistic differences
# ------------------------------------------------

print("\n" + "=" * 70)
print("PERFORMANCE DIFFERENCE — HGB MINUS LOGISTIC")
print("=" * 70)

for split in ["validation", "test"]:

    split_df = comparison[
        comparison["split"].astype(str).str.lower() == split
    ].copy()

    if len(split_df) < 2:
        print(f"\n{split.upper()}: insufficient comparable records")
        continue

    hgb_row = split_df[
        split_df["model"] == "Histogram Gradient Boosting"
    ]

    log_row = split_df[
        split_df["model"] == "Regularized Logistic Regression"
    ]

    if len(hgb_row) == 1 and len(log_row) == 1:

        roc_diff = (
            hgb_row["roc_auc"].iloc[0]
            - log_row["roc_auc"].iloc[0]
        )

        pr_diff = (
            hgb_row["pr_auc"].iloc[0]
            - log_row["pr_auc"].iloc[0]
        )

        brier_diff = (
            hgb_row["brier_score"].iloc[0]
            - log_row["brier_score"].iloc[0]
        )

        print(f"\n{split.upper()}")
        print(f"ROC-AUC difference : {roc_diff:+.6f}")
        print(f"PR-AUC difference  : {pr_diff:+.6f}")
        print(f"Brier difference   : {brier_diff:+.6f}")

# ------------------------------------------------
# 7. Save comparison
# ------------------------------------------------

comparison.to_csv(
    COMPARISON_FILE,
    index=False
)

print("\nComparison saved:")
print(COMPARISON_FILE)

# ------------------------------------------------
# 8. Integrity checks
# ------------------------------------------------

assert comparison["roc_auc"].notna().all()
assert comparison["pr_auc"].notna().all()
assert comparison["brier_score"].notna().all()

assert np.isfinite(
    comparison[
        ["roc_auc", "pr_auc", "brier_score"]
    ].to_numpy()
).all()

print("\n" + "=" * 70)
print("CELL 29 RESULT: PASS")
print("=" * 70)

MODEL PERFORMANCE COMPARISON

Checking metric files...
Logistic baseline        : True
HGB validation           : True
HGB test                 : True

Loaded metric records:
Logistic rows       : 3
HGB validation rows : 1
HGB test rows       : 1

MODEL PERFORMANCE


,model,split,roc_auc,pr_auc,brier_score
2,Regularized Logistic Regression,TEST,0.503883,0.031180,0.956208
0,Regularized Logistic Regression,TRAIN,0.546637,0.035473,0.791905
1,Regularized Logistic Regression,VALIDATION,0.502889,0.032638,0.959384
4,Histogram Gradient Boosting,test,0.828852,0.289215,0.094350
3,Histogram Gradient Boosting,validation,0.810745,0.256778,0.103845



PERFORMANCE DIFFERENCE — HGB MINUS LOGISTIC

VALIDATION
ROC-AUC difference : +0.307856
PR-AUC difference  : +0.224140
Brier difference   : -0.855540

TEST
ROC-AUC difference : +0.324969
PR-AUC difference  : +0.258034
Brier difference   : -0.861858

Comparison saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\model_performance_comparison.csv

CELL 29 RESULT: PASS


## Model Selection and Benchmark Registry

The regularized Logistic Regression model serves as the linear benchmark,
while Histogram Gradient Boosting serves as the nonlinear benchmark.

Model selection is based primarily on validation ROC-AUC and PR-AUC because
the validation period is reserved for model comparison and selection.

The test period is not used for tuning. Its results are retained only as
out-of-time confirmation of generalization.

The selected candidate model will be carried forward to explainability,
risk ranking, calibration analysis, and the dynamic early-warning framework.

In [31]:
# ================================================================
# CELL 30 — MODEL SELECTION AND BENCHMARK REGISTRY
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

print("=" * 70)
print("MODEL SELECTION — BENCHMARK REGISTRY")
print("=" * 70)

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

MODELS_DIR = PROJECT_DIR / "models"
COMPARISON_FILE = MODELS_DIR / "model_performance_comparison.csv"
REGISTRY_FILE = MODELS_DIR / "model_registry.json"

assert COMPARISON_FILE.exists(), (
    f"Comparison file not found: {COMPARISON_FILE}"
)

# ------------------------------------------------
# 1. Load comparison results
# ------------------------------------------------

comparison = pd.read_csv(COMPARISON_FILE)

print("\nLoaded comparison results:")
display(comparison)

# ------------------------------------------------
# 2. Separate validation results
# ------------------------------------------------

validation_df = comparison[
    comparison["split"].astype(str).str.lower() == "validation"
].copy()

test_df = comparison[
    comparison["split"].astype(str).str.lower() == "test"
].copy()

assert len(validation_df) >= 2, (
    "Validation results for both benchmark models are required."
)

# ------------------------------------------------
# 3. Select model using validation PR-AUC
# ------------------------------------------------
# PR-AUC is particularly informative for the imbalanced
# 3.21% event rate. ROC-AUC is retained as a secondary metric.

best_validation_idx = validation_df["pr_auc"].idxmax()

selected_model = validation_df.loc[
    best_validation_idx,
    "model"
]

selected_validation_pr_auc = validation_df.loc[
    best_validation_idx,
    "pr_auc"
]

selected_validation_roc_auc = validation_df.loc[
    best_validation_idx,
    "roc_auc"
]

selected_validation_brier = validation_df.loc[
    best_validation_idx,
    "brier_score"
]

print("\n" + "=" * 70)
print("VALIDATION-BASED MODEL SELECTION")
print("=" * 70)

print(f"Selected model : {selected_model}")
print(f"Validation ROC-AUC : {selected_validation_roc_auc:.6f}")
print(f"Validation PR-AUC  : {selected_validation_pr_auc:.6f}")
print(f"Validation Brier   : {selected_validation_brier:.6f}")

# ------------------------------------------------
# 4. Retrieve out-of-time test performance
# ------------------------------------------------

selected_test = test_df[
    test_df["model"] == selected_model
].copy()

assert len(selected_test) == 1, (
    "Exactly one test result is required for the selected model."
)

selected_test_roc_auc = selected_test["roc_auc"].iloc[0]
selected_test_pr_auc = selected_test["pr_auc"].iloc[0]
selected_test_brier = selected_test["brier_score"].iloc[0]

print("\nOut-of-time test confirmation:")
print(f"Test ROC-AUC : {selected_test_roc_auc:.6f}")
print(f"Test PR-AUC  : {selected_test_pr_auc:.6f}")
print(f"Test Brier   : {selected_test_brier:.6f}")

# ------------------------------------------------
# 5. Model registry
# ------------------------------------------------

registry = {
    "selection_basis": {
        "primary_metric": "validation_pr_auc",
        "secondary_metric": "validation_roc_auc",
        "test_usage": "out_of_time_confirmation_only"
    },

    "selected_model": {
        "name": selected_model,
        "validation": {
            "roc_auc": float(selected_validation_roc_auc),
            "pr_auc": float(selected_validation_pr_auc),
            "brier_score": float(selected_validation_brier)
        },
        "test": {
            "roc_auc": float(selected_test_roc_auc),
            "pr_auc": float(selected_test_pr_auc),
            "brier_score": float(selected_test_brier)
        }
    },

    "benchmark_models": [
        "Regularized Logistic Regression",
        "Histogram Gradient Boosting"
    ],

    "dataset_design": {
        "training_period": "2018-01 to 2022-12",
        "validation_period": "2023-01 to 2023-12",
        "test_period": "2024-01 to 2025-09",
        "target_horizon": "6 calendar months",
        "event_definition": "30+ days delinquent within next 6 calendar months"
    }
}

with open(REGISTRY_FILE, "w", encoding="utf-8") as f:
    json.dump(registry, f, indent=2)

print("\nModel registry saved:")
print(REGISTRY_FILE)

# ------------------------------------------------
# 6. Integrity checks
# ------------------------------------------------

assert selected_model == "Histogram Gradient Boosting"

assert np.isfinite(selected_validation_pr_auc)
assert np.isfinite(selected_validation_roc_auc)
assert np.isfinite(selected_test_pr_auc)
assert np.isfinite(selected_test_roc_auc)

assert selected_validation_pr_auc > 0
assert selected_test_pr_auc > 0

print("\n" + "=" * 70)
print("CELL 30 RESULT: PASS")
print("=" * 70)
print(f"Selected candidate: {selected_model}")

MODEL SELECTION — BENCHMARK REGISTRY

Loaded comparison results:


,model,split,roc_auc,pr_auc,brier_score,observations,events,non_events,event_rate_pct,min_date,max_date
0,Regularized Logistic Regression,TRAIN,0.546637,0.035473,0.791905,16101651,NaN,NaN,NaN,NaN,NaN
1,Regularized Logistic Regression,VALIDATION,0.502889,0.032638,0.959384,1569879,NaN,NaN,NaN,NaN,NaN
2,Regularized Logistic Regression,TEST,0.503883,0.031180,0.956208,2483739,NaN,NaN,NaN,NaN,NaN
3,Histogram Gradient Boosting,validation,0.810745,0.256778,0.103845,1569879,50952.0,1518927.0,NaN,NaN,NaN
4,Histogram Gradient Boosting,test,0.828852,0.289215,0.094350,2483739,76861.0,2406878.0,3.094568,2024-01-01,2025-09-01



VALIDATION-BASED MODEL SELECTION
Selected model : Histogram Gradient Boosting
Validation ROC-AUC : 0.810745
Validation PR-AUC  : 0.256778
Validation Brier   : 0.103845

Out-of-time test confirmation:
Test ROC-AUC : 0.828852
Test PR-AUC  : 0.289215
Test Brier   : 0.094350

Model registry saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\model_registry.json

CELL 30 RESULT: PASS
Selected candidate: Histogram Gradient Boosting


## MEMORY-EFFICIENT MODEL DIAGNOSTICS

The selected Histogram Gradient Boosting model is evaluated using
batched prediction to avoid loading the full temporal datasets into
memory simultaneously.

Only the predictor columns and target are read from the Parquet files.
Validation observations are processed in batches, while the training
dataset is not unnecessarily materialized in memory.

This preserves the temporal split and prevents memory-related failures.

In [33]:
# ================================================================
# MEMORY-EFFICIENT MODEL DIAGNOSTICS
# ================================================================

from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import joblib
import gc
import time

print("=" * 70)
print("MEMORY-EFFICIENT MODEL DIAGNOSTICS")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

TRAIN_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "transformed_datasets"
    / "train_transformed.parquet"
)

VALIDATION_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model_preprocessing"
    / "transformed_datasets"
    / "validation_transformed.parquet"
)

MODEL_FILE = (
    PROJECT_DIR
    / "models"
    / "hist_gradient_boosting_baseline.joblib"
)

OUTPUT_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_FILE.exists(), "Training dataset not found."
assert VALIDATION_FILE.exists(), "Validation dataset not found."
assert MODEL_FILE.exists(), "HGB model not found."

# ------------------------------------------------
# 2. Load model
# ------------------------------------------------

print("\nLoading selected model...")

hgb_model = joblib.load(MODEL_FILE)

print("Model loaded successfully.")
print("Expected predictor count :", hgb_model.n_features_in_)

EXPECTED_FEATURES = int(hgb_model.n_features_in_)

# ------------------------------------------------
# 3. Inspect schema WITHOUT loading full data
# ------------------------------------------------

con = duckdb.connect()

con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")
con.execute(
    f"SET temp_directory = '{PROJECT_DIR / 'data' / 'processed' / 'model_preprocessing' / 'diagnostics_duckdb_temp'}'"
)

schema_df = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{VALIDATION_FILE}')
    """
).fetchdf()

all_columns = schema_df["column_name"].tolist()

CORE_COLUMNS = [
    "loan_id",
    "reporting_date",
    "target_6m"
]

PREDICTOR_COLUMNS = [
    c for c in all_columns
    if c not in CORE_COLUMNS
]

print("\nValidation schema:")
print("Total columns        :", len(all_columns))
print("Predictor columns    :", len(PREDICTOR_COLUMNS))

assert len(PREDICTOR_COLUMNS) == EXPECTED_FEATURES, (
    f"Expected {EXPECTED_FEATURES} predictors, "
    f"found {len(PREDICTOR_COLUMNS)}."
)

print("Predictor alignment   : PASS")

# ------------------------------------------------
# 4. Validation dataset summary
# ------------------------------------------------

validation_summary = con.execute(
    f"""
    SELECT
        COUNT(*) AS observations,
        SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END) AS events,
        SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
        MIN(reporting_date) AS min_date,
        MAX(reporting_date) AS max_date
    FROM read_parquet('{VALIDATION_FILE}')
    """
).fetchdf()

print("\nValidation dataset:")
print(validation_summary.to_string(index=False))

# ------------------------------------------------
# 5. Batched validation prediction
# ------------------------------------------------

BATCH_SIZE = 50_000

select_columns = (
    ", ".join([f'"{c}"' for c in PREDICTOR_COLUMNS])
    + ', "target_6m"'
)

validation_query = f"""
    SELECT {select_columns}
    FROM read_parquet('{VALIDATION_FILE}')
    ORDER BY reporting_date, loan_id
"""

reader = con.execute(validation_query).fetch_record_batch(BATCH_SIZE)

all_probabilities = []
all_targets = []

rows_processed = 0
batch_number = 0

start_time = time.time()

print("\nStarting batched validation prediction...")
print("Batch size :", f"{BATCH_SIZE:,}")

for batch in reader:

    batch_number += 1

    batch_df = batch.to_pandas()

    y_batch = batch_df["target_6m"].to_numpy(
        dtype=np.int8,
        copy=False
    )

    X_batch = batch_df[PREDICTOR_COLUMNS].to_numpy(
        dtype=np.float32,
        copy=False
    )

    probabilities = hgb_model.predict_proba(X_batch)[:, 1]

    all_probabilities.append(probabilities.astype(np.float32))
    all_targets.append(y_batch)

    rows_processed += len(batch_df)

    if batch_number == 1 or batch_number % 5 == 0:
        elapsed = (time.time() - start_time) / 60

        print(
            f"Batch {batch_number:>3} | "
            f"Rows processed: {rows_processed:,} | "
            f"Elapsed: {elapsed:.2f} min"
        )

    del batch_df, X_batch, y_batch, probabilities
    gc.collect()

# ------------------------------------------------
# 6. Combine only prediction vectors
# ------------------------------------------------

validation_probabilities = np.concatenate(
    all_probabilities
)

validation_targets = np.concatenate(
    all_targets
)

del all_probabilities
del all_targets
gc.collect()

print("\nPrediction completed.")

print("Rows evaluated :", f"{len(validation_targets):,}")
print("Probability count :", f"{len(validation_probabilities):,}")

# ------------------------------------------------
# 7. Probability integrity
# ------------------------------------------------

assert len(validation_targets) == len(validation_probabilities)

assert np.all(
    np.isfinite(validation_probabilities)
)

assert np.all(
    (validation_probabilities >= 0)
    & (validation_probabilities <= 1)
)

assert set(np.unique(validation_targets)).issubset({0, 1})

print("\nProbability integrity : PASS")

# ------------------------------------------------
# 8. Save validation predictions
# ------------------------------------------------

validation_predictions_file = (
    OUTPUT_DIR / "hist_gradient_boosting_validation_predictions.csv"
)

predictions_df = pd.DataFrame({
    "target_6m": validation_targets,
    "predicted_probability": validation_probabilities
})

predictions_df.to_csv(
    validation_predictions_file,
    index=False
)

print("\nValidation predictions saved:")
print(validation_predictions_file)

# ------------------------------------------------
# 9. Final checks
# ------------------------------------------------

expected_validation_rows = int(
    validation_summary.loc[0, "observations"]
)

expected_validation_events = int(
    validation_summary.loc[0, "events"]
)

assert len(predictions_df) == expected_validation_rows
assert int(predictions_df["target_6m"].sum()) == expected_validation_events

print("\nRow reconciliation : PASS")
print("Event reconciliation : PASS")

print("\n" + "=" * 70)
print("MEMORY-EFFICIENT DIAGNOSTICS RESULT: PASS")
print("=" * 70)

MEMORY-EFFICIENT MODEL DIAGNOSTICS

Loading selected model...
Model loaded successfully.
Expected predictor count : 129

Validation schema:
Total columns        : 132
Predictor columns    : 129
Predictor alignment   : PASS

Validation dataset:
 observations  events  non_events   min_date   max_date
      1569879 50952.0   1518927.0 2023-01-01 2023-12-01

Starting batched validation prediction...
Batch size : 50,000


C:\Users\madhu\AppData\Local\Temp\ipykernel_13856\989732083.py:148: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = con.execute(validation_query).fetch_record_batch(BATCH_SIZE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(


Batch   1 | Rows processed: 50,000 | Elapsed: 0.02 min


C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(


... (9 outputs left out)

C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have val

Batch  30 | Rows processed: 1,500,000 | Elapsed: 0.45 min


C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\madhu\Downloads\anaconda 3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(



Prediction completed.
Rows evaluated : 1,569,879
Probability count : 1,569,879

Probability integrity : PASS

Validation predictions saved:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\models\hist_gradient_boosting_validation_predictions.csv

Row reconciliation : PASS
Event reconciliation : PASS

MEMORY-EFFICIENT DIAGNOSTICS RESULT: PASS


## VALIDATION-BASED DECISION THRESHOLD ANALYSIS

The Histogram Gradient Boosting model produces continuous probabilities of
six-month delinquency.

Because the target is imbalanced, the default 0.50 classification threshold
is not assumed to be appropriate for an early-warning application.

Decision thresholds are therefore evaluated exclusively on the validation
set. Candidate thresholds are compared using precision, recall, F1-score,
F2-score, and the number of observations flagged as high risk.

F2-score is included because the early-warning objective places greater
importance on identifying potentially deteriorating borrowers than on
maximizing precision alone.

The test set is not used for threshold selection.

In [34]:
# ================================================================
# VALIDATION-BASED DECISION THRESHOLD ANALYSIS
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("=" * 70)
print("VALIDATION-BASED DECISION THRESHOLD ANALYSIS")
print("=" * 70)

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

PREDICTION_FILE = (
    PROJECT_DIR
    / "models"
    / "hist_gradient_boosting_validation_predictions.csv"
)

OUTPUT_FILE = (
    PROJECT_DIR
    / "models"
    / "hist_gradient_boosting_validation_threshold_analysis.csv"
)

assert PREDICTION_FILE.exists(), (
    "Validation prediction file not found."
)

# ------------------------------------------------
# 2. Load validation predictions
# ------------------------------------------------

validation_pred_df = pd.read_csv(PREDICTION_FILE)

assert "target_6m" in validation_pred_df.columns
assert "predicted_probability" in validation_pred_df.columns

y_val = validation_pred_df["target_6m"].to_numpy(dtype=np.int8)
p_val = validation_pred_df["predicted_probability"].to_numpy(
    dtype=np.float32
)

assert len(y_val) == 1_569_879
assert len(p_val) == 1_569_879

assert set(np.unique(y_val)).issubset({0, 1})
assert np.all(np.isfinite(p_val))
assert np.all((p_val >= 0) & (p_val <= 1))

print("\nValidation observations :", f"{len(y_val):,}")
print("Validation events       :", f"{y_val.sum():,}")
print(
    "Validation event rate   :",
    f"{100 * y_val.mean():.4f}%"
)

# ------------------------------------------------
# 3. Candidate thresholds
# ------------------------------------------------

thresholds = np.round(
    np.arange(0.01, 1.00, 0.01),
    2
)

results = []

for threshold in thresholds:

    y_pred = (
        p_val >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f2 = fbeta_score(
        y_val,
        y_pred,
        beta=2,
        zero_division=0
    )

    flagged = int(y_pred.sum())

    flagged_pct = (
        flagged / len(y_val) * 100
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "f2_score": f2,
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn),
        "flagged_observations": flagged,
        "flagged_rate_pct": flagged_pct
    })

threshold_df = pd.DataFrame(results)

# ------------------------------------------------
# 4. Identify validation-optimal thresholds
# ------------------------------------------------

best_f1 = threshold_df.loc[
    threshold_df["f1_score"].idxmax()
]

best_f2 = threshold_df.loc[
    threshold_df["f2_score"].idxmax()
]

best_precision = threshold_df.loc[
    threshold_df["precision"].idxmax()
]

# ------------------------------------------------
# 5. Display key results
# ------------------------------------------------

print("\n" + "=" * 70)
print("BEST F1 THRESHOLD")
print("=" * 70)

print(
    f"Threshold          : {best_f1['threshold']:.2f}"
)
print(
    f"Precision          : {best_f1['precision']:.4f}"
)
print(
    f"Recall             : {best_f1['recall']:.4f}"
)
print(
    f"F1-score           : {best_f1['f1_score']:.4f}"
)
print(
    f"F2-score           : {best_f1['f2_score']:.4f}"
)
print(
    f"Flagged observations: {int(best_f1['flagged_observations']):,}"
)
print(
    f"Flagged rate       : {best_f1['flagged_rate_pct']:.2f}%"
)

print("\n" + "=" * 70)
print("BEST F2 THRESHOLD — EARLY-WARNING ORIENTED")
print("=" * 70)

print(
    f"Threshold          : {best_f2['threshold']:.2f}"
)
print(
    f"Precision          : {best_f2['precision']:.4f}"
)
print(
    f"Recall             : {best_f2['recall']:.4f}"
)
print(
    f"F1-score           : {best_f2['f1_score']:.4f}"
)
print(
    f"F2-score           : {best_f2['f2_score']:.4f}"
)
print(
    f"Flagged observations: {int(best_f2['flagged_observations']):,}"
)
print(
    f"Flagged rate       : {best_f2['flagged_rate_pct']:.2f}%"
)

# ------------------------------------------------
# 6. Fixed recall-oriented operating points
# ------------------------------------------------

print("\n" + "=" * 70)
print("RECALL-ORIENTED OPERATING POINTS")
print("=" * 70)

for target_recall in [0.50, 0.60, 0.70, 0.80, 0.90]:

    eligible = threshold_df[
        threshold_df["recall"] >= target_recall
    ]

    if len(eligible) == 0:
        print(
            f"Recall >= {target_recall:.0%}: "
            "No threshold available"
        )
        continue

    # Highest threshold that still achieves target recall.
    selected = eligible.sort_values(
        "threshold",
        ascending=False
    ).iloc[0]

    print(
        f"\nRecall >= {target_recall:.0%}"
    )
    print(
        f"  Threshold : {selected['threshold']:.2f}"
    )
    print(
        f"  Recall    : {selected['recall']:.4f}"
    )
    print(
        f"  Precision : {selected['precision']:.4f}"
    )
    print(
        f"  F2-score  : {selected['f2_score']:.4f}"
    )
    print(
        f"  Flag rate : {selected['flagged_rate_pct']:.2f}%"
    )

# ------------------------------------------------
# 7. Save complete threshold analysis
# ------------------------------------------------

threshold_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("THRESHOLD ANALYSIS SAVED")
print("=" * 70)

print(OUTPUT_FILE)
print(
    "\nValidation-only threshold analysis : PASS"
)

# ------------------------------------------------
# 8. Integrity assertions
# ------------------------------------------------

assert len(threshold_df) == 99
assert threshold_df["threshold"].min() == 0.01
assert threshold_df["threshold"].max() == 0.99

assert np.all(
    threshold_df["precision"].between(0, 1)
)

assert np.all(
    threshold_df["recall"].between(0, 1)
)

assert np.all(
    threshold_df["f1_score"].between(0, 1)
)

assert np.all(
    threshold_df["f2_score"].between(0, 1)
)

print("\n" + "=" * 70)
print("CELL RESULT: PASS")
print("=" * 70)

VALIDATION-BASED DECISION THRESHOLD ANALYSIS

Validation observations : 1,569,879
Validation events       : 50,952
Validation event rate   : 3.2456%

BEST F1 THRESHOLD
Threshold          : 0.76
Precision          : 0.3467
Recall             : 0.3455
F1-score           : 0.3461
F2-score           : 0.3458
Flagged observations: 50,785
Flagged rate       : 3.23%

BEST F2 THRESHOLD — EARLY-WARNING ORIENTED
Threshold          : 0.58
Precision          : 0.2262
Recall             : 0.4992
F1-score           : 0.3113
F2-score           : 0.4021
Flagged observations: 112,474
Flagged rate       : 7.16%

RECALL-ORIENTED OPERATING POINTS

Recall >= 50%
  Threshold : 0.57
  Recall    : 0.5062
  Precision : 0.2206
  F2-score  : 0.4021
  Flag rate : 7.45%

Recall >= 60%
  Threshold : 0.44
  Recall    : 0.6086
  Precision : 0.1372
  F2-score  : 0.3607
  Flag rate : 14.40%

Recall >= 70%
  Threshold : 0.36
  Recall    : 0.7002
  Precision : 0.0942
  F2-score  : 0.3063
  Flag rate : 24.12%

Recall >= 8